# S³DT — Django Case Study 3 and System-Level Quantitative Evaluation

**IEEE TDSC major revision · Google Colab · Original S³DT equations retained**

This notebook has two clearly separated components:

1. **Earlier exploratory Case Study 3 demonstration (Sections 1–12)**: Retains the original Django view-only access-control pilot, full claim graph, retrospective reference fix, and ordinary post-fix changes for comparison with the first two reconstructed case studies. Its SIC parameters were analyst-assigned and are **not** used by the quantitative study.
2. **Active system-level quantitative evaluation (Section 13 onward)**: Defines **one Django-wide SIC collection before the batch**, then applies *every* SIC to *every* applicable historical Django checkpoint, including four independent security-fix histories and seven genuine ordinary commits. It reuses the released S³DT claim, graph, and hybrid-scoring routines. A separate policy library would be required for Requests.

The program exports original and new figures, policy hash, claim inventories, provenance, graph, per-SIC structured/embedding/hybrid outputs, drift explanations, timing records and an independent-label worksheet.

**Readiness limitation:** The initial system-SIC policy definitions and C/O/H assignments are retrospective and provisional. The results are *diagnostic*, not independent accuracy measurements, until policy calibration, security reference labels, source-projection validation, and evaluator independence are documented. Fix states are reference-only, not pre-fix detector inputs.

**Updated figures:** Section 13.1C-2 exports integrated graphs for every evaluated Django history, including an inspectable claim-level HTML graph. Pilot figures in Sections 7--8 remain specific to Django ticket #30289.


## 0. Environment and input files

On Colab, the first cell installs the required packages. Then upload the two required inputs (plus optional Django post-fix control snapshot cache) when prompted. The optional previous-output ZIP avoids downloading the two historical source files again.

In [ ]:
# Colab installation. Skip this cell only if you already installed the dependencies.
%pip -q install "pandas>=2" "numpy>=1.24" "networkx>=3" "matplotlib>=3.8" "pydantic>=2" "sentence-transformers>=3" "tabulate>=0.9"

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import ast, copy, csv, hashlib, importlib.metadata as metadata, io, json, os, platform, re, sys, time, urllib.request, zipfile
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from IPython.display import display, Image

IS_COLAB = 'google.colab' in sys.modules
BASE = Path('/content') if IS_COLAB else Path('/mnt/data')
INPUTS = BASE / 'tdsc_case3_inputs'
OUT = BASE / 'tdsc_case3_finished_outputs'
INPUTS.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

REQUIRED = ['swe_bench_verified(1).csv', 'S3DT-main.zip']
OPTIONAL = ['tdsc_django_s3dt_pilot_outputs.zip']
if IS_COLAB:
    missing = [p for p in REQUIRED if not (INPUTS/p).exists()]
    if missing:
        from google.colab import files
        print('Upload these two original files (optional third file is accepted):', missing)
        for name, blob in files.upload().items():
            (INPUTS/name).write_bytes(blob)
else:
    for name in REQUIRED + OPTIONAL:
        if not (INPUTS/name).exists() and (BASE/name).exists():
            (INPUTS/name).write_bytes((BASE/name).read_bytes())
for fn in REQUIRED:
    if not (INPUTS/fn).exists():
        raise FileNotFoundError(f'Missing required file: {fn}')

def sha256(value):
    if isinstance(value, str): value = value.encode('utf-8')
    return hashlib.sha256(value).hexdigest()

def save_json(name, obj):
    (OUT/name).write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding='utf-8')

TIMERS = {}
print('Working directory:', OUT)

## 1. Reconstruct the independent case record

The input SWE-bench task and source revisions are immutable. This notebook keeps the human fix and tests **out of the pre-fix detector input**.

In [ ]:
t0=time.perf_counter()
DATA = pd.read_csv(INPUTS/REQUIRED[0]).fillna('')
TASK_ID='django__django-11149'
case_rows=DATA.loc[DATA.instance_id.eq(TASK_ID)]
assert len(case_rows)==1, f'Expected one task for {TASK_ID}; got {len(case_rows)}'
case=case_rows.iloc[0].to_dict()
BASE_COMMIT=str(case['base_commit'])
FIX_COMMIT='8335d59200e4c64dfe3348ea93989d95e0107439'
SOURCE_PATH='django/contrib/admin/options.py'
ISSUE_URL='https://code.djangoproject.com/ticket/30289'
ISSUE_TIME=str(case['created_at'])
assert 'view permission' in str(case['problem_statement']).lower()
(OUT/'reference_human_fix.patch').write_text(str(case['patch']),encoding='utf-8')
(OUT/'reference_regression_tests.patch').write_text(str(case['test_patch']),encoding='utf-8')
(OUT/'issue_report_excerpt.txt').write_text(str(case['problem_statement']),encoding='utf-8')
TIMERS['load_case_seconds']=time.perf_counter()-t0
print('Task',TASK_ID,'Base:',BASE_COMMIT,'Reference fix:',FIX_COMMIT,'Issue published:',ISSUE_TIME)
print('The fix and tests are reference-only at the pre-fix checkpoint.')

## 2. Obtain the source at exact historical commits

Use the offline cache ZIP if present; otherwise download each file from its immutable commit URL. The cache is checked against the same SHA-256 values recorded in the earlier Django pilot when available.

In [ ]:
t0=time.perf_counter()
CACHED={}
CACHE_HASHES={}
cache_zip=INPUTS/OPTIONAL[0]
if cache_zip.exists():
    with zipfile.ZipFile(cache_zip) as z:
        for label,name in [('before','source_before.py'),('after','source_after.py')]:
            if name in z.namelist(): CACHED[label]=z.read(name).decode('utf-8')
        if 'pilot_manifest.json' in z.namelist():
            prior=json.loads(z.read('pilot_manifest.json'))
            CACHE_HASHES={rec['commit']:rec['sha256'] for rec in prior.get('historical_sources',[])}

def fetch_source(commit, label):
    url=f'https://raw.githubusercontent.com/django/django/{commit}/{SOURCE_PATH}'
    if label in CACHED:
        text=CACHED[label]; mode='verified_previous_run_cache'
    else:
        req=urllib.request.Request(url,headers={'User-Agent':'S3DT-TDSC-reproducibility/1.0'})
        with urllib.request.urlopen(req,timeout=60) as r:
            text=r.read().decode('utf-8')
        mode='commit_pinned_download'
    ast.parse(text)
    if label in CACHED and commit in CACHE_HASHES:
        assert sha256(text)==CACHE_HASHES[commit], 'Cached source does not match prior recorded SHA-256'
    return text, {'commit':commit,'path':SOURCE_PATH,'url':url,
                  'sha256':sha256(text),'source_mode':mode}

BEFORE, before_prov=fetch_source(BASE_COMMIT,'before')
AFTER, after_prov=fetch_source(FIX_COMMIT,'after')
(OUT/'source_before.py').write_text(BEFORE,encoding='utf-8')
(OUT/'source_after.py').write_text(AFTER,encoding='utf-8')
TIMERS['load_sources_seconds']=time.perf_counter()-t0
print('Base source SHA-256:',before_prov['sha256'])
print('Reference source SHA-256:',after_prov['sha256'])
print('Source acquisition modes:',before_prov['source_mode'],after_prov['source_mode'])

## 3. Extract source-backed claims (no LLM)

Each relevant `InlineModelAdmin` method is parsed from the actual source. We record the **implicit-through-model** branch separately, so the claim is more informative than a fingerprint alone. All extracted claims have a commit/file/line citation.

The risk-stage claim is deterministic: the three relevant methods either delegate to view permission, require change permission, or are marked **unresolved**. No stage is derived from the future fix when classifying the earlier code.

In [ ]:
t0=time.perf_counter()
METHODS=['has_add_permission','has_change_permission','has_delete_permission']

def analyze_source(text, artifact_id, prov):
    tree=ast.parse(text)
    target=next((n for n in tree.body if isinstance(n,ast.ClassDef) and n.name=='InlineModelAdmin'),None)
    assert target is not None,'InlineModelAdmin missing: inspect source version'
    records=[]
    for method in METHODS:
        fn=next((n for n in target.body if isinstance(n,ast.FunctionDef) and n.name==method),None)
        assert fn is not None,f'Missing InlineModelAdmin.{method}'
        auto_if=next((n for n in fn.body if isinstance(n,ast.If) and
                      'self.opts.auto_created' in (ast.get_source_segment(text,n.test) or '')),None)
        assert auto_if is not None, f'No implicit through-model guard in {method}'
        branch=ast.get_source_segment(text,auto_if) or ''
        # This is intentionally a narrow AST-backed classification of source structure,
        # not a general claim that a Django runtime vulnerability has been reproduced.
        view_calls=any(isinstance(call,ast.Call) and isinstance(call.func,ast.Attribute)
                       and call.func.attr=='has_view_permission'
                       for stmt in auto_if.body for call in ast.walk(stmt))
        change_guard=any(isinstance(call,ast.Call) and isinstance(call.func,ast.Attribute)
                         and call.func.attr=='_has_any_perms_for_target_model'
                         and any(isinstance(x,ast.Constant) and x.value=='change' for arg in call.args for x in ast.walk(arg))
                         for stmt in auto_if.body for call in ast.walk(stmt))
        decision=('delegates_to_view_permission' if view_calls and not change_guard else
                  'requires_related_model_change_permission' if change_guard and not view_calls else 'unresolved')
        records.append({'artifact':artifact_id,'subject':f'InlineModelAdmin.{method}',
                        'predicate':'code_behavior','object':decision,
                        'evidence':f"{prov['commit']}:{prov['path']}:{auto_if.lineno}-{auto_if.end_lineno}",
                        'line_start':auto_if.lineno,'line_end':auto_if.end_lineno,
                        'provenance_sha256':prov['sha256'], 'source_excerpt':branch[:900],
                        'extraction_method':'Python AST / implicit through-model branch'})
    status={r['object'] for r in records}
    stage=('view_permission_controls_mutation' if status=={'delegates_to_view_permission'} else
           'change_permission_controls_mutation' if status=={'requires_related_model_change_permission'} else
           'unresolved')
    return records,stage

base_facts,stage_before=analyze_source(BEFORE,'source_base',before_prov)
fix_facts,stage_after=analyze_source(AFTER,'source_fix_reference',after_prov)
claim_facts=pd.DataFrame(base_facts+fix_facts)
claim_facts.to_csv(OUT/'source_backed_claims.csv',index=False)
if 'unresolved' in [stage_before,stage_after]:
    raise ValueError('Permission source pattern not covered; do not guess SIC outcomes or risk weights.')
TIMERS['ast_projection_seconds']=time.perf_counter()-t0
print('Base stage:',stage_before,'Reference stage:',stage_after)
display(claim_facts[['artifact','subject','object','evidence']])

## 4. Load the *released* S³DT framework code

The released notebook contains `Claim`, `ClaimSet`, `ClaimDelta`, `ProjectAndAnnotateThreadEdge`, SIC, scoring, and classification implementations. To allow graph construction without downloading a language model, this cell **delays only the model initialisation**. It does not replace the claim/delta/graph/scoring equations. The actual `all-MiniLM-L6-v2` model is initialised in the scoring stage.

In [ ]:
t0=time.perf_counter()
with zipfile.ZipFile(INPUTS/'S3DT-main.zip') as z:
    path='S3DT-main/case_studies/Git_Anthropic_case_study_setup.ipynb'
    nb_raw=json.loads(z.read(path))
core=''.join(nb_raw['cells'][1]['source'])
CORE_SHA256=sha256(core)
needle_import='from sentence_transformers import SentenceTransformer'
needle_load='_EMBEDDER = SentenceTransformer("all-MiniLM-L6-v2")'
assert core.count(needle_import)==1 and core.count(needle_load)==1, 'Original core changed; inspect before using this adapter.'
core_for_lazy_model=core.replace(needle_import,'# imported only at scoring time').replace(needle_load,'_EMBEDDER = None')
exec(compile(core_for_lazy_model,'released_s3dt_case_study_core.py','exec'),globals())
# Resolve Pydantic forward references after dynamically loading the original code.
for _cls in (Claim, ClaimSet, ClaimDelta, SIC, DriftCandidate, DriftEvent):
    _cls.model_rebuild(_types_namespace=globals(), force=True)
TIMERS['load_s3dt_core_seconds']=time.perf_counter()-t0
assert len(CORE_SHA256)==64
print('Original S³DT core hash:',CORE_SHA256)

## 5. Construct the S³DT semantic digital thread

The **detector thread** contains only (a) the reported access-control expectation and (b) the base source. The **historical reference thread** also includes the later fix and its source-backed claims. Claims use the original five-field representation with evidence and provenance. Semantic deltas and traceability links are computed using the original framework.

In [ ]:
t0=time.perf_counter()
Q={'domain':'django_admin','relation':'implicit_many_to_many_inline'}
policy_claim={'s':'admin_view_only_user','p':'policy_constraint',
              'o':'cannot_modify_implicit_m2m_relationships','q':Q,'gamma':1.0,
              'evidence':[ISSUE_URL],
              'provenance':{'source':'reported_security_expectation','available_after_issue_report':True}}

def claim_record(s,p,o,evidence,artifact):
    return {'s':s,'p':p,'o':o,'q':Q,'gamma':1.0,
            'evidence':[evidence], 'provenance':{'artifact_id':artifact,'method':'AST structural extraction'}}

def build_artifact(stage,facts,prov,name):
    rows=[claim_record(r['subject'],'code_behavior',r['object'],r['evidence'],name) for r in facts]
    rows.append(claim_record('admin_view_only_user','risk_stage',stage,
                             f"{prov['commit']}:{prov['path']}",name))
    activity=('implicit_m2m_edit_gated_by_view' if stage=='view_permission_controls_mutation' else
              'implicit_m2m_edit_requires_change' if stage=='change_permission_controls_mutation' else 'unresolved')
    rows.append(claim_record('admin_view_only_user','activity',activity,
                             f"{prov['commit']}:{prov['path']}",name))
    return {'claims':rows,'meta':{'commit':prov['commit'],'path':prov['path'],
            'sha256':prov['sha256'],'artifact_id':name}}

ISSUE_ART={'claims':[policy_claim,claim_record('admin_view_only_user','risk_stage','policy_expectation',ISSUE_URL,'reported_issue')],
           'meta':{'kind':'issue_report','reported_at':ISSUE_TIME,'source':ISSUE_URL}}
PRE_ART=build_artifact(stage_before,base_facts,before_prov,'source_base')
FIX_ART=build_artifact(stage_after,fix_facts,after_prov,'source_fix_reference_only')

THREAD_DETECTOR=nx.DiGraph()
ProjectAndAnnotateThreadEdge(THREAD_DETECTOR,ISSUE_ART,PRE_ART,
    {'relation':'reported_security_requirement_vs_pre_fix_implementation',
     'checkpoint':'after_issue_report','reference_only':False},
    'Reported security expectation','Pre-fix implementation')
THREAD_REFERENCE=THREAD_DETECTOR.copy()
ProjectAndAnnotateThreadEdge(THREAD_REFERENCE,PRE_ART,FIX_ART,
    {'relation':'historical_fix_evolution','reference_only':True,
     'checkpoint':'after_fix'},'Pre-fix implementation','Fix implementation (reference)')
# Test functions are derived from +def lines of the SWE-bench regression patch,
# NOT executed and not supplied to the pre-fix detector.
TEST_NAMES=sorted(set(re.findall(r'^\+\s+def\s+(test_inline_(?:add|change)_m2m_view_only_perm)\(',
                              str(case['test_patch']),re.MULTILINE)))
assert len(TEST_NAMES)==2, 'Expected two new explicit view-only regression test cases'
TEST_ART={'claims':[claim_record('django_admin_inline', 'test_requirement',name,
                                 'SWE-bench test patch: tests/admin_inlines/tests.py',
                                 'reference_regression_tests') for name in TEST_NAMES],
          'meta':{'reference_only':True,'test_patch_sha256':sha256(str(case['test_patch'])),
                  'execution_status':'NOT_RUN'}}
ProjectAndAnnotateThreadEdge(THREAD_REFERENCE,FIX_ART,TEST_ART,
    {'relation':'regression_tests_added_by_fix','reference_only':True,
     'checkpoint':'after_fix','tests_executed':False},
    'Fix implementation (reference)','Regression test patch (reference)')

rows=[]
for artifact_name,art in [('reported_issue',ISSUE_ART),('source_base',PRE_ART),('source_fix_reference',FIX_ART),('regression_tests_reference',TEST_ART)]:
    for c in art['claims']:
        rows.append({'artifact':artifact_name,'subject':c['s'],'predicate':c['p'],
                     'object':c['o'],'qualifiers':json.dumps(c['q'],sort_keys=True),
                     'evidence':' | '.join(c['evidence']),
                     'reference_only':artifact_name in ('source_fix_reference','regression_tests_reference')})
pd.DataFrame(rows).to_csv(OUT/'semantic_claim_inventory.csv',index=False)

transition_rows=[]
for label,g in [('at_issue_report',THREAD_DETECTOR),('historical_reference',THREAD_REFERENCE)]:
    for u,v,d in g.edges(data=True):
        delta=get_edge_delta(g,u,v)
        transition_rows.append({'view':label,'source':u,'target':v,
            'added':len(delta.added),'removed':len(delta.removed),
            'modified':len(delta.modified),'reference_only':'reference' in v.lower()})
transition_df=pd.DataFrame(transition_rows)
transition_df.to_csv(OUT/'semantic_delta_summary.csv',index=False)
TIMERS['thread_construction_seconds']=time.perf_counter()-t0
print('Detector thread:',THREAD_DETECTOR.number_of_nodes(),'nodes,',THREAD_DETECTOR.number_of_edges(),'edge')
print('Reference thread:',THREAD_REFERENCE.number_of_nodes(),'nodes,',THREAD_REFERENCE.number_of_edges(),'edges')
print('Post-fix test methods (reference only):',TEST_NAMES)
display(transition_df)

## 6. Independent SIC configuration and evidence register

**SIC-DJANGO-VIEW-ONLY** expresses the rule that view-only users must not be able to add/change/delete implicitly generated ManyToMany inline relations. Stage mapping is derived from AST-observed permission branches, while the numeric factors are *explicit, documented analyst settings*. The structured formula and fusion weights are unchanged from the original case studies.

This configuration is **illustrative, not validated on an independent held-out corpus**. We export both the numbers and their rationale so a later independent reviewer can inspect them. We do not revise them after seeing the hybrid score.

In [ ]:
SIC_CONFIG={
 'case_id':TASK_ID,
 'status':'ILLUSTRATIVE_ANALYST_CONFIGURATION_NOT_INDEPENDENT_VALIDATION',
 'sic':{'id':'SIC-DJANGO-VIEW-ONLY',
        'description':'View-only admin users must not modify implicit ManyToMany inline relations',
        'subjects':['admin_view_only_user']+[f'InlineModelAdmin.{m}' for m in METHODS],
        'predicates':['policy_constraint','risk_stage','activity','code_behavior'],
        'epsilon':0.25,'lambda_p':0.5,'w_p':1.0},
 'constraint_rules':{
    'SATISFIED':['policy_expectation','change_permission_controls_mutation'],
    'SUSPECT':[],
    'VIOLATION':['view_permission_controls_mutation']},
 'stage_contradiction':{'policy_expectation':0.0,
                         'view_permission_controls_mutation':1.0,
                         'change_permission_controls_mutation':0.0},
 'stage_operationalization':{'policy_expectation':0.0,
                              'view_permission_controls_mutation':0.75,
                              'change_permission_controls_mutation':0.0},
 'activity_harm':{'implicit_m2m_edit_gated_by_view':0.75,
                   'implicit_m2m_edit_requires_change':0.0},
 'factor_weights':{'C':0.4,'O':0.2,'H':0.4},
 'rationale':{
   'C_view': '1.0: explicit contradiction of the view-only mutation policy in the guarded source branch',
   'O_view': '0.75: delegation exists in executable code; exploit not reproduced in this notebook',
   'H_view': '0.75: unauthorised relationship modification, analyst-assigned ordinal severity, not calibrated',
   'C_O_H_after': '0.0 for source-backed change-permission guard, not a measured guarantee of complete safety',
   'lambda_epsilon': '0.50 and 0.25 reused from the submitted case studies, not tuned on Django'},
 'assessment_cutoff':'after_issue_report_and_before_fix',
 'limitations':['Issue-derived SIC is not proof of detection before disclosure',
                'Ordinal C/O/H values require independent validation',
                'No behavioural test execution in this notebook',
                'No independent claim-extraction benchmark']}
save_json('django_sic_config_EXPLORATORY.json',SIC_CONFIG)
P=SIC(**SIC_CONFIG['sic'])
assert P.lambda_p==0.5 and P.epsilon==0.25

evidence_register=[
 {'id':'issue','role':'reported_requirement','source':ISSUE_URL,'reference_only':False,
  'visible_after_report':True,'sha256':sha256(str(case['problem_statement']))},
 {'id':'base_code','role':'evaluated_implementation','source':before_prov['url'],'reference_only':False,
  'visible_after_report':True,'sha256':before_prov['sha256']},
 {'id':'fix_code','role':'later_reference','source':after_prov['url'],'reference_only':True,
  'visible_after_report':False,'sha256':after_prov['sha256']},
 {'id':'fix_patch','role':'later_reference','source':'SWE-bench human patch','reference_only':True,
  'visible_after_report':False,'sha256':sha256(str(case['patch']))},
 {'id':'regression_tests','role':'later_reference','source':'SWE-bench test patch','reference_only':True,
  'visible_after_report':False,'sha256':sha256(str(case['test_patch']))}]
pd.DataFrame(evidence_register).to_csv(OUT/'evidence_register.csv',index=False)
assert not any(e['reference_only'] and e['visible_after_report'] for e in evidence_register)
print('SIC:',P.id,'| Stage at base:',stage_before,'| Stage after fix (reference only):',stage_after)
print('C/O/H view-only case:', 1.0,0.75,0.75,
      '=> structured =',0.4*1+0.2*0.75+0.4*0.75)

## 7. Rich claim-aware digital-thread figures

The first figure is the **detector-visible** evidence/claim view (issue + pre-fix code), while the second is the **historical reference** view (includes the later fix and test patch). Distinct node and edge types are shown and exported in machine-readable GraphML. They are *visual projections of the S³DT digital thread*, not new stages or additional framework algorithms.

The reference regression tests were extracted from a published test **patch**; this notebook does not claim they were executed.

In [ ]:
t0=time.perf_counter()
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

COLORS={'issue':'#DDEAFF','sic':'#FFEAC7','source':'#DEF3E8','claim':'#F0E9FF',
        'stage':'#E3F3FF','delta':'#F9E7E7','decision':'#E2F2EA',
        'test':'#FFEDCC','reference':'#E8E9ED'}

G_VIS_DET=nx.DiGraph()
def visnode(g,k,label,typ,subtitle='',ref=False):
    g.add_node(k,label=label,category=typ,subtitle=subtitle,reference_only=bool(ref))
def visedge(g,a,b,kind,ref=False):
    g.add_edge(a,b,relation=kind,reference_only=bool(ref))

visnode(G_VIS_DET,'issue','Django ticket #30289','issue','View-only edit reported')
visnode(G_VIS_DET,'sic','SIC: view-only ⇒ no edits','sic','Access-control rule')
visnode(G_VIS_DET,'base','Pre-fix options.py','source',BASE_COMMIT[:12])
visnode(G_VIS_DET,'claim_add','ADD check','claim','delegates to view')
visnode(G_VIS_DET,'claim_change','CHANGE check','claim','delegates to view')
visnode(G_VIS_DET,'claim_delete','DELETE check','claim','delegates to view')
visnode(G_VIS_DET,'stage','Risk stage','stage','View gates mutation')
visnode(G_VIS_DET,'delta','Semantic delta','delta','Policy / code mismatch')
visnode(G_VIS_DET,'assessment','S³DT assessment','decision','SIC + hybrid scores')

for a,b,kind in [
 ('issue','sic','defines expectation'),('issue','base','reported concern'),
 ('base','claim_add','AST claim'),('base','claim_change','AST claim'),('base','claim_delete','AST claim'),
 ('claim_add','stage','stage evidence'),('claim_change','stage','stage evidence'),
 ('claim_delete','stage','stage evidence'),('sic','delta','constraint scope'),
 ('stage','delta','aligned semantic claims'),('delta','assessment','evaluated by SIC')]:
    visedge(G_VIS_DET,a,b,kind)

G_VIS_REF=G_VIS_DET.copy()
visnode(G_VIS_REF,'fix','Later fixing commit','reference','8335d59200e4',True)
visnode(G_VIS_REF,'fix_claim','Change-permission guard','reference','ADD / CHANGE / DELETE',True)
visnode(G_VIS_REF,'tests','Regression test patch','test','View-only assertions',True)
visnode(G_VIS_REF,'fix_delta','Pre-fix → fix delta','reference','Recorded claim modifications',True)
for a,b,kind in [('base','fix','historical repair'),('fix','fix_claim','AST claims'),
                  ('fix','tests','introduced tests'),('fix_claim','fix_delta','claim changes'),
                  ('stage','fix_delta','claim changes')]: visedge(G_VIS_REF,a,b,kind,True)

# Exact export of graph semantics before drawing. Node and edge labels remain traceable.
for key,fact in [('claim_add',base_facts[0]),('claim_change',base_facts[1]),('claim_delete',base_facts[2])]:
    for gv in (G_VIS_DET,G_VIS_REF):
        gv.nodes[key]['evidence']=fact['evidence']
for gv in (G_VIS_DET,G_VIS_REF):
    gv.nodes['issue']['evidence']=ISSUE_URL
    gv.nodes['sic']['policy_source']=ISSUE_URL
    gv.nodes['base']['evidence']=before_prov['url']
G_VIS_REF.nodes['fix']['evidence']=after_prov['url']
G_VIS_REF.nodes['tests']['evidence']='SWE-bench test patch: tests/admin_inlines/tests.py'
for name,g in [('detector',G_VIS_DET),('reference',G_VIS_REF)]:
    nx.write_graphml(g,OUT/f'case3_{name}_claim_graph.graphml')

DETPOS={
 'issue':(.14,.77),'sic':(.14,.38),
 'base':(.49,.83),'claim_add':(.355,.60),'claim_change':(.495,.60),'claim_delete':(.635,.60),
 'stage':(.49,.29),'delta':(.82,.66),'assessment':(.82,.29)}
REFPOS={
 'issue':(.11,.79),'sic':(.11,.46),'base':(.35,.82),
 'claim_add':(.245,.61),'claim_change':(.35,.61),'claim_delete':(.455,.61),
 'stage':(.35,.29),'delta':(.60,.65),'assessment':(.60,.33),
 'fix':(.83,.82),'fix_claim':(.83,.60),'tests':(.83,.35),'fix_delta':(.60,.13)}

def rich_graph(g,pos,path,title,reference=False):
    fig,ax=plt.subplots(figsize=(16,8.5))
    fig.patch.set_facecolor('#FFFFFF');ax.set_facecolor('#FFFFFF')
    ax.set_xlim(0,1);ax.set_ylim(0,1);ax.axis('off')
    ax.set_title(title,fontsize=18,fontweight='bold',loc='left',pad=22,color='#1A3545')
    bgzones=([(.025,.025,.24,.91,'SECURITY EXPECTATION','#F2F6FF'),
              (.28,.025,.42,.91,'IMPLEMENTATION AND CLAIMS','#F6FBF8'),
              (.73,.025,.25,.91,'SEMANTIC ASSESSMENT','#F8F9FD')]
             if not reference else
             [(.015,.025,.18,.91,'SECURITY INTENT','#F2F6FF'),
              (.22,.025,.26,.91,'PRE-FIX CODE','#F6FBF8'),
              (.51,.025,.20,.91,'SEMANTIC CHANGE','#FCF7F5'),
              (.74,.025,.245,.91,'FIX & TEST EVIDENCE','#F5F5F8')])
    for x,y,w,h,label,fill in bgzones:
        ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.01,rounding_size=.02',
                    facecolor=fill,edgecolor='#DFE5EB',linewidth=1.0,zorder=0))
        ax.text(x+.012,.925,label,fontsize=9.2,fontweight='bold',color='#677588')
    # Edge paths: route thin arrows behind nodes. Draw selected edges to limit clutter.
    for u,v,d in g.edges(data=True):
        x1,y1=pos[u];x2,y2=pos[v]
        if abs(x1-x2)+abs(y1-y2)<.015: continue
        isref=d.get('reference_only',False)
        ax.add_patch(FancyArrowPatch((x1,y1),(x2,y2),arrowstyle='-|>',
              mutation_scale=11,lw=1.35,color='#A2ABB8' if isref else '#6C8193',
              linestyle='--' if isref else '-',connectionstyle='arc3,rad=0.08',
              shrinkA=28,shrinkB=28,zorder=1,alpha=.9))
    # Nodes use rounded cards with a single title and short evidence qualifier.
    baseW=(.115 if reference else .150);height=.086 if reference else .100
    for key,d in g.nodes(data=True):
        x,y=pos[key]; cat=d['category'];w=baseW
        if cat=='claim': w=(.093 if reference else .117)
        if key in ('issue','sic','assessment','base','fix'):w+=.010
        patch=FancyBboxPatch((x-w/2,y-height/2),w,height,
                boxstyle='round,pad=0.010,rounding_size=.018',
                facecolor=COLORS.get(cat,'#FFFFFF'),edgecolor='#667C8A',linewidth=1.2,zorder=3)
        ax.add_patch(patch)
        label=d['label']; sub=d.get('subtitle','')
        if len(label)>26: label=label[:24]+'…'
        ax.text(x,y+.013,label,ha='center',va='center',fontsize=8.6 if reference else 9.4,
                fontweight='bold',color='#1D3545',zorder=4)
        if sub:
            sub=sub if len(sub)<=32 else sub[:30]+'…'
            ax.text(x,y-.020,sub,ha='center',va='center',fontsize=7.0 if reference else 8.0,
                    color='#486276',zorder=4)
    ax.text(.025,-.005,('Solid: evidence in the issue/pre-fix checkpoint. Dashed: later repair/test reference.'
                        if reference else 'Only issue evidence and pre-fix source are detector-visible. No later fix or tests are used.'),
            transform=ax.transAxes,fontsize=9.5,color='#5D6975')
    fig.tight_layout(pad=2)
    for suffix in ('.png','.svg'):
        fig.savefig(OUT/(path+suffix),dpi=220,bbox_inches='tight',facecolor=fig.get_facecolor())
    display(fig);plt.close(fig)

rich_graph(G_VIS_DET,DETPOS,'case3_detector_claim_graph',
          'Case Study 3 — Django detector-visible semantic thread')
rich_graph(G_VIS_REF,REFPOS,'case3_reference_claim_graph',
          'Case Study 3 — Historical reference and correction',reference=True)
TIMERS['visualisation_seconds']=time.perf_counter()-t0

## 8. Execute the released S³DT scoring on the case-specific SIC

- The **released** `ScoreClassifyExplain` routine computes normalized MiniLM cosine distance and the same hybrid equation. We supply **only case-specific hooks** (`evaluate_constraint`, `risk_weighted_structured_delta`, `render_summary`) as the original cases do.
- We report both the **SIC outcome** (policy rule) and the **score-only comparison label**, because the submitted case-study notebooks use different labelling pathways.
- The primary scoring transition is **reported expectation → pre-fix implementation**. The historical **pre-fix → fix** comparison occurs only at the post-fix checkpoint and is explicitly marked reference-only relative to the earlier detector.
- If embeddings cannot load, structured factors remain available, but the table reports `NOT_RUN` and *does not* report fabricated hybrid values.

In [ ]:
def stage_of(C):
    vals=[c.o for c in C.claims if c.p=='risk_stage']
    return vals[0] if len(vals)==1 else 'unresolved'

def evaluate_case_constraint(p,Ci,Cj,delta):
    stage=stage_of(select_claims(Cj,p))
    for label in ('VIOLATION','SUSPECT','SATISFIED'):
        if stage in SIC_CONFIG['constraint_rules'][label]:return label
    raise ValueError(f'Risk stage not configured: {stage}')

def structured_case_risk(delta,p,Cj_p):
    st=stage_of(Cj_p)
    if st not in SIC_CONFIG['stage_contradiction'] or st not in SIC_CONFIG['stage_operationalization']:
        raise ValueError(f'Risk stage cannot be scored: {st}')
    C=float(SIC_CONFIG['stage_contradiction'][st]);O=float(SIC_CONFIG['stage_operationalization'][st])
    acts=[c.o for c in Cj_p.claims if c.p=='activity']
    if any(a not in SIC_CONFIG['activity_harm'] for a in acts):raise ValueError('Unmapped activity: '+repr(acts))
    H=max([float(SIC_CONFIG['activity_harm'][a]) for a in acts],default=0.0)
    assert all(0<=x<=1 for x in (C,O,H))
    return 0.4*C+0.2*O+0.4*H

def render_case_summary(Cp):
    order={'policy_constraint':0,'risk_stage':1,'code_behavior':2,'activity':3}
    ordered=sorted([c for c in Cp.claims if c.p in order],
                   key=lambda c:(order[c.p],c.s,c.o))
    return ' '.join(f'{c.s.replace("_"," ")}: {c.p.replace("_"," ")} = {c.o.replace("_"," ")}.'
                    for c in ordered) or '<empty>'

# This helper reports C/O/H even when model weights cannot be loaded.
def score_factor_record(C):
    stage=stage_of(select_claims(C,P));acts=[c.o for c in select_claims(C,P).claims if c.p=='activity']
    Cval=SIC_CONFIG['stage_contradiction'][stage];Oval=SIC_CONFIG['stage_operationalization'][stage]
    Hval=max([SIC_CONFIG['activity_harm'][a] for a in acts],default=0.0)
    return stage,Cval,Oval,Hval,float(0.4*Cval+0.2*Oval+0.4*Hval)

CHECKPOINTS=[('reported_requirement_to_pre_fix',THREAD_DETECTOR,
              'Reported security expectation','Pre-fix implementation','detector_eligible'),
             ('pre_fix_to_fix_reference',THREAD_REFERENCE,
              'Pre-fix implementation','Fix implementation (reference)','post_fix_reference_only')]

from contextlib import contextmanager
@contextmanager
def installed_case_hooks():
    names=['evaluate_constraint','risk_weighted_structured_delta','render_summary']
    saved={key:globals()[key] for key in names}
    globals().update(evaluate_constraint=evaluate_case_constraint,
                     risk_weighted_structured_delta=structured_case_risk,
                     render_summary=render_case_summary)
    try:yield
    finally:globals().update(saved)

# Verify all scoped stage-to-decision mappings before any model inference.
for key,g,u,v,visibility in CHECKPOINTS:
    outcome=evaluate_case_constraint(P,get_claims(g,u),get_claims(g,v),get_edge_delta(g,u,v))
    print(key,'SIC outcome=',outcome,'stage=',stage_of(get_claims(g,v)))

In [ ]:
t0=time.perf_counter()
embedding_ready=False
try:
    from sentence_transformers import SentenceTransformer
    if _EMBEDDER is None:
        _EMBEDDER=SentenceTransformer('all-MiniLM-L6-v2',device='cpu')
    embedding_ready=True
    print('Loaded all-MiniLM-L6-v2; original S³DT MiniLM scoring is enabled.')
except Exception as exc:
    print('MiniLM model unavailable; no embedding/hybrid results will be invented.')
    print(type(exc).__name__+':',str(exc)[:450])
TIMERS['embedding_model_load_seconds']=time.perf_counter()-t0

In [ ]:
t0=time.perf_counter()
TRANSITIONS=[];EVENTS=[]
for key,g,u,v,visibility in CHECKPOINTS:
    Ci=get_claims(g,u);Cj=get_claims(g,v);delta=get_edge_delta(g,u,v)
    result=evaluate_case_constraint(P,Ci,Cj,delta)
    stage,C,O,H,d_structured=score_factor_record(Cj)
    delta_scoped=restrict_delta_to_scope(delta,P)
    row={'transition_id':key,'source':u,'target':v,'checkpoint':visibility,
         'sic_id':P.id,'sic_outcome':result,'stage':stage,
         'C':C,'O':O,'H':H,'D_structured':round(d_structured,6),
         'D_embed':np.nan,'D_hybrid':np.nan,'score_only_label':'NOT_RUN',
         'detector_output':'NOT_RUN','added':len(delta_scoped.added),
         'removed':len(delta_scoped.removed),'modified':len(delta_scoped.modified),
         'score_status':'EMBEDDING_MODEL_UNAVAILABLE'}
    if embedding_ready:
        with installed_case_hooks():
            source_summary=render_case_summary(select_claims(Ci,P))
            destination_summary=render_case_summary(select_claims(Cj,P))
            x=_EMBEDDER.encode([source_summary],normalize_embeddings=True)[0]
            y=_EMBEDDER.encode([destination_summary],normalize_embeddings=True)[0]
            d_embed=calibrate_embed(float(1-np.dot(x,y)),P)
            d_hybrid=P.lambda_p*d_structured+(1-P.lambda_p)*d_embed
            row['D_embed']=round(d_embed,6);row['D_hybrid']=round(d_hybrid,6)
            row['score_only_label']=('PASS' if d_hybrid<=.25 else
                                      'SUSPECT' if d_hybrid<.65 else 'VIOLATION')
            row['score_status']='EXPLORATORY_ANALYST_CONFIGURATION'
            if result in ('VIOLATION','SUSPECT'):
                cand=DriftCandidate(source=u,target=v,constraint_id=P.id,result=result,delta=delta)
                event=ScoreClassifyExplain(g,cand,P)
                assert abs(event.severity-d_hybrid)<1e-5, 'Original S3DT scorer disagrees with component calculations'
                row['detector_output']=event.drift_type
                EVENTS.append({'checkpoint':visibility,**event.model_dump()})
            else:
                row['detector_output']='not emitted (SIC satisfied)'
    TRANSITIONS.append(row)

result_df=pd.DataFrame(TRANSITIONS)
result_df.to_csv(OUT/'case3_transition_results.csv',index=False)
save_json('case3_explainable_drift_events.json',EVENTS)
TIMERS['sic_and_score_seconds']=time.perf_counter()-t0
print('RESULT STATUS:', 'EXPLORATORY CASE-SPECIFIC SCORES (not independent accuracy)' if embedding_ready
      else 'STRUCTURED ONLY — model not available')
display(result_df[['transition_id','checkpoint','sic_outcome','C','O','H','D_structured',
                   'D_embed','D_hybrid','score_only_label','detector_output']])

## 8A. Real post-fix development controls in the **same** S³DT digital thread

To examine irrelevant development activity under **SIC-DJANGO-VIEW-ONLY**, follow seven **consecutive real Django commits** that descend from the independently documented fixing commit. They include tests, documentation, translations, and a distinct admin bug fix. Every commit is identified by its immutable Git SHA and parent.

- These commits **follow the fix**; they are not inserted between the vulnerable commit and its fix.
- They are **reference/control history**, not evidence available to the pre-fix detector.
- Each snapshot is checked against its own commit-pinned `django/contrib/admin/options.py`. A commit qualifies as an ordinary control **for this SIC** only when the three guarded permission behaviours remain unchanged.
- The full digital thread records a *maintenance-change claim* for each actual Git commit. These claims create genuine full-graph semantic deltas, but fall **outside** the access-control SIC's scope. This tests whether unrelated changes are ignored rather than pretending nothing changed.
- The risk parameters **remain exactly those in the exploratory Django SIC configuration**. No new scoring rule or post-hoc calibration is introduced. A PASS on these controls does not establish the absence of all other vulnerabilities.

Optional offline cache: `Django_PostFix_Control_Snapshots.zip` (uploaded alongside the two main input files). If it is absent, the notebook fetches exact pinned revisions from the public Django GitHub repository.

In [ ]:
# Exact first-parent descendant chain checked in the historical Django Git repository.
CONTROL_SHAS = [
    '0027ca76c7d1b4eba1578bf3d132b8c0ff87f829',
    'f9f09a8a0259e75a5009ce04e662a5a759f8a12f',
    '7191beec4e2140bf2d6ff7e6ad2e1c25913c9c48',
    '3d465f77e5ca382a4db99ed4ec44b1941cb60aff',
    '3e01aab5335394201701710d7fcd67f523878c5b',
    '93daed25a41b5cf2f7804d91630caa1cd0786a9a',
    '7cbcf2e2cbca1acfccfd973f746a9e6abfc0052e',
]
CTRL_CACHE = INPUTS / 'Django_PostFix_Control_Snapshots.zip'
CONTROL_METADATA=[];CONTROL_SOURCES={};CONTROL_SOURCE_MODES={}

def fetch_json_commit(sha):
    url=f'https://api.github.com/repos/django/django/commits/{sha}'
    req=urllib.request.Request(url,headers={
        'User-Agent':'S3DT-TDSC-Historical-Controls/1.0',
        'Accept':'application/vnd.github+json'})
    with urllib.request.urlopen(req,timeout=45) as r:
        obj=json.loads(r.read())
    if obj['sha']!=sha:
        raise ValueError('Commit SHA mismatch from GitHub')
    return {
        'commit':sha,
        'parent':obj['parents'][0]['sha'],
        'message':obj['commit']['message'].splitlines()[0],
        'committed_at':obj['commit']['committer']['date'],
        'changed_files':[f['filename'] for f in obj.get('files',[])],
        'github_commit_url':obj['html_url'],
        'metadata_mode':'github_commit_api_verified'
    }

if CTRL_CACHE.exists():
    with zipfile.ZipFile(CTRL_CACHE) as zz:
        raw=json.loads(zz.read('post_fix_control_manifest.json'))
        assert [x['commit'] for x in raw]==CONTROL_SHAS
        for rec in raw:
            sha=rec['commit']
            source=zz.read('options_snapshots/'+sha+'.py').decode('utf-8')
            assert sha256(source)==rec['source_sha256']
            CONTROL_SOURCES[sha]=source
            CONTROL_SOURCE_MODES[sha]='git_object_verified_cache'
            CONTROL_METADATA.append({**rec,'metadata_mode':'git_object_verified_cache'})
else:
    for sha in CONTROL_SHAS:
        rec=fetch_json_commit(sha)
        source_url=f'https://raw.githubusercontent.com/django/django/{sha}/{SOURCE_PATH}'
        req=urllib.request.Request(source_url,headers={'User-Agent':'S3DT-TDSC-Historical-Controls/1.0'})
        with urllib.request.urlopen(req,timeout=60) as r:
            source=r.read().decode('utf-8')
        rec['source_sha256']=sha256(source)
        rec['source_path']=SOURCE_PATH
        rec['changed_file_count']=len(rec['changed_files'])
        CONTROL_SOURCES[sha]=source
        CONTROL_SOURCE_MODES[sha]='commit_pinned_raw_download'
        CONTROL_METADATA.append(rec)

assert len(CONTROL_METADATA)==7
previous=FIX_COMMIT
for pos,rec in enumerate(CONTROL_METADATA):
    sha=rec['commit']
    assert rec['parent']==previous, f'Control chain is not contiguous at {sha}'
    assert sha==CONTROL_SHAS[pos]
    assert SOURCE_PATH not in rec['changed_files'],f'SIC-relevant source modified by {sha}'
    assert sha256(CONTROL_SOURCES[sha])==rec['source_sha256']
    ast.parse(CONTROL_SOURCES[sha])
    previous=sha

# This is a Git ancestry ordering, NOT a sort by author timestamp.
print('Seven consecutive descendants of the fixed Django state, all SHA-verified:')
display(pd.DataFrame([{
    'step':idx+1,'commit':r['commit'][:12], 'changed_files':r['changed_file_count'],
    'message':r['message'][:80]}
    for idx,r in enumerate(CONTROL_METADATA)]))

In [ ]:
## NOT A SECOND EXPERIMENT: these are controls evaluated by the original S³DT code.
t0=time.perf_counter()
THREAD_EXTENDED=THREAD_REFERENCE.copy()
CTRL_ARTS=[];CONTROL_RESULTS=[];CONTROL_EVIDENCE=[];maintenance_claims=[]
previous_art=FIX_ART
previous_node='Fix implementation (reference)'
fixed_guard={(r['subject'],r['predicate'],r['object']) for r in fix_facts}

for idx,(rec) in enumerate(CONTROL_METADATA,1):
    sha=rec['commit']
    prov={'commit':sha,'path':SOURCE_PATH,
          'sha256':rec['source_sha256'],
          'url':f'https://raw.githubusercontent.com/django/django/{sha}/{SOURCE_PATH}'}
    facts,observed_stage=analyze_source(CONTROL_SOURCES[sha],f'control_{idx:02}',prov)
    observed_guard={(r['subject'],r['predicate'],r['object']) for r in facts}
    unchanged=(observed_stage==stage_after and observed_guard==fixed_guard and
               rec['source_sha256']==after_prov['sha256'])
    # Do not silently relabel a changed permission guard as an ordinary control.
    assert unchanged, f'Control at {sha} changes the relevant permission guard'

    # Commit-specific claim, with a UNIQUE subject key: maintain cumulative evidence
    # without overwriting earlier claims under exact-key alignment.
    maintenance_claims.append({
        's':'django_maintenance_'+sha[:12],
        'p':'maintenance_change',
        'o':rec['message'],
        'q':{'repository':'django/django','kind':'post_fix_ordinary_commit'},
        'gamma':1.0,
        'evidence':[rec['github_commit_url']],
        'provenance':{'commit':sha,'parent':rec['parent'],
                      'changed_file_count':rec['changed_file_count'],
                      'changed_files':rec['changed_files']}
    })
    art=build_artifact(observed_stage,facts,prov,f'control_{idx:02}')
    art['claims'].extend(copy.deepcopy(maintenance_claims))
    art['meta'].update({'reference_only':True,'ordinary_control_for_sic':P.id,
                        'commit_message':rec['message'],'parent_commit':rec['parent'],
                        'changed_files':rec['changed_files']})
    node=f'Ordinary Django change {idx:02} [{sha[:10]}]'
    ProjectAndAnnotateThreadEdge(
        THREAD_EXTENDED,previous_art,art,
        {'relation':'real_git_parent_transition','parent_commit':rec['parent'],
         'commit':sha,'reference_only':True,'ordinary_control':True},
        previous_node,node)
    delta=get_edge_delta(THREAD_EXTENDED,previous_node,node)
    scoped=restrict_delta_to_scope(delta,P)
    # The full digital thread MUST register the actual maintenance change.
    assert len(delta.added)>=1
    assert len(scoped.added)==len(scoped.removed)==len(scoped.modified)==0
    Ci=get_claims(THREAD_EXTENDED,previous_node)
    Cj=get_claims(THREAD_EXTENDED,node)
    result=evaluate_case_constraint(P,Ci,Cj,delta)
    assert result=='SATISFIED'
    stage,C,O,H,D=score_factor_record(Cj)
    assert stage==stage_after and D==0.0
    # Score all transitions for the COMPARISON table. As in the original detector,
    # SATISFIED transitions do NOT enter ScoreClassifyExplain.
    d_embed=float('nan');d_hybrid=float('nan');score_label='NOT_RUN'
    if embedding_ready:
        before_summary=render_case_summary(select_claims(Ci,P))
        after_summary=render_case_summary(select_claims(Cj,P))
        vecs=_EMBEDDER.encode([before_summary,after_summary],normalize_embeddings=True)
        d_embed=calibrate_embed(float(1.0-np.dot(vecs[0],vecs[1])),P)
        d_hybrid=float(P.lambda_p*D+(1.0-P.lambda_p)*d_embed)
        score_label=('PASS' if d_hybrid<=.25 else 'SUSPECT' if d_hybrid<.65 else 'VIOLATION')
    result_row={
        'transition_id':f'ordinary_post_fix_{idx:02}',
        'source':previous_node,'target':node,
        'checkpoint':'post_fix_ordinary_control_reference',
        'sic_id':P.id,'sic_outcome':result,'stage':stage,
        'C':C,'O':O,'H':H,'D_structured':round(D,6),
        'D_embed':round(d_embed,6),'D_hybrid':round(d_hybrid,6),
        'score_only_label':score_label,
        'detector_output':'not emitted (SIC satisfied)',
        'added':len(scoped.added),'removed':len(scoped.removed),
        'modified':len(scoped.modified),
        'full_delta_added':len(delta.added),
        'full_delta_removed':len(delta.removed),
        'full_delta_modified':len(delta.modified),
        'score_status':'CONTROL_POST_FIX_FROZEN_EXPLORATORY_SIC' if embedding_ready
                       else 'EMBEDDING_NOT_RUN',
    }
    assert score_label in ('PASS','NOT_RUN'),f'Unexpected control score: {sha}, {score_label}'
    TRANSITIONS.append(result_row)
    CONTROL_RESULTS.append(result_row)
    CONTROL_EVIDENCE.append({
        'order':idx,'commit':sha,'parent':rec['parent'],
        'message':rec['message'],'changed_file_count':rec['changed_file_count'],
        'changed_files':' | '.join(rec['changed_files']),
        'source_sha256':rec['source_sha256'],
        'fix_source_sha256':after_prov['sha256'],
        'guard_unchanged':unchanged,'sic_scoped_delta_count':0,
        'full_delta_count':len(delta.added)+len(delta.removed)+len(delta.modified),
        'status':'PASS_FOR_THIS_SIC_ONLY',
        'provenance_url':rec['github_commit_url'],
        'metadata_mode':rec['metadata_mode']
    })
    CTRL_ARTS.append((node,art,facts,prov))
    previous_art=art
    previous_node=node

# One continuous reference knowledge graph: issue → pre-fix → fix → real development.
assert THREAD_EXTENDED.number_of_nodes()==THREAD_REFERENCE.number_of_nodes()+7
assert 'Ordinary Django change 01 [' not in THREAD_DETECTOR
assert not any('Ordinary Django change ' in n for n in THREAD_DETECTOR)

result_df=pd.DataFrame(TRANSITIONS)
result_df.to_csv(OUT/'case3_transition_results.csv',index=False)
pd.DataFrame(CONTROL_RESULTS).to_csv(OUT/'case3_ordinary_control_results.csv',index=False)
pd.DataFrame(CONTROL_EVIDENCE).to_csv(OUT/'case3_ordinary_control_evidence.csv',index=False)
(OUT/'case3_historical_controls_manifest.json').write_text(
    json.dumps(CONTROL_METADATA,indent=2,ensure_ascii=False))

# Separate security-scoped claim inventory from changed-file evidence.
ctrl_claim_rows=[]
for node,art,facts,prov in CTRL_ARTS:
    for c in art['claims']:
        ctrl_claim_rows.append({'snapshot':node,'commit':prov['commit'],
            'subject':c['s'],'predicate':c['p'],'object':c['o'],
            'sic_relevant':(c['s'] in P.subjects and c['p'] in P.predicates),
            'evidence':' | '.join(c['evidence'])})
pd.DataFrame(ctrl_claim_rows).to_csv(OUT/'case3_ordinary_control_claim_inventory.csv',index=False)

TIMERS['ordinary_control_execution_seconds']=time.perf_counter()-t0
print('Real development controls:',len(CONTROL_RESULTS),
      '| Full semantic deltas:',sum(r['full_delta_count'] for r in CONTROL_EVIDENCE),
      '| Scoped changes:',sum(r['sic_scoped_delta_count'] for r in CONTROL_EVIDENCE))
display(result_df[['transition_id','sic_outcome','full_delta_added' if 'full_delta_added' in result_df else 'added',
                   'D_structured','D_embed','D_hybrid','detector_output']].fillna('—'))

## 8B. Integrated historical **knowledge graph** with ordinary development noise

The detector-visible graph remains restricted to the report-time evidence. The following **single reference-and-control graph** joins (i) that detector-visible evidence, (ii) the later repair and tests, and (iii) the seven actual, consecutive post-fix commits. Each normal change appears as a commit artifact with a provenance link, changed-file group, and constraint-scoped result. GraphML retains the detailed node/edge relations for audit; the figure is a readable projection of the same graph.

In [ ]:
t0=time.perf_counter()
# Full RDF-like node/edge topology for reproducibility; keep detector graph separate.
G_KG_FULL=G_VIS_REF.copy()
for idx,rec in enumerate(CONTROL_METADATA,1):
    ck='ordinary_commit_'+str(idx)
    ak='ordinary_changes_'+str(idx)
    claimk='permission_guard_snapshot_'+str(idx)
    visnode(G_KG_FULL,ck,f'Ordinary change {idx}', 'source',rec['commit'][:12],True)
    visnode(G_KG_FULL,ak,'Changed files', 'reference',f"{rec['changed_file_count']} files",True)
    visnode(G_KG_FULL,claimk,'Permission guard', 'claim','unchanged from fix',True)
    G_KG_FULL.nodes[ck].update({
        'commit':rec['commit'],'parent_commit':rec['parent'],
        'commit_title':rec['message'],'evidence':rec['github_commit_url'],
        'changed_files_json':json.dumps(rec['changed_files']),
        'security_scope':'post_fix_control_only'})
    G_KG_FULL.nodes[ak].update({
        'evidence':rec['github_commit_url'],
        'changed_files_json':json.dumps(rec['changed_files'])})
    G_KG_FULL.nodes[claimk].update({
        'evidence':f'https://raw.githubusercontent.com/django/django/{rec["commit"]}/{SOURCE_PATH}',
        'sha256':rec['source_sha256']})
    visedge(G_KG_FULL,ck,ak,'modifies_out_of_scope_artifacts',True)
    visedge(G_KG_FULL,ck,claimk,'retains_sic_claims',True)
    visedge(G_KG_FULL,claimk,'sic','satisfies_guard_condition',True)
    predecessor='fix' if idx==1 else 'ordinary_commit_'+str(idx-1)
    visedge(G_KG_FULL,predecessor,ck,'first_parent_git_history',True)

nx.write_graphml(G_KG_FULL,OUT/'case3_integrated_semantic_thread.graphml')
# The full S3DT graph records ClaimSets and ClaimDeltas which are not natively
# GraphML serializable. Emit an explicit, lossless JSON representation as well.
full_nodes=[];full_edges=[]
for u,attrs in THREAD_EXTENDED.nodes(data=True):
    full_nodes.append({'id':u,
        'claims':[c.model_dump() for c in attrs['claims'].claims],
        'metadata':attrs.get('meta',{})})
for u,v,attrs in THREAD_EXTENDED.edges(data=True):
    full_edges.append({'source':u,'target':v,
                       'delta':attrs['delta'].model_dump(),
                       'link_meta':attrs.get('meta',{})})
save_json('case3_integrated_semantic_thread.json',{'nodes':full_nodes,'edges':full_edges})

fig,ax=plt.subplots(figsize=(23,11.5));fig.patch.set_facecolor('white')
ax.set_xlim(0,1);ax.set_ylim(0,1);ax.axis('off')
ax.set_title('Case Study 3 — Django semantic integrity across real development history',
             fontsize=20,fontweight='bold',loc='left',pad=22,color='#223845')

# The upper lane follows the same security pipeline as the initial two cases.
sections=[(.025,.59,.950,.32,'SECURITY EVIDENCE AND CONSTRAINT EVALUATION','#F4F8FC'),
          (.025,.12,.950,.42,'REAL REPOSITORY EVOLUTION AFTER THE FIX','#F4F9F6')]
for x,y,w,h,label,fill in sections:
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=.009,rounding_size=.016',
        facecolor=fill,edgecolor='#D4DFE6',lw=1.1,zorder=0))
    ax.text(x+.015,y+h-.037,label,fontsize=10.8,fontweight='bold',color='#507084')

def card(x,y,heading,detail='',w=.135,h=.102,fill='#FFFFFF',edge='#547B90',size=10.2):
    ax.add_patch(FancyBboxPatch((x-w/2,y-h/2),w,h,
        boxstyle='round,pad=.007,rounding_size=.013',
        facecolor=fill,edgecolor=edge,lw=1.4,zorder=4))
    ax.text(x,y+.016,heading,ha='center',va='center',fontweight='bold',fontsize=size,color='#233E4A',zorder=5)
    if detail:
        ax.text(x,y-.022,detail,ha='center',va='center',fontsize=size-1.3,
                color='#516D7B',zorder=5)

def arrow(x0,y0,x1,y1,dotted=False,rad=0):
    ax.add_patch(FancyArrowPatch((x0,y0),(x1,y1),arrowstyle='-|>',mutation_scale=12,
             color='#8C9FAB',lw=1.6,linestyle='--' if dotted else '-',
             connectionstyle=f'arc3,rad={rad}',zorder=2,shrinkA=4,shrinkB=4))

# Assessment-time evidence, then independent reference correction.
card(.105,.77,'Django issue','Ticket #30289',w=.136,fill='#E5EEFF')
card(.285,.77,'Pre-fix source','view-only can edit',w=.164,fill='#E3F4E9')
card(.48,.77,'Semantic claims','ADD / CHANGE / DELETE',w=.175,fill='#F0E9FA')
card(.678,.77,'SIC evaluation','permission rule',w=.150,fill='#FFF0D6')
card(.864,.77,'Reference fix','change permission',w=.155,fill='#E9EDF1')
for xx0,xx1 in [(.178,.202),(.376,.389),(.572,.603),(.757,.785)]:arrow(xx0,.77,xx1,.77,dotted=(xx0>.7))
ax.text(.485,.650,'SIC: view-only users must not mutate implicit ManyToMany relations',
        ha='center',fontsize=11.2,color='#324E5B')
ax.text(.864,.637,'Later tests and repair are reference-only',ha='center',fontsize=9.8,color='#677582')

# Lower lane: one first-parent chain, not a reordered set of task descriptions.
coords=np.linspace(.095,.907,len(CONTROL_METADATA)+1)
y=.365
card(coords[0],y,'Fix applied',FIX_COMMIT[:10],w=.110,h=.090,
     fill='#DBEDE3',size=8.8)
for i,rec in enumerate(CONTROL_METADATA,1):
    x=coords[i]
    msg=rec['message']
    files=rec['changed_files']
    if all(k.startswith('docs/') for k in files): kind='Docs'
    elif all('/locale/' in k for k in files):kind='Translations'
    elif any(k.startswith('django/contrib/admin/') for k in files):kind='Other admin work'
    elif any(k.startswith('tests/') for k in files):kind='Tests'
    else:kind='Routine change'
    rec['scope_category']=kind
    color='#E8F0FA' if kind in ('Docs','Tests') else '#EDF2F6' if kind=='Translations' else '#E9F4EC'
    card(x,y,f'Commit {i}',rec['commit'][:10],w=.092,h=.090,fill=color,size=8.65)
    ax.text(x,.267,kind,ha='center',fontsize=9.6,fontweight='bold',color='#435E70')
    ax.text(x,.237,str(rec['changed_file_count'])+' changed files',ha='center',fontsize=8.8,color='#617586')
    arrow(coords[i-1]+.050,y,coords[i]-.050,y)

ax.text(.5,.154,
  '7 consecutive Git commits  •  7 genuine out-of-scope maintenance deltas  •  0 changes to SIC-scoped permission claims',
  ha='center',fontsize=11,fontweight='bold',color='#22674F')
ax.text(.035,.064,
  'Historical control states occur AFTER the repair; they are not available to the earlier detector.  '
  'PASS means no violation of this SIC was identified, not that the repository is free of security defects.',
  fontsize=10.0,color='#536674')
fig.tight_layout(pad=2.5)
for ext in ('.png','.svg'):
    fig.savefig(OUT/('case3_integrated_semantic_thread'+ext),
                dpi=240,bbox_inches='tight',facecolor='white')
display(fig);plt.close(fig)
TIMERS['integrated_graph_construction_seconds']=time.perf_counter()-t0
print('Extended S3DT thread:',THREAD_EXTENDED.number_of_nodes(),'nodes,',
      THREAD_EXTENDED.number_of_edges(),'edges')
print('Claim-aware graph:',G_KG_FULL.number_of_nodes(),'nodes,',G_KG_FULL.number_of_edges(),'edges')

## 9. Journal-style result table, plots and auditable explanation

Tables and bars are computed from actual model outputs; no hard-coded drift scores are used. The chart is labelled *exploratory* because C/O/H are analyst-assigned for this case. A confirmed defect in Django's issue tracker is reference evidence; a positive S³DT output is still **a separate result**.

In [ ]:
import textwrap
# Keep the numerical presentation compact for the paper; do not hide unresolved values.
columns=['transition_id','sic_outcome','D_structured','D_embed','D_hybrid','score_only_label','detector_output']
(OUT/'case3_result_table.md').write_text(result_df[columns].to_markdown(index=False),encoding='utf-8')
(OUT/'case3_result_table.tex').write_text(result_df[columns].to_latex(index=False,na_rep='N/A',
    float_format=lambda x:f'{x:.3f}'),encoding='utf-8')
if embedding_ready:
    bar_df=result_df.set_index('transition_id')[['D_structured','D_embed','D_hybrid']]
    ax=bar_df.plot(kind='bar',figsize=(15,6),rot=35,ylim=(0,1.05),width=.72)
    ax.set_ylabel('Drift score [0, 1]');ax.set_xlabel('Historical transition')
    ax.set_title('Case Study 3: structured, embedding and hybrid drift (EXPLORATORY)')
    ax.legend(loc='upper right');plt.xticks(ha='right');plt.tight_layout()
    plt.savefig(OUT/'case3_drift_scores.png',dpi=220,bbox_inches='tight');plt.close()
else:
    print('Full score chart omitted (no real MiniLM inference available).')

explanation_rows=[]
for e in EVENTS:
    explanation_rows.append({'source':e['source'],'target':e['target'],
      'checkpoint':e['checkpoint'],'drift_type':e['drift_type'],
      'score':e['severity'],'framework_explanation':e['explanation'],
      'claim_evidence':'; '.join(r['evidence'] for r in base_facts)})
pd.DataFrame(explanation_rows,
    columns=['source','target','checkpoint','drift_type','score','framework_explanation','claim_evidence']).to_csv(
    OUT/'case3_explanations.csv',index=False)
print('Result files:',[p.name for p in OUT.glob('case3_*')])

## 10. Integrity checks, runtime and reproducibility manifest

Tests fail if any later fix is in the pre-fix detector graph, if the expected structural claims are absent, or if drift scores leave the [0,1] interval. This is **not** a projection accuracy benchmark or a multi-case evaluation. Machine and environment details are recorded for the later reviewer-requested runtime analysis.

In [ ]:
assert 'Fix implementation (reference)' not in THREAD_DETECTOR
assert not any(d.get('reference_only',False) for _,d in G_VIS_DET.nodes(data=True))
assert stage_before=='view_permission_controls_mutation'
assert stage_after=='change_permission_controls_mutation'
assert len(base_facts)==3 and len(fix_facts)==3
if embedding_ready:
    assert result_df[['D_structured','D_embed','D_hybrid']].apply(lambda s:s.between(0,1).all()).all()
    assert (result_df.loc[0,'detector_output']!='not emitted (SIC satisfied)')
else:
    assert result_df['D_hybrid'].isna().all()

runtime=pd.DataFrame([{'stage':k,'elapsed_seconds':round(v,6)} for k,v in TIMERS.items()])
runtime.to_csv(OUT/'runtime_breakdown.csv',index=False)
manifest={'case_id':TASK_ID,'issue_source':ISSUE_URL,'issue_timestamp_from_dataset':ISSUE_TIME,
   'snapshot_timestamp_caveat':'Git commit date not independently verified against issue date',
   'base_commit':BASE_COMMIT,'reference_fix_commit':FIX_COMMIT,
   'source_provenance':[before_prov,after_prov],
   'original_s3dt_core_sha256':CORE_SHA256,
   'config_sha256':sha256((OUT/'django_sic_config_EXPLORATORY.json').read_bytes()),
   'config_status':SIC_CONFIG['status'],
   'claim_extraction':'deterministic Python AST, no manual extraction accuracy evaluated',
   'score_status':('exploratory numerical score from original S3DT MiniLM model' if embedding_ready
                   else 'no embedding/hybrid results; historical graph and structured scores only'),
   'model':'sentence-transformers/all-MiniLM-L6-v2',
   'model_revision':'not pinned; as in released S3DT notebook',
   'python':sys.version,'platform':platform.platform(),
   'environment':{p:metadata.version(p) for p in ['pandas','numpy','networkx','matplotlib','pydantic']},
   'timings':TIMERS,'limitations':SIC_CONFIG['limitations']}
save_json('reproducibility_manifest.json',manifest)
display(runtime)
print('Integrity checks passed. Scoring status:',manifest['score_status'])

## 11. Historical validation and independent evidence checks

The earlier scoring remains unchanged. The following cells provide **reference-only evidence**, a test replay, and an explicit assessment of remaining limitations. **The historical fix and regression-test patch are never introduced into the pre-fix S³DT detector graph.**

### 11.1 Validate provenance, SIC assignments and chronological evidence

This step audits the current exploratory SIC configuration rather than treating its assigned risk values as independent calibration. Where possible it obtains the GitHub commit timestamps to confirm the ordering of the snapshots.

In [ ]:
import subprocess, shutil, urllib.error
from datetime import datetime, timezone

AUDIT = OUT / 'validation'
AUDIT.mkdir(exist_ok=True, parents=True)

# Use the same values that were used to produce the pilot. No scores are changed.
config_path=OUT/'django_sic_config_EXPLORATORY.json'
assert config_path.exists(), 'Run the earlier S3DT cells first.'
cfg=json.loads(config_path.read_text())
rows=[]
for stage in cfg['stage_contradiction']:
    rows.append({
        'stage':stage,
        'constraint_rule':next((label for label,stages in cfg['constraint_rules'].items() if stage in stages),'UNMAPPED'),
        'contradiction_C':cfg['stage_contradiction'][stage],
        'operationalisation_O':cfg['stage_operationalization'][stage],
        'source_of_values':'analyst ordinal assignment from reported case',
        'independently_calibrated':False,
        'review_status':'REQUIRES_INDEPENDENT_REVIEW'
    })
pd.DataFrame(rows).to_csv(AUDIT/'sic_stage_review.csv', index=False)

# The two statements below are intentionally distinct.
assert len(base_facts)==3 and all(x['object']=='delegates_to_view_permission' for x in base_facts)
assert len(fix_facts)==3 and all(x['object']=='requires_related_model_change_permission' for x in fix_facts)
independent_source_checks=pd.DataFrame([
 {'snapshot':'base', 'method':f['subject'],'code_guard':f['object'],
  'evidence':f['evidence'],'role':'pre_fix_detector_evidence'} for f in base_facts] +
 [{'snapshot':'fix','method':f['subject'],'code_guard':f['object'],
  'evidence':f['evidence'],'role':'post_fix_reference_only'} for f in fix_facts]
)
independent_source_checks.to_csv(AUDIT/'source_permission_guard_checks.csv', index=False)

# GitHub commit API is used only to verify public provenance, not for pre-fix detection.
def github_commit_metadata(sha):
    url='https://api.github.com/repos/django/django/commits/'+sha
    try:
        req=urllib.request.Request(url,headers={'User-Agent':'S3DT-TDSC-reproduction/1.0',
                                                 'Accept':'application/vnd.github+json'})
        with urllib.request.urlopen(req,timeout=25) as response:
            obj=json.loads(response.read())
        return {'sha':sha,'status':'VERIFIED','commit_date':obj['commit']['committer']['date'],
                'parents':[x['sha'] for x in obj.get('parents',[])],
                'html_url':obj.get('html_url',url)}
    except Exception as exc:
        return {'sha':sha,'status':'UNAVAILABLE','error':type(exc).__name__+': '+str(exc)[:180]}

commit_meta=[github_commit_metadata(BASE_COMMIT), github_commit_metadata(FIX_COMMIT)]
issue_time = pd.Timestamp(ISSUE_TIME)
if all(x.get('status')=='VERIFIED' for x in commit_meta):
    base_time=pd.Timestamp(commit_meta[0]['commit_date']);fix_time=pd.Timestamp(commit_meta[1]['commit_date'])
    chronological_statement = 'VERIFIED' if base_time<fix_time else 'CONFLICT_REVIEW_REQUIRED'
    post_report_statement = 'VERIFIED' if issue_time<=fix_time else 'CONFLICT_REVIEW_REQUIRED'
else:
    chronological_statement='NOT_VERIFIED_NETWORK_REQUIRED'
    post_report_statement='NOT_VERIFIED_NETWORK_REQUIRED'
provenance_audit={'issue_timestamp':ISSUE_TIME,'base_commit':BASE_COMMIT,'fix_commit':FIX_COMMIT,
                  'commit_records':commit_meta,
                  'base_before_fix_chronology':chronological_statement,
                  'issue_before_fix':post_report_statement,
                  'assessment_scope':'post-issue-report analysis; no pre-disclosure detection claim'}
(AUDIT/'commit_chronology.json').write_text(json.dumps(provenance_audit,indent=2))
print('Source-backed permissions:',len(independent_source_checks),'(3 pre-fix, 3 post-fix)')
print('Commit timestamp verification:',chronological_statement)
print('Risk-factor status: provisional, requires independent review; no automatic calibration')
display(pd.DataFrame(rows))

### 11.2 Run the historical regression tests (reference-only oracle)

The original SWE-bench test patch contains two regression tests for view-only ManyToMany admin inlines. We run them against **both** pinned source revisions:

- **Pre-fix snapshot:** apply the later regression tests **only inside an isolated reference-test copy**, never to the pre-fix detector evidence. The expectation is failure, but the observed outcome is recorded without forcing it.
- **Fixed snapshot:** run the same two tests already included in the upstream fix commit.

The old Django release predates the current Colab Python interpreter. The test runner therefore uses an isolated Python 3.8 environment via `uv`. Failed environment setup is recorded as **ENVIRONMENT_ERROR**, not as a security test failure. This step executes historical repository code; run it only in a disposable Colab runtime.

In [ ]:
# Automatically execute in Colab, but skip local replay in this file-generation environment.
RUN_HISTORICAL_TESTS = IS_COLAB
PYTHON_VERSION_FOR_REPLAY = '3.8'
HIST_ROOT = BASE / 'tdsc_django_historical_reference_tests'
HIST_ROOT.mkdir(exist_ok=True,parents=True)

TEST_METHODS=[
    'admin_inlines.tests.TestInlinePermissions.test_inline_add_m2m_view_only_perm',
    'admin_inlines.tests.TestInlinePermissions.test_inline_change_m2m_view_only_perm'
]

# All results must show what actually happened; expected labels are separate.
test_records=[]

def get_pinned_django_tree(commit, target):
    if (target/'tests/runtests.py').exists():
        return target,'CACHED'
    url='https://codeload.github.com/django/django/zip/'+commit
    req=urllib.request.Request(url,headers={'User-Agent':'S3DT-TDSC-reproducibility/1.0'})
    with urllib.request.urlopen(req,timeout=120) as f:
        payload=f.read()
    archive_hash=sha256(payload)
    with zipfile.ZipFile(io.BytesIO(payload)) as z:
        # Reject directory traversal before extracting.
        for member in z.infolist():
            dst=(HIST_ROOT/member.filename).resolve()
            if not dst.is_relative_to(HIST_ROOT.resolve()):
                raise ValueError('Untrusted archive path detected')
        z.extractall(HIST_ROOT)
        roots=sorted({n.split('/')[0] for n in z.namelist() if '/' in n})
    candidates=[HIST_ROOT/name for name in roots if (HIST_ROOT/name/'tests/runtests.py').exists()]
    if len(candidates)!=1:
        raise RuntimeError('Expected one immutable Django snapshot, got '+str(candidates))
    src=candidates[0]
    if src != target:
        src.rename(target)
    (AUDIT/(target.name+'_source_archive_sha256.txt')).write_text(archive_hash+'  '+url+'\n')
    return target,'PINNED_CODELOAD_ARCHIVE'

def apply_reference_test_patch(base_tree):
    patch_file = OUT/'reference_regression_tests.patch'
    assert patch_file.exists()
    # git apply works on a downloaded source archive without .git history.
    check=subprocess.run(['git','apply','--check',str(patch_file)],cwd=base_tree,
                         capture_output=True,text=True,timeout=40)
    if check.returncode:
        raise RuntimeError('Regression patch did not apply cleanly: '+check.stderr[:500])
    apply=subprocess.run(['git','apply',str(patch_file)],cwd=base_tree,
                         capture_output=True,text=True,timeout=40)
    if apply.returncode:
        raise RuntimeError('Failed to apply regression patch: '+apply.stderr[:500])


def classify_test_run(proc):
    text=(proc.stdout or '')+'\n'+(proc.stderr or '')
    if proc.returncode==0 and ('OK' in text or 'Ran 1 test' in text):return 'PASS'
    if 'FAIL:' in text and ('AssertionError' in text or 'FAILED' in text):return 'ASSERTION_FAILURE'
    if 'ERROR:' in text or 'ModuleNotFoundError' in text or 'ImportError' in text or 'SyntaxError' in text:
        return 'ENVIRONMENT_OR_TEST_ERROR'
    return 'INCONCLUSIVE_NONZERO_EXIT'

if RUN_HISTORICAL_TESTS:
    try:
        if shutil.which('uv') is None:
            setup=subprocess.run([sys.executable,'-m','pip','install','-q','uv'],capture_output=True,text=True,timeout=180)
            if setup.returncode:raise RuntimeError('uv package install failed: '+setup.stderr[-700:])
        uv=shutil.which('uv')
        if uv is None:raise RuntimeError('uv executable unavailable after installation')
        setup=subprocess.run([uv,'python','install',PYTHON_VERSION_FOR_REPLAY],
                             capture_output=True,text=True,timeout=360)
        if setup.returncode:raise RuntimeError('historical Python install failed: '+setup.stderr[-700:])
        trees={}
        for label,sha in [('pre_fix',BASE_COMMIT),('post_fix',FIX_COMMIT)]:
            tree=HIST_ROOT/f'django_{label}_{sha[:10]}'
            trees[label],_ = get_pinned_django_tree(sha,tree)
        apply_reference_test_patch(trees['pre_fix'])
        test_sources={label:(tree/'tests/admin_inlines/tests.py').read_text()
                      for label,tree in trees.items()}
        for label,test_src in test_sources.items():
            for method in TEST_METHODS:
                if ('def '+method.rsplit('.',1)[-1]+'(') not in test_src:
                    raise RuntimeError('Missing expected test after applying reference-only test patch: '+label+', '+method)
        for label,tree in trees.items():
            for test in TEST_METHODS:
                t0=time.perf_counter()
                env=os.environ.copy()
                env['PYTHONPATH']=str(tree)+os.pathsep+str(tree/'tests')
                cmd=[uv,'run','--python',PYTHON_VERSION_FOR_REPLAY,'--no-project',
                     '--with','sqlparse==0.4.4','--with','pytz==2023.3',
                     '--with','asgiref==3.4.1',
                     'python','tests/runtests.py',test,'--verbosity','1']
                try:
                    proc=subprocess.run(cmd,cwd=tree,env=env,capture_output=True,text=True,timeout=240)
                    status=classify_test_run(proc)
                    log=((proc.stdout or '')+'\n'+(proc.stderr or ''))
                    (AUDIT/f'test_{label}_{test.split(".")[-1]}.log').write_text(log,errors='replace')
                    record={'snapshot':label,'commit':BASE_COMMIT if label=='pre_fix' else FIX_COMMIT,
                            'test':test,'observed_status':status,'exit_code':proc.returncode,
                            'elapsed_seconds':round(time.perf_counter()-t0,3),
                            'expected_for_reference_only':'ASSERTION_FAILURE' if label=='pre_fix' else 'PASS',
                            'log_file':f'test_{label}_{test.split(".")[-1]}.log'}
                except Exception as exc:
                    record={'snapshot':label,'commit':BASE_COMMIT if label=='pre_fix' else FIX_COMMIT,
                            'test':test,'observed_status':'ENVIRONMENT_ERROR',
                            'elapsed_seconds':round(time.perf_counter()-t0,3),
                            'error':type(exc).__name__+': '+str(exc)[:500]}
                test_records.append(record)
                print(label,test.split('.')[-1],record['observed_status'])
    except Exception as exc:
        error=type(exc).__name__+': '+str(exc)
        print('Historical execution was blocked:',error[:700])
        for label,sha in [('pre_fix',BASE_COMMIT),('post_fix',FIX_COMMIT)]:
            for test in TEST_METHODS:
                if not any(x['snapshot']==label and x['test']==test for x in test_records):
                    test_records.append({'snapshot':label,'commit':sha,'test':test,
                                         'observed_status':'ENVIRONMENT_ERROR',
                                         'error':error[:700]})
else:
    print('Skipping historical execution outside Colab. Static evidence audit still runs.')
    for label,sha in [('pre_fix',BASE_COMMIT),('post_fix',FIX_COMMIT)]:
        for test in TEST_METHODS:
            test_records.append({'snapshot':label,'commit':sha,'test':test,
                                 'observed_status':'NOT_RUN_LOCAL_SMOKE_TEST'})

pd.DataFrame(test_records).to_csv(AUDIT/'historical_regression_test_results.csv',index=False)
display(pd.DataFrame(test_records))

### 11.3 Scope of the current case and reproducibility gates

A single known access-control defect is valuable for explaining the pipeline but does not measure generalisation. This audit **does not claim** independent risk calibration, pre-disclosure detection, false-positive rate, or projection accuracy. These remain separate evaluation requirements. Test outcomes count as behavioural evidence only if historical tests actually executed. The current independent reference labels are never derived from the hybrid score.

In [ ]:
statuses = {(r['snapshot'],r['test']):r['observed_status'] for r in test_records}
ref_tests_verified=(len(statuses)==4 and
    all(statuses[('pre_fix',t)]=='ASSERTION_FAILURE' for t in TEST_METHODS) and
    all(statuses[('post_fix',t)]=='PASS' for t in TEST_METHODS))

GATES={
 'historical_source_provenance':'PASS' if len(base_facts)==3 and len(fix_facts)==3 else 'FAIL',
 'detector_does_not_see_later_fix':'PASS' if 'Fix implementation (reference)' not in THREAD_DETECTOR else 'FAIL',
 'commit_dates_checked':provenance_audit['base_before_fix_chronology'],
 'regression_tests_reproduced':'PASS' if ref_tests_verified else 'PENDING_OR_INCONCLUSIVE',
 'sic_risk_parameter_independent_review':'PENDING',
 'pre_disclosure_detection':'NOT_CLAIMED_POST_REPORT_EVALUATION',
 'ordinary_nonviolating_history_controls':('PASS_7_POST_FIX_SIC_SCOPED_CONTROLS' if len(CONTROL_EVIDENCE)==7 and all(r['guard_unchanged'] and r['sic_scoped_delta_count']==0 for r in CONTROL_EVIDENCE) else 'INCOMPLETE'),
 'claim_projection_accuracy':'NOT_YET_EVALUATED',
 'multi_history_detection_performance':'NOT_YET_EVALUATED',
 'runtime_scaling':'NOT_YET_EVALUATED'
}
(AUDIT/'case3_validation_gates.json').write_text(json.dumps(GATES,indent=2))

validation_table=pd.DataFrame([{'gate':a,'status':b} for a,b in GATES.items()])
validation_table.to_csv(AUDIT/'case3_validation_gates.csv',index=False)
text_lines=['# Django Case Study 3 — validation status','',
            'The initial detector result is an **exploratory, post-report SIC assessment**.',
            'It is not evidence that S3DT detected this vulnerability prior to disclosure.','',
            '## Evidence gates','']
for k,v in GATES.items():text_lines.append(f'- **{k.replace("_"," ")}**: {v}')
text_lines += ['','## What remains for the reviewer responses','',
 '1. Have SIC risk values independently reviewed; retain an immutable configuration record.',
 '2. Seven post-fix ordinary Git commits now provide a scoped negative-control check; these are not a general false-positive estimate.',
 '3. Evaluate projection accuracy with independently annotated raw artifacts.',
 '4. Run additional independent security histories and report per-class metrics, uncertainty, and runtime scaling.',
 '5. Keep reference fixes and regression tests unavailable to the detector at pre-fix checkpoints.']
(AUDIT/'case3_validation_status.md').write_text('\n'.join(text_lines)+'\n')
print('Historical test evidence sufficient?',ref_tests_verified)
display(validation_table)

## 12. Export combined Case Study 3 archive

Exports the original S³DT graphs and transition results **plus** the provenance audit, historical regression-test results, SIC review worksheet and evidence gates. The notebook remains a single Colab workflow. No missing test or false-positive result is silently filled in.

In [ ]:
# Update the original manifest without altering the pre-existing scores.
manifest_path=OUT/'reproducibility_manifest.json'
manifest=json.loads(manifest_path.read_text())
manifest['validation_addendum']={
    'schema':'django_case3_validation_v1',
    'historical_test_runner': 'uv isolated Python 3.8; original Django tests/runtests.py',
    'reference_test_source':'SWE-bench Verified test_patch, not detector input',
    'gates':GATES,
    'observed_test_results':test_records,
    'notes':'Analyst-assigned SIC C/O/H values remain exploratory. Reproduction of regression tests does not validate drift calibration.'
}
manifest['ordinary_development_control_addendum']={
    'case_scope':'Django #30289 view-only implicit M2M inline SIC',
    'placement':'7 consecutive FIRST-PARENT descendants of the fix; not inserted before the fix',
    'controls':len(CONTROL_EVIDENCE),
    'full_semantic_claim_deltas':sum(r['full_delta_count'] for r in CONTROL_EVIDENCE),
    'sic_scoped_claim_deltas':sum(r['sic_scoped_delta_count'] for r in CONTROL_EVIDENCE),
    'security_claims_unchanged':all(r['guard_unchanged'] for r in CONTROL_EVIDENCE),
    'classification_scope':'PASS with frozen exploratory case-specific SIC; not general vulnerability absence',
    'model_inference':'MiniLM when available; controls scored diagnostically even though SATISFIED transitions are not emitted by the main detector',
    'ground_truth_limit':'structural guard equality and commit ancestry; not a separate executed security test for every control',
    'provenance':CONTROL_METADATA,
}
manifest_path.write_text(json.dumps(manifest,indent=2,default=str))

bundle=shutil.make_archive(str(BASE/'TDSC_Django_Case3_With_Validation'),'zip',root_dir=OUT)
print('Combined archive:',bundle)
print('Contents:',len(list(OUT.rglob('*'))),'entries')
if IS_COLAB:
    from google.colab import files
    files.download(bundle)

# 13. Broader quantitative evaluation — system-level SICs (TDSC only)

The following sections replace incident-specific constraint configuration with **one Django-wide security policy library**, evaluated at all relevant real historical checkpoints. We distinguish the older exploratory case-study outputs (retained above) from the new system-level results. The numerical **S³DT formulation remains unchanged**.

The first run is intended to show reproducible policy coverage and per-SIC predictions, **not publish detection accuracy**. Independent labels, policy review and incident-clustered uncertainty are required before statistical claims can be made.

In [ ]:
from pathlib import Path
import csv, io, json, zipfile, hashlib, math, shutil, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

QUANT_ROOT = (Path('/content') if 'google.colab' in sys.modules else Path('/mnt/data')) / 'tdsc_quantitative'
QUANT_ROOT.mkdir(parents=True, exist_ok=True)
EXTRA_ARCHIVES = (Path('/content') if 'google.colab' in sys.modules else Path('/mnt/data')) / 'tdsc_additional_case_archives'
EXTRA_ARCHIVES.mkdir(parents=True, exist_ok=True)
UPLOAD_EXTRA_CASE_ARCHIVES = False  # Change to True to upload further *completed* case-study ZIP files
if UPLOAD_EXTRA_CASE_ARCHIVES and 'google.colab' in sys.modules:
    from google.colab import files
    for name,blob in files.upload().items():
        (EXTRA_ARCHIVES / Path(name).name).write_bytes(blob)

print('Quantitative output directory:',QUANT_ROOT)
print('Additional case archives directory:',EXTRA_ARCHIVES)


## 13.1 Candidate case registry (not reference labels)

The Django registry identifies documented fixing commits used to choose historical *source states*. **All four Django histories are evaluated with the same predeclared system-level SIC library** below. The registry and security advisories never become detector input at earlier checkpoints. The Requests case remains a candidate for a **separate Requests-wide policy library** and is not evaluated with Django SICs.

In [ ]:
CANDIDATE_CASES = [
    dict(case_id='django__django-11149',repository='django/django',security_issue='View-only inline ManyToMany permission bypass',
         advisory_url='https://code.djangoproject.com/ticket/30289',
         fix_commit='8335d59200e4c64dfe3348ea93989d95e0107439',
         state='EXISTING_EXPLORATORY_DJANGO_PILOT', independent_history='Django issue 30289'),
    dict(case_id='django__CVE-2019-12308',repository='django/django',security_issue='Admin URL widget unsafe link (XSS)',
         advisory_url='https://www.djangoproject.com/weblog/2019/jun/03/security-releases/',
         fix_commit='deeba6d92006999fee9adfbd8be79bf0a59e8008',
         state='DOCUMENTED_CANDIDATE_NOT_EVALUATED', independent_history='CVE-2019-12308'),
    dict(case_id='django__CVE-2021-31542',repository='django/django',security_issue='Uploaded filename directory traversal',
         advisory_url='https://www.djangoproject.com/weblog/2021/may/04/security-releases/',
         fix_commit='0b79eb36915d178aef5c6a7bbce71b1e76d376d3',
         state='DOCUMENTED_CANDIDATE_NOT_EVALUATED', independent_history='CVE-2021-31542'),
    dict(case_id='django__CVE-2021-33203',repository='django/django',security_issue='Admindocs template path traversal',
         advisory_url='https://www.djangoproject.com/weblog/2021/jun/02/security-releases/',
         fix_commit='46572de2e92fdeaf047f80c44d52269e54ad68db',
         state='DOCUMENTED_CANDIDATE_NOT_EVALUATED', independent_history='CVE-2021-33203'),
    dict(case_id='requests__CVE-2023-32681',repository='psf/requests',security_issue='Proxy-Authorization credential leak on HTTPS redirects',
         advisory_url='https://github.com/advisories/GHSA-j8r2-6x86-q33q',
         fix_commit='74ea7cf7a6a27a4eeb2ae24e162bcc942a6706d5',
         state='DOCUMENTED_CANDIDATE_NOT_EVALUATED', independent_history='CVE-2023-32681'),
]
case_registry=pd.DataFrame(CANDIDATE_CASES)
case_registry.to_csv(QUANT_ROOT/'security_history_registry.csv',index=False)
print('Documented incident candidates:',len(case_registry),'— already evaluated as pilot:',1)
display(case_registry[['case_id','repository','security_issue','state']])


## 13.1A. Django-wide Security Integrity Constraint collection — **defined once, applied everywhere**

**This is the active TDSC broader-evaluation mode.** The earlier Case Study 3 cells remain as a clearly labelled exploratory illustration; they do **not** supply labels or incident-specific risk values to this run.

The four SICs below represent **Django system requirements**, not individual CVE labels:

1. Administrative access control (`DJANGO-AUTHZ-001`)
2. Safe URL rendering in the Django admin (`DJANGO-URL-001`)
3. File-upload and storage path handling (`DJANGO-UPLOAD-001`)
4. Template-documentation path handling (`DJANGO-TEMPLATE-001`)

The *same four SICs*, the *same evidence-extraction rules*, the *same numerical C/O/H mapping*, and the **original S³DT hybrid formula** are applied to **every Django historical snapshot**: vulnerable states, reference fixes, and routine post-fix commits. A different repository, such as Requests, requires its own system-level SIC collection and is not silently evaluated using Django rules.

**Research safeguards:** (i) the requirements are stored once and SHA-256 fingerprinted before the multi-history run; (ii) labels and security advisories are not inputs to the source-state evaluator; (iii) missing or unresolved evidence is reported as `UNKNOWN`, never counted as PASS; (iv) post-fix states are marked reference-only; (v) parameters are provisional until independently reviewed; (vi) the code is a narrow, evidence-backed source-check adapter and **not** a new security formulation or a proof of arbitrary-program safety. Since the policy families were selected retrospectively from publicly known defects, this is **not a blind pre-disclosure benchmark**.

In [ ]:
"""System-wide Django SIC evaluation adapter for released S3DT notebooks.

Requires S3DT's Claim, ClaimSet, ProjectAndAnnotateThreadEdge,
ScoreClassifyExplain, and model helper functions to have been loaded already.
No ground-truth labels are read by this module.
"""
import ast, contextlib, hashlib, json, os, re, subprocess, time, urllib.request, urllib.error
from pathlib import Path
import numpy as np
import networkx as nx
import pandas as pd

# One fixed policy library applies to every historical Django state, irrespective
# of which CVE or ordinary change motivated its selection. No case IDs here.
SYSTEM_SIC_BLUEPRINT = [
    dict(id='DJANGO-AUTHZ-001', title='Admin mutation permissions',
         description='Admin modification of implicit ManyToMany inlines must require change permission.',
         anchor_paths=['django/contrib/admin/options.py'],
         subjects=['InlineModelAdmin.has_add_permission','InlineModelAdmin.has_change_permission',
                   'InlineModelAdmin.has_delete_permission','policy:DJANGO-AUTHZ-001']),
    dict(id='DJANGO-URL-001', title='Admin URL output safety',
         description='Admin URL widgets must validate URL schemes before creating clickable links.',
         anchor_paths=['django/contrib/admin/widgets.py','django/contrib/admin/templates/admin/widgets/url.html'],
         subjects=['AdminURLFieldWidget.get_context','admin/widgets/url.html','policy:DJANGO-URL-001']),
    dict(id='DJANGO-UPLOAD-001', title='Uploaded filename and path validation',
         description='Uploaded filenames and storage paths must reject traversal and unsafe path elements.',
         anchor_paths=['django/core/files/storage.py','django/core/files/uploadedfile.py'],
         subjects=['Storage.get_available_name','Storage.generate_filename','UploadedFile._set_name',
                   'policy:DJANGO-UPLOAD-001']),
    dict(id='DJANGO-TEMPLATE-001', title='Admin documentation template paths',
         description='Template documentation lookup must keep untrusted paths inside permitted directories.',
         anchor_paths=['django/contrib/admindocs/views.py'],
         subjects=['TemplateDetailView.get_context_data','policy:DJANGO-TEMPLATE-001']),
]
SYSTEM_SCORING = {
    'lambda_p':0.5, 'epsilon':0.25, 'w_p':1.0,
    'factor_weights':{'C':0.4,'O':0.2,'H':0.4},
    # Common numerical mapping for ALL policies (not tuned to each incident).
    'factor_map':{
        'SATISFIED':{'C':0.0,'O':0.0,'H':0.0},
        'SUSPECT':{'C':0.5,'O':0.4,'H':0.5},
        'VIOLATION':{'C':1.0,'O':0.75,'H':0.75},
    },
    'score_only_cutoffs':{'pass':0.25,'violation':0.65},
}
SYSTEM_CONFIG_STATUS='UNREVIEWED_PREDECLARED_FOR_PILOT_NOT_ELIGIBLE_FOR_ACCURACY'
SYSTEM_CONFIG = {'repository':'django/django','policies':SYSTEM_SIC_BLUEPRINT,
                 'scoring':SYSTEM_SCORING,'status':SYSTEM_CONFIG_STATUS,
                 'policy_scope':'All applicable Django security checkpoints, not incident-specific',
                 'detector_input_policy':'Pinned source code only. Fixes/advisories used as reference only.',
                 'limitations':['Retrospectively selected policy families; not a prospective blind benchmark',
                                'Source rules are narrow structural checks, not general program proofs',
                                'C/O/H values are provisional ordinal settings requiring independent review',
                                'Independent reference labels and behavior-oracle checks must be established separately']}
SYSTEM_CONFIG_SHA256=hashlib.sha256(json.dumps(SYSTEM_CONFIG,sort_keys=True).encode()).hexdigest()
SYSTEM_ANCHORS=sorted({p for spec in SYSTEM_SIC_BLUEPRINT for p in spec['anchor_paths']})
POLICY_BY_ID={p['id']:p for p in SYSTEM_SIC_BLUEPRINT}


def _method(source, klass, meth):
    try: tree=ast.parse(source)
    except (SyntaxError,ValueError):return None
    k=next((n for n in ast.walk(tree) if isinstance(n,ast.ClassDef) and n.name==klass),None)
    return next((n for n in (k.body if k else []) if isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef)) and n.name==meth),None)


def _segment(source,node):
    return ast.get_source_segment(source,node) if node is not None else ''


def _ev(commit,path,node,method='AST evidence'):
    return {'commit':commit,'path':path,'line_start':getattr(node,'lineno',None),
            'line_end':getattr(node,'end_lineno',None),'extraction_method':method}


def _fact(subject,behaviour,evidence):
    return {'subject':subject,'behaviour':behaviour,'evidence':evidence}


def inspect_admin_guard(sources,commit):
    path='django/contrib/admin/options.py';source=sources.get(path)
    if source is None:return 'UNKNOWN',[], 'Required source file unavailable'
    facts=[]
    for meth in ('has_add_permission','has_change_permission','has_delete_permission'):
        fn=_method(source,'InlineModelAdmin',meth)
        if fn is None:return 'UNKNOWN',facts,'Relevant permission method unavailable'
        guard=next((n for n in ast.walk(fn) if isinstance(n,ast.If) and
                    'self.opts.auto_created' in (_segment(source,n.test) or '')),None)
        if guard is None:return 'SUSPECT',facts,'Auto-created through-model branch unresolved'
        body='\n'.join(_segment(source,s) or '' for s in guard.body)
        if 'has_view_permission' in body and "'change'" not in body and '"change"' not in body:
            b='view_permission_controls_mutation'
        elif '_has_any_perms_for_target_model' in body and ("'change'" in body or '"change"' in body):
            b='change_permission_controls_mutation'
        else:b='unresolved'
        facts.append(_fact('InlineModelAdmin.'+meth,b,_ev(commit,path,guard)))
    observed=[f['behaviour'] for f in facts]
    if all(b=='change_permission_controls_mutation' for b in observed):return 'SATISFIED',facts,'Change guard is present in all three mutation checks'
    if any(b=='view_permission_controls_mutation' for b in observed):return 'VIOLATION',facts,'Mutation delegated to view permission'
    return 'SUSPECT',facts,'Permission guard pattern not conclusively resolved'


def inspect_admin_url(sources,commit):
    path='django/contrib/admin/widgets.py';template_path='django/contrib/admin/templates/admin/widgets/url.html'
    source=sources.get(path);tmpl=sources.get(template_path)
    if source is None or tmpl is None:return 'UNKNOWN',[], 'URL widget source or template unavailable'
    fn=_method(source,'AdminURLFieldWidget','get_context')
    if fn is None:return 'UNKNOWN',[], 'AdminURLFieldWidget.get_context unavailable'
    ctx=_segment(source,fn) or ''
    valid=('self.validator(' in ctx and 'ValidationError' in ctx and 'url_valid' in ctx)
    safe_template=('{% if url_valid %}' in tmpl and 'widget.href' in tmpl)
    html_unsafe=('widget.href' in tmpl and '{% if widget.value %}' in tmpl and not safe_template)
    evidence=_ev(commit,path,fn)
    facts=[_fact('AdminURLFieldWidget.get_context','scheme_validation_present' if valid else 'scheme_validation_not_identified',evidence),
           _fact('admin/widgets/url.html','url_valid_gate' if safe_template else 'url_valid_gate_missing',
                 _ev(commit,template_path,None,'template text'))]
    if valid and safe_template:return 'SATISFIED',facts,'Validation and link-rendering guard both present'
    if html_unsafe and not valid:return 'VIOLATION',facts,'Clickable link rendered without recognised URL-scheme validation'
    return 'SUSPECT',facts,'URL validation and template guard cannot be established conclusively'


def inspect_upload_paths(sources,commit):
    storage='django/core/files/storage.py';uploads='django/core/files/uploadedfile.py'
    src=sources.get(storage);up=sources.get(uploads)
    if src is None or up is None:return 'UNKNOWN',[], 'Storage or upload implementation unavailable'
    checks=[('Storage','get_available_name',src,storage),('Storage','generate_filename',src,storage),
            ('UploadedFile','_set_name',up,uploads)]
    facts=[]
    for klass,meth,raw,path in checks:
        fn=_method(raw,klass,meth)
        if fn is None:
            facts.append(_fact(klass+'.'+meth,'unresolved',_ev(commit,path,None)))
            continue
        block=_segment(raw,fn) or ''
        if klass=='Storage' and meth=='get_available_name':
            guarded=('PurePath' in block and 'SuspiciousFileOperation' in block and 'validate_file_name' in block)
        elif klass=='Storage' and meth=='generate_filename':
            guarded=('PurePath' in block and 'SuspiciousFileOperation' in block)
        else:
            guarded=('validate_file_name(' in block)
        facts.append(_fact(klass+'.'+meth,'path_validation_present' if guarded else 'path_validation_not_identified',
                           _ev(commit,path,fn)))
    states=[x['behaviour'] for x in facts]
    if all(s=='path_validation_present' for s in states):return 'SATISFIED',facts,'Required upload-path validation checks found'
    if any(s=='unresolved' for s in states):return 'SUSPECT',facts,'A required function or validation path is unresolved'
    return 'VIOLATION',facts,'At least one required upload-path validation check is absent'


def inspect_template_paths(sources,commit):
    path='django/contrib/admindocs/views.py';src=sources.get(path)
    if src is None:return 'UNKNOWN',[], 'Admindocs source file unavailable'
    fn=_method(src,'TemplateDetailView','get_context_data')
    if fn is None:return 'UNKNOWN',[], 'Admindocs TemplateDetailView unavailable'
    block=_segment(src,fn) or ''
    safe=('safe_join(' in block and 'Path(' in block)
    unsafe=bool(re.search(r'Path\s*\(\s*directory\s*\)\s*/\s*template',block))
    b='safe_join_path' if safe else ('direct_untrusted_path_join' if unsafe else 'unresolved')
    facts=[_fact('TemplateDetailView.get_context_data',b,_ev(commit,path,fn))]
    if safe:return 'SATISFIED',facts,'safe_join used before reading the template'
    if unsafe:return 'VIOLATION',facts,'Untrusted template path joined directly without recognised containment check'
    return 'SUSPECT',facts,'Template path check is not resolved'

INSPECTORS={'DJANGO-AUTHZ-001':inspect_admin_guard,'DJANGO-URL-001':inspect_admin_url,
            'DJANGO-UPLOAD-001':inspect_upload_paths,'DJANGO-TEMPLATE-001':inspect_template_paths}


def static_projection(sources, commit):
    """Project all configured SICs from a single time-consistent repository snapshot."""
    records=[];statuses={};rationales={}
    for spec in SYSTEM_SIC_BLUEPRINT:
        sid=spec['id'];status,facts,reason=INSPECTORS[sid](sources,commit)
        statuses[sid]=status;rationales[sid]=reason
        for f in facts:
            records.append({'sic_id':sid,'subject':f['subject'],'predicate':'code_behavior',
                            'object':f['behaviour'],'evidence':f['evidence']})
        if status!='UNKNOWN':
            records.append({'sic_id':sid,'subject':'policy:'+sid,'predicate':'risk_stage',
                            'object':status,'evidence':{'commit':commit,'source':'policy-scoped AST inspection'}})
            records.append({'sic_id':sid,'subject':'policy:'+sid,'predicate':'activity',
                            'object':('checked_'+sid.lower().replace('-','_')),
                            'evidence':{'commit':commit,'source':'policy-scoped activity'}})
    return records,statuses,rationales


def _claim_projection(facts,commit):
    """Build original Claim dictionaries, one stable key per subject/predicate/policy."""
    out=[]
    for f in facts:
        ev=f['evidence'];ev_str=f"{ev.get('commit',commit)}:{ev.get('path',ev.get('source',''))}:{ev.get('line_start','')}"
        out.append({'s':f['subject'],'p':f['predicate'],'o':f['object'],
                    'q':{'policy_id':f['sic_id']},'gamma':1.0,
                    'evidence':[ev_str], 'provenance':ev})
    return out


def system_policy_artifact():
    claims=[]
    for spec in SYSTEM_SIC_BLUEPRINT:
        claims.append({'s':'policy:'+spec['id'],'p':'policy_constraint',
                       'o':spec['description'],'q':{'policy_id':spec['id']},'gamma':1.0,
                       'evidence':['predeclared_system_sic_collection.json'],
                       'provenance':{'source':'system_requirements','configuration_sha256':SYSTEM_CONFIG_SHA256}})
    return {'claims':claims,'meta':{'source':'system requirements','configuration_sha256':SYSTEM_CONFIG_SHA256}}


def make_sic_instances(SIC):
    return {p['id']:SIC(id=p['id'],description=p['description'],subjects=p['subjects'],
                        predicates=['policy_constraint','risk_stage','activity','code_behavior'],
                        epsilon=SYSTEM_SCORING['epsilon'],lambda_p=SYSTEM_SCORING['lambda_p'],
                        w_p=SYSTEM_SCORING['w_p']) for p in SYSTEM_SIC_BLUEPRINT}


def _status_from_claimset(scoped_claims):
    found=[c.o for c in scoped_claims.claims if c.p=='risk_stage']
    return found[0] if len(found)==1 else 'UNKNOWN'


def system_evaluate_constraint(p,Ci,Cj,delta):
    matches=[c.o for c in Cj.claims if c.p=='risk_stage' and c.s=='policy:'+p.id]
    status=matches[0] if len(matches)==1 else 'UNKNOWN'
    if status=='SATISFIED':return 'SATISFIED'
    if status in ('SUSPECT','VIOLATION'):return status
    raise ValueError('Insufficient evidence for SIC '+p.id)


def system_structured_drift(delta,p,Cj_p):
    status=_status_from_claimset(Cj_p)
    if status not in SYSTEM_SCORING['factor_map']:
        raise ValueError('No structured score with insufficient evidence: '+p.id)
    scores=SYSTEM_SCORING['factor_map'][status]
    return float(sum(SYSTEM_SCORING['factor_weights'][k]*scores[k] for k in ('C','O','H')))


def system_render_summary(Cp):
    order={'policy_constraint':0,'code_behavior':1,'risk_stage':2,'activity':3}
    cl=sorted(Cp.claims,key=lambda c:(order.get(c.p,99),c.s,c.o))
    return ' '.join(f'{c.s.replace("_"," ")}: {c.p.replace("_"," ")} = {c.o.replace("_"," ")}.' for c in cl) or '<empty>'

@contextlib.contextmanager
def system_hooks(namespace):
    names=('evaluate_constraint','risk_weighted_structured_delta','render_summary')
    stored={k:namespace[k] for k in names}
    namespace['evaluate_constraint']=system_evaluate_constraint
    namespace['risk_weighted_structured_delta']=system_structured_drift
    namespace['render_summary']=system_render_summary
    try:yield
    finally:namespace.update(stored)


def _input_sha(data):return hashlib.sha256(data.encode()).hexdigest()


def get_source_for_snapshot(repository,commit,path,cache_dir,local_git=None):
    cache_dir=Path(cache_dir);cache_file=cache_dir/repository.replace('/','_')/commit/path
    if cache_file.exists():return cache_file.read_text(encoding='utf8'), 'cache'
    if local_git and repository=='django/django' and Path(local_git,'.git').exists():
        p=subprocess.run(['git','-C',str(local_git),'show',f'{commit}:{path}'],capture_output=True)
        if p.returncode!=0:return None,'not_in_repository'
        body=p.stdout
    else:
        url=f'https://raw.githubusercontent.com/{repository}/{commit}/{path}'
        request=urllib.request.Request(url,headers={'User-Agent':'S3DT-TDSC-SIC-evaluation'})
        try:
            with urllib.request.urlopen(request,timeout=45) as fp:body=fp.read()
        except urllib.error.HTTPError as e:
            if e.code==404:return None,'not_in_repository'
            raise
    content=body.decode('utf8')
    cache_file.parent.mkdir(parents=True,exist_ok=True)
    cache_file.write_text(content,encoding='utf8')
    return content, 'pinned_source'


def get_parent(repository,commit,local_git=None):
    if local_git and repository=='django/django' and Path(local_git,'.git').exists():
        r=subprocess.run(['git','-C',str(local_git),'rev-parse',commit+'^'],capture_output=True,text=True,check=True)
        return r.stdout.strip()
    url=f'https://api.github.com/repos/{repository}/commits/{commit}'
    req=urllib.request.Request(url,headers={'User-Agent':'S3DT-TDSC-SIC-evaluation','Accept':'application/vnd.github+json'})
    with urllib.request.urlopen(req,timeout=45) as fp:obj=json.load(fp)
    if obj['sha']!=commit:raise ValueError('Commit mismatch')
    return obj['parents'][0]['sha']


def collect_snapshot(repository,commit,cache_dir,local_git=None):
    start=time.perf_counter();files={};prov=[]
    for path in SYSTEM_ANCHORS:
        body,source=get_source_for_snapshot(repository,commit,path,cache_dir,local_git=local_git)
        files[path]=body
        prov.append({'repository':repository,'commit':commit,'path':path,
                     'status':source,'sha256':_input_sha(body) if body is not None else '',
                     'bytes':len(body.encode('utf8')) if body is not None else 0})
    return files,prov,time.perf_counter()-start


def _map_score_only(d):
    if pd.isna(d):return 'NOT_RUN'
    if d<=SYSTEM_SCORING['score_only_cutoffs']['pass']:return 'PASS'
    if d<SYSTEM_SCORING['score_only_cutoffs']['violation']:return 'SUSPECT'
    return 'VIOLATION'


def run_system_history(case_id,repository,checkpoints,cache_dir,output_dir,namespace,local_git=None):
    """Evaluate the complete fixed SIC library at each historical state.

    checkpoints: ordered list of dict(commit, role), e.g. pre_fix_detector_state,
    post_fix_reference_only.  Never uses post-fix source to evaluate pre-fix node.
    """
    Path(output_dir).mkdir(parents=True,exist_ok=True)
    sic_by_id=make_sic_instances(namespace['SIC'])
    snapshots=[];provenance=[];elapsed={};graphs=[]
    for cp in checkpoints:
        sha=cp['commit'];role=cp['role']
        src,prov,t=collect_snapshot(repository,sha,cache_dir,local_git)
        projected,statuses,reasons=static_projection(src,sha)
        art={'claims':_claim_projection(projected,sha),
             'meta':{'commit':sha,'role':role,'repo':repository}}
        snapshots.append({'commit':sha,'role':role,'direct_parent':cp.get('direct_parent'),
                          'artifact':art,'statuses':statuses,'reasons':reasons,'files':src})
        provenance.extend([dict(case_id=case_id,role=role,**p) for p in prov]);elapsed[sha]=t
    rows=[];events=[];claim_rows=[];edge_rows=[]
    # All checkpoints are compared against the SAME requirements. Adjacent
    # historical commits may also be linked if the caller supplies actual history.
    policy=system_policy_artifact()
    for cp in snapshots:
        graph=nx.DiGraph();node='state:'+cp['commit'];pol='policy:django'
        namespace['ProjectAndAnnotateThreadEdge'](graph,policy,cp['artifact'],
            {'kind':'constraint_trace','snapshot':cp['commit'],'role':cp['role'],
             'case_id':case_id},pol,node)
        Ci=namespace['get_claims'](graph,pol);Cj=namespace['get_claims'](graph,node)
        delta=namespace['get_edge_delta'](graph,pol,node)
        for c in Cj.claims:
            claim_rows.append({'case_id':case_id,'checkpoint':cp['role'],'commit':cp['commit'],
                'subject':c.s,'predicate':c.p,'object':c.o,
                'policy_id':c.q.get('policy_id'),'evidence':' | '.join(c.evidence)})
        for sid,p in sic_by_id.items():
            status=cp['statuses'][sid]
            dsc=namespace['restrict_delta_to_scope'](delta,p)
            r={'case_id':case_id,'repository':repository,
               'transition_id':f'{cp["role"]}:{sid}',
               'checkpoint':cp['role'],'commit':cp['commit'],
               'sic_id':sid,'sic_outcome':status,
               'applicability':('UNRESOLVED_EVIDENCE' if status=='UNKNOWN' else 'APPLICABLE'),
               'evidence_reason':cp['reasons'][sid],
               'full_added':len(delta.added),'full_removed':len(delta.removed),
               'full_modified':len(delta.modified),
               'scoped_added':len(dsc.added),'scoped_removed':len(dsc.removed),
               'scoped_modified':len(dsc.modified),
               'C':np.nan,'O':np.nan,'H':np.nan,
               'D_structured':np.nan,'D_embed':np.nan,'D_hybrid':np.nan,
               'score_only_label':'NOT_RUN','detector_output':'NOT_RUN',
               'detector_generated_event':False,
               'source_sha256':hashlib.sha256(json.dumps({k:_input_sha(v) if v is not None else '' for k,v in cp['files'].items()},sort_keys=True).encode()).hexdigest(),
               'config_sha256':SYSTEM_CONFIG_SHA256,
               'evaluation_status':SYSTEM_CONFIG_STATUS}
            if status in ('SATISFIED','SUSPECT','VIOLATION'):
                vals=SYSTEM_SCORING['factor_map'][status];r.update(vals)
                d_structured=system_structured_drift(dsc,p,namespace['select_claims'](Cj,p))
                r['D_structured']=d_structured
                encoder=namespace.get('_EMBEDDER')
                if encoder is not None:
                    with system_hooks(namespace):
                        Ti=system_render_summary(namespace['select_claims'](Ci,p))
                        Tj=system_render_summary(namespace['select_claims'](Cj,p))
                        ei=encoder.encode([Ti],normalize_embeddings=True)[0]
                        ej=encoder.encode([Tj],normalize_embeddings=True)[0]
                        d_embed=namespace['calibrate_embed'](float(1-np.dot(ei,ej)),p)
                        d_hybrid=p.lambda_p*d_structured+(1-p.lambda_p)*d_embed
                        r['D_embed']=d_embed;r['D_hybrid']=d_hybrid
                        r['score_only_label']=_map_score_only(d_hybrid)
                        if status in ('SUSPECT','VIOLATION'):
                            cand=namespace['DriftCandidate'](source=pol,target=node,
                                constraint_id=sid,result=status,delta=delta)
                            event=namespace['ScoreClassifyExplain'](graph,cand,p)
                            assert abs(event.severity-d_hybrid)<1e-5
                            r['detector_output']=event.drift_type
                            r['detector_generated_event']=True
                            events.append({'case_id':case_id,'checkpoint':cp['role'],
                                           'commit':cp['commit'],**event.model_dump()})
                        else:r['detector_output']='not emitted (SIC satisfied)'
                else:r['detector_output']='EMBEDDING_NOT_RUN'
            rows.append(r)
        graphs.append(graph)
    for prev,curr in zip(snapshots,snapshots[1:]):
        # This edge represents a real historical transition ONLY when verified
        # as consecutive parent->child; callers otherwise get separate policy edges.
        if curr.get('direct_parent')==prev['commit']:
            Ci=namespace['project_semantic_claims'](prev['artifact'])
            Cj=namespace['project_semantic_claims'](curr['artifact'])
            ed=namespace['align_and_diff'](Ci,Cj)
            edge_rows.append({'case_id':case_id,'source_commit':prev['commit'],'target_commit':curr['commit'],
                              'added':len(ed.added),'removed':len(ed.removed),'modified':len(ed.modified)})
    return pd.DataFrame(rows),pd.DataFrame(claim_rows),pd.DataFrame(provenance),pd.DataFrame(edge_rows),events,snapshots,elapsed


In [ ]:
# Freeze the single system-level policy library BEFORE the batch is scored.
SYSTEM_SIC_DIR = QUANT_ROOT / 'system_sic_run'
SYSTEM_SIC_DIR.mkdir(parents=True, exist_ok=True)
SYSTEM_CACHE = QUANT_ROOT / 'pinned_source_cache'
SYSTEM_CACHE.mkdir(parents=True, exist_ok=True)
(system_config_path := SYSTEM_SIC_DIR/'predeclared_django_system_sics.json').write_text(
    json.dumps(SYSTEM_CONFIG,indent=2,ensure_ascii=False),encoding='utf8')
(system_hash_path := SYSTEM_SIC_DIR/'predeclared_sic_sha256.txt').write_text(
    SYSTEM_CONFIG_SHA256+'\n',encoding='ascii')

for spec in SYSTEM_SIC_BLUEPRINT:
    print(spec['id'], ':', spec['description'])
print('Same library for all histories; policy fingerprint:',SYSTEM_CONFIG_SHA256)
print('Calibration status:',SYSTEM_CONFIG_STATUS)


## 13.1B. Run all Django SICs over each actual historical snapshot

For every documented Django fix, retrieve its **first-parent pre-fix source state**, then the **later fixing state only as reference evidence**. Each state is analysed independently with all four system SICs. The view-only permission history additionally includes the seven verified consecutive ordinary changes from the existing Case Study 3 notebook.

This is **not** an independent CVE-specific scorer. It runs the same `static_projection`, S³DT claim alignment, SIC evaluation, structured drift, MiniLM embedding drift, hybrid fusion, and explanation functions across every historical checkpoint. The source checker covers only explicitly documented mechanisms; ambiguous/missing structures remain unresolved.

A list of identified source vulnerabilities is **not** an independently curated ground-truth dataset. We export a blank reference-label worksheet and refrain from publishing accuracy until external evidence and policy review are complete.

In [ ]:
# The case registry above contains four Django histories plus one Requests candidate.
# DO NOT create incident-specific SICs inside this loop.
RUN_SYSTEM_EVALUATION = True
INCLUDE_ORDINARY_DJANGO_CONTROLS = True
# Local testing can use a complete Git checkout; Colab uses commit-pinned GitHub files.
LOCAL_DJANGO_GIT = (Path('/mnt/data/django_gitdata')
                    if (not IS_COLAB and Path('/mnt/data/django_gitdata/.git').exists()) else None)

SYSTEM_RESULTS=[]; SYSTEM_CLAIMS=[]; SYSTEM_PROVENANCE=[];SYSTEM_EDGES=[]
SYSTEM_EVENTS=[];SYSTEM_RUNTIME=[];SYSTEM_SNAPSHOTS=[];SYSTEM_RUN_ERRORS=[]

if RUN_SYSTEM_EVALUATION:
    for entry in CANDIDATE_CASES:
        cid=entry['case_id'];repository=entry['repository']
        if repository!='django/django':
            print('SKIPPED (different system, no Django SICs applied):',cid)
            SYSTEM_RUN_ERRORS.append({'case_id':cid,'status':'SEPARATE_SYSTEM_POLICY_LIBRARY_REQUIRED'})
            continue
        t_case=time.perf_counter()
        print('Evaluating predefined system SICs:',cid)
        try:
            fix=entry['fix_commit']
            parent=get_parent(repository,fix,local_git=LOCAL_DJANGO_GIT)
            checkpoints=[{'role':'pre_fix_detector_state','commit':parent},
                         {'role':'post_fix_reference_only','commit':fix,'direct_parent':parent}]
            if cid=='django__django-11149' and INCLUDE_ORDINARY_DJANGO_CONTROLS:
                # The preceding notebook cell verified each control is a direct descendant.
                for i,meta in enumerate(CONTROL_METADATA,1):
                    checkpoints.append({'role':f'ordinary_post_fix_{i:02d}',
                                        'commit':meta['commit'],'direct_parent':meta['parent']})
            d,c,pr,e,ev,states,loads=run_system_history(
                cid,repository,checkpoints,SYSTEM_CACHE,SYSTEM_SIC_DIR,globals(),
                local_git=LOCAL_DJANGO_GIT)
            SYSTEM_RESULTS.append(d);SYSTEM_CLAIMS.append(c);SYSTEM_PROVENANCE.append(pr)
            SYSTEM_EDGES.append(e);SYSTEM_EVENTS.extend(ev)
            for snap in states:
                SYSTEM_SNAPSHOTS.append({'case_id':cid,'checkpoint':snap['role'],
                    'commit':snap['commit'],'statuses':snap['statuses'],
                    'rationales':snap['reasons']})
            SYSTEM_RUNTIME.append({'case_id':cid,'n_snapshots':len(states),
                                   'n_policy_checks':len(d),
                                   'elapsed_seconds':time.perf_counter()-t_case,
                                   'source_fetch_seconds':sum(loads.values())})
        except Exception as err:
            SYSTEM_RUN_ERRORS.append({'case_id':cid,'status':'ERROR','reason':str(err)[:500]})
            print('Failed:',cid,type(err).__name__,str(err)[:200])

system_transition_rows_df=(pd.concat(SYSTEM_RESULTS,ignore_index=True)
                           if SYSTEM_RESULTS else pd.DataFrame())
system_claims_df=(pd.concat(SYSTEM_CLAIMS,ignore_index=True)
                  if SYSTEM_CLAIMS else pd.DataFrame())
system_provenance_df=(pd.concat(SYSTEM_PROVENANCE,ignore_index=True)
                      if SYSTEM_PROVENANCE else pd.DataFrame())
system_edges_df=(pd.concat(SYSTEM_EDGES,ignore_index=True)
                 if SYSTEM_EDGES else pd.DataFrame())

assert not system_transition_rows_df.empty,'No historical Django snapshots evaluated'
assert len(system_transition_rows_df)==len(system_transition_rows_df[['case_id','transition_id']].drop_duplicates())
assert system_transition_rows_df.groupby(['case_id','checkpoint']).sic_id.nunique().eq(len(SYSTEM_SIC_BLUEPRINT)).all(),        'Every Django checkpoint must have a row for each system SIC'
assert all(entry['repository']=='django/django' for entry in CANDIDATE_CASES
           if entry['case_id'] in set(system_transition_rows_df.case_id))
assert not (system_transition_rows_df.loc[
     system_transition_rows_df.applicability=='UNRESOLVED_EVIDENCE','sic_outcome']=='SATISFIED').any()

# Never publish a source-state assessment of the later fixing commit as a
# prospective pre-fix detection.
system_transition_rows_df['is_reference_only']=(
    system_transition_rows_df.checkpoint=='post_fix_reference_only')

system_transition_rows_df.to_csv(SYSTEM_SIC_DIR/'system_sic_transition_results.csv',index=False)
system_claims_df.to_csv(SYSTEM_SIC_DIR/'system_sic_claim_inventory.csv',index=False)
system_provenance_df.to_csv(SYSTEM_SIC_DIR/'system_sic_source_provenance.csv',index=False)
system_edges_df.to_csv(SYSTEM_SIC_DIR/'verified_consecutive_commit_edges.csv',index=False)
pd.DataFrame(SYSTEM_RUNTIME).to_csv(SYSTEM_SIC_DIR/'system_sic_runtime.csv',index=False)
pd.DataFrame(SYSTEM_RUN_ERRORS).to_csv(SYSTEM_SIC_DIR/'system_sic_case_status.csv',index=False)
(SYSTEM_SIC_DIR/'system_sic_evidence_records.json').write_text(
    json.dumps(SYSTEM_SNAPSHOTS,indent=2,default=str),encoding='utf8')
(SYSTEM_SIC_DIR/'system_sic_drift_events.json').write_text(
    json.dumps(SYSTEM_EVENTS,indent=2,default=str),encoding='utf8')

print('Evaluated system-SIC rows:',len(system_transition_rows_df))
print('Django histories:',system_transition_rows_df.case_id.nunique())
print('Real embedding scores available:',int(system_transition_rows_df.D_embed.notna().sum()))
display(pd.crosstab(system_transition_rows_df.sic_id,
                    system_transition_rows_df.sic_outcome))
display(system_transition_rows_df[['case_id','checkpoint','sic_id','sic_outcome',
                                   'D_structured','D_embed','D_hybrid','detector_output']].head(24))


## 13.1C. Integrated historical knowledge graph and source-linked explanations

The graph includes policy nodes, real repository checkpoints, per-SIC assessments, and claim-evidence nodes. **Each security incident is a separate historical branch**, because the commits span different periods; no fictitious sequence is drawn between those branches. Fix states are labelled reference-only and ordinary commits are shown as real consecutive descendants.

In [ ]:
from matplotlib.lines import Line2D

KG=nx.DiGraph()
root_id='django:system_sic_collection'
KG.add_node(root_id,kind='system',label='Django system-wide SICs')
for spec in SYSTEM_SIC_BLUEPRINT:
    KG.add_node(spec['id'],kind='sic',label=spec['title'])
    KG.add_edge(root_id,spec['id'],kind='defines')
for row in system_transition_rows_df.itertuples(index=False):
    state=f'checkpoint:{row.case_id}:{row.commit}'
    KG.add_node(state,kind='checkpoint',label=row.commit[:10],
                role=row.checkpoint,reference_only=bool(row.is_reference_only))
    observation=f'outcome:{row.case_id}:{row.commit}:{row.sic_id}'
    KG.add_node(observation,kind='sic_result',label=f'{row.sic_id}: {row.sic_outcome}',
                status=row.sic_outcome,checkpoint=row.checkpoint)
    KG.add_edge(state,observation,kind='evaluated_under')
    KG.add_edge(observation,row.sic_id,kind='constraint_result')
for row in system_edges_df.itertuples(index=False):
    KG.add_edge(f'checkpoint:{row.case_id}:{row.source_commit}',
                f'checkpoint:{row.case_id}:{row.target_commit}',
                kind='verified_git_parent',modified=int(row.modified))
for row in system_claims_df.itertuples(index=False):
    if row.predicate!='code_behavior':continue
    nid=f'claim:{row.case_id}:{row.commit}:{row.policy_id}:{row.subject}'
    KG.add_node(nid,kind='source_claim',label=f'{row.subject}: {row.object}')
    KG.add_edge(f'checkpoint:{row.case_id}:{row.commit}',nid,kind='source_backed_claim')

# The graph is generated from the SAME saved evaluation rows; it does not add
# a new set of inferred relationships for visual appearance.
node_types={n:k['kind'] for n,k in KG.nodes(data=True)}
assert all(node_types[u]=='checkpoint' and node_types[v]=='checkpoint'
           for u,v,a in KG.edges(data=True) if a['kind']=='verified_git_parent')
nx.write_graphml(KG,SYSTEM_SIC_DIR/'django_system_sic_thread.graphml')

# Compact journal-preview visual: histories are independent lanes.
fig,ax=plt.subplots(figsize=(14,6.7))
fig.patch.set_facecolor('white');ax.set_facecolor('white')
colors={'SATISFIED':'#27744c','SUSPECT':'#b07819','VIOLATION':'#a62f3a','UNKNOWN':'#777777'}
case_order=list(dict.fromkeys(system_transition_rows_df.case_id))
policy_order=[x['id'] for x in SYSTEM_SIC_BLUEPRINT]
for j,cid in enumerate(case_order):
    case=system_transition_rows_df[system_transition_rows_df.case_id==cid]
    roles=list(dict.fromkeys(case.checkpoint))
    y=len(case_order)-j
    ax.text(-0.2,y,cid.replace('django__',''),ha='right',va='center',fontsize=9)
    for x,role in enumerate(roles):
        rr=case[case.checkpoint==role]
        codes=' '.join({'SATISFIED':'✓','SUSPECT':'?','VIOLATION':'!','UNKNOWN':'–'}.get(
            rr.loc[rr.sic_id==p,'sic_outcome'].iat[0],'–') for p in policy_order)
        icon='(ref)' if role=='post_fix_reference_only' else '(ordinary)' if role.startswith('ordinary') else '(pre)'
        ax.text(x,y,role.replace('ordinary_post_fix_','control_').replace('pre_fix_detector_state','pre-fix').replace('post_fix_reference_only','post-fix')+' '+icon+'\n'+codes,
                ha='center',va='center',fontsize=7.1,
                bbox={'boxstyle':'round,pad=0.5','facecolor':'#eef3f7','edgecolor':'#8291a5'})
        if x>0:ax.annotate('',xy=(x-0.28,y),xytext=(x-0.72,y),arrowprops={'arrowstyle':'->','color':'#637081','lw':1})
ax.set_xlim(-0.4,max(3,max(system_transition_rows_df.groupby('case_id').checkpoint.nunique())-0.6))
ax.set_ylim(0,len(case_order)+1);ax.set_axis_off()
ax.set_title('Django: same system-level SIC collection across real history',fontsize=14,pad=18)
ax.text(0,-0.02,'Each checkpoint: '+', '.join([x.replace('DJANGO-','') for x in policy_order])+'; ✓ satisfied, ! violation, ? suspect, – unresolved',
        transform=ax.transAxes,ha='left',fontsize=8)
fig.tight_layout()
fig.savefig(SYSTEM_SIC_DIR/'django_system_sic_overview.png',dpi=180,bbox_inches='tight')
plt.show()
print('Exported GraphML with',KG.number_of_nodes(),'nodes and',KG.number_of_edges(),'edges.')


## 13.1C-2. Full system-level Django digital thread: **all four incident histories**

The earlier pilot's two figures intentionally cover **ticket #30289 only**. The system-level evaluation above runs **all four Django histories**. This cell now renders its complete results in three complementary ways:

1. **Full historical multi-history overview** — four independent incident windows, their verified pre-fix → correction edges, seven real ordinary subsequent commits, and *all four* SIC outcomes at every checkpoint.
2. **Detector-source pre-fix comparison** — only pre-fix code at each historical window; later fixing commits are never treated as earlier detector input.
3. **Source-backed claim deltas and interactive full evidence graph** — for the documented corrections, show which code-behaviour claims actually changed; explore *all 215 provenance-connected graph nodes and 270 verified or model-recorded relations* in interactive HTML.

No additional incidents, synthetic artifacts, inferred cross-incident chronology, classifications or ground-truth labels are created for these figures. The shared policy library is a retrospective research configuration, not evidence of prospective discovery. SVG figures and complete GraphML are exportable for the paper. The outputs are automatically bundled in the system-level results ZIP below.

In [ ]:
"""S3DT Django multi-history graph export (visualization only; never sets labels or scores).

Inputs: existing system_sic_run CSV and GraphML exports.
Outputs: full history figure, detector-only pre-fix figure, HTML interactive claim graph,
         graph manifest. No policy/extraction/scoring changes.
"""
from __future__ import annotations
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import json, shutil
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
import plotly.graph_objects as go

CASES = [
 ('django__django-11149','Ticket #30289','Admin inline permissions'),
 ('django__CVE-2019-12308','CVE-2019-12308','Unsafe admin URL rendering'),
 ('django__CVE-2021-31542','CVE-2021-31542','Uploaded-file path traversal'),
 ('django__CVE-2021-33203','CVE-2021-33203','Admin documentation paths')
]
SICS = [
 ('DJANGO-AUTHZ-001','AUTHZ','Admin mutation permissions'),
 ('DJANGO-URL-001','URL','Admin URL safety'),
 ('DJANGO-UPLOAD-001','UPLOAD','Upload path validation'),
 ('DJANGO-TEMPLATE-001','TEMPLATE','Documentation paths')
]
STATUS_COLOR = {'SATISFIED':'#2D8067','VIOLATION':'#B54852','SUSPECT':'#E1A341','UNKNOWN':'#9BA6B2'}
STATUS_MARK = {'SATISFIED':'✓','VIOLATION':'!','SUSPECT':'?','UNKNOWN':'·'}
BACKGROUND='#F8FAFD'; NAVY='#16344A'; SLATE='#52677B'


def ordered_rows(df, caseid, detector_only=False):
    d=df[df.case_id==caseid].copy()
    roles=list(dict.fromkeys(d.checkpoint))
    # Always render true within-window historical order rather than source CSV grouping
    roles=sorted(roles,key=lambda s:0 if s=='pre_fix_detector_state' else (1 if s=='post_fix_reference_only' else 2+int(s.rsplit('_',1)[-1]) if s.startswith('ordinary_post_fix_') else 100))
    if detector_only:
        roles=[r for r in roles if r=='pre_fix_detector_state']
    return [d[d.checkpoint==r] for r in roles]


def draw_overview(results, claims, edges, outdir, detector_only=False):
    path=outdir / ('django_all_histories_detector_snapshots' if detector_only else 'django_all_histories_full_thread')
    fig,ax=plt.subplots(figsize=(21.0,12.6),dpi=150)
    fig.patch.set_facecolor('white');ax.set_facecolor('white')
    ax.set_xlim(0,21);ax.set_ylim(0,12.6);ax.axis('off')
    ax.add_patch(Rectangle((0,0),21,12.6,color=BACKGROUND,zorder=-100))
    title='S³DT | Django — four incident histories, one SIC library'
    subtitle=('Pre-fix detector-source views only. Each branch is an independently reconstructed historical state.'
              if detector_only else 'Historical pre-fix sources, fixing commits (reference only), and genuine post-fix ordinary changes')
    ax.text(.58,12.07,title,fontsize=19,fontweight='bold',color=NAVY,ha='left',va='center')
    ax.text(.6,11.64,subtitle,fontsize=9.8,color=SLATE,ha='left',va='center')
    # reusable policy library -- not four isolated SICs per incident
    header_x=.65; header_y=10.72
    ax.text(header_x,header_y+.39,'SHARED DJANGO SECURITY REQUIREMENTS',color=NAVY,fontsize=9,fontweight='bold')
    for i,(sid,short,desc) in enumerate(SICS):
        x=header_x+i*5.04
        fc=['#DAECF7','#DEF4ED','#EDE7F8','#FFF0DD'][i]
        edge=['#487A9A','#3C8773','#76619F','#B77B3C'][i]
        b=FancyBboxPatch((x,header_y-.29),4.75,.53,boxstyle='round,pad=0.06,rounding_size=.12',
                         linewidth=1,edgecolor=edge,facecolor=fc)
        ax.add_patch(b)
        ax.text(x+.16,header_y-.00,f'{i+1}. {short}',fontsize=10,fontweight='bold',ha='left',va='center',color=NAVY)
        ax.text(x+1.48,header_y-.00,desc,fontsize=8.8,ha='left',va='center',color=SLATE)
    # Small shared spine conveys the same library applies to all four distinct history lanes.
    spine_x=.55
    ax.plot([spine_x,spine_x],[1.0,9.65],lw=1.5,color='#88A7BC',alpha=.9,zorder=0)
    for j,(cid,label,topic) in enumerate(CASES):
        top=9.71-j*2.33; bottom=top-2.10
        b=FancyBboxPatch((.86,bottom),19.58,2.11,boxstyle='round,pad=0.03,rounding_size=.17',
                         edgecolor='#CAD6E0',linewidth=.9,facecolor='white',zorder=0)
        ax.add_patch(b)
        ax.plot([spine_x,.86],[bottom+1.1,bottom+1.1],color='#88A7BC',lw=1.6,zorder=2)
        ax.text(1.04,top-.30,label,fontsize=11,fontweight='bold',color=NAVY,ha='left')
        ax.text(1.04,top-.64,topic,fontsize=8.9,color=SLATE,ha='left')
        lane=ordered_rows(results,cid,detector_only=detector_only)
        # Four policy results are always rendered per checkpoint, no fabricated PASS.
        nx_snap=len(lane)
        card_w = 1.65 if nx_snap>3 else 2.30
        card_step=1.86 if nx_snap>3 else 2.62
        start_x=3.63
        y=bottom+.35
        for i,rows in enumerate(lane):
            first=rows.iloc[0];role=first.checkpoint;sha=str(first.commit)[:8]
            x=start_x+i*card_step
            if role=='pre_fix_detector_state':
                state_lbl='PRE-FIX';edge_col='#A8656C';fill='#FFF5F3'
            elif role=='post_fix_reference_only':
                state_lbl='FIX · REF ONLY';edge_col='#7186A1';fill='#F1F5FA'
            else:
                state_lbl='ORDINARY '+role.rsplit('_',1)[-1];edge_col='#84A8A2';fill='#F4FAF7'
            ax.add_patch(FancyBboxPatch((x,y),card_w,1.07,
                boxstyle='round,pad=0.065,rounding_size=.11',edgecolor=edge_col,
                facecolor=fill,linewidth=1.0,zorder=3))
            ax.text(x+.13,y+.86,state_lbl,fontsize=6.8,ha='left',color=SLATE,fontweight='bold',zorder=4)
            ax.text(x+.13,y+.60,sha,fontsize=9,ha='left',color=NAVY,fontweight='bold',zorder=4)
            outcome_map=dict(zip(rows.sic_id,rows.sic_outcome))
            # miniature policy-check cells in stable order (1--4)
            for k,(sid,short,_) in enumerate(SICS):
                v=outcome_map.get(sid,'UNKNOWN');bx=x+.15+k*.355
                ax.add_patch(FancyBboxPatch((bx,y+.13),.28,.30,
                        boxstyle='round,pad=0.012,rounding_size=.045',edgecolor='none',
                        facecolor=STATUS_COLOR.get(v,STATUS_COLOR['UNKNOWN']),zorder=4))
                ax.text(bx+.14,y+.28,STATUS_MARK.get(v,'·'),fontsize=9,color='white',
                        ha='center',va='center',fontweight='bold',zorder=5)
            if i>0:
                px=start_x+(i-1)*card_step+card_w+.09
                qx=x-.08
                ax.annotate('',xy=(qx,y+.56),xytext=(px,y+.56),
                            arrowprops=dict(arrowstyle='->',color='#658397',lw=1.2),zorder=2)
        if detector_only:
            ax.text(start_x+2.85,y+.52,'Independent pre-fix source snapshot; later fixes excluded',fontsize=9,
                    color=SLATE,ha='left',va='center')
        elif nx_snap==2:
            ax.text(start_x+5.6,y+.57,'Separate historical window · no inferred links to other cases',
                    fontsize=8.6,color=SLATE,ha='left',va='center')
        else:
            ax.text(20.05,bottom+.16,'7 successive ordinary commits',fontsize=7.7,color=SLATE,ha='right')
    legend='Each checkpoint:  1 AUTHZ   2 URL   3 UPLOAD   4 TEMPLATE      ✓ rule satisfied   ! rule indicates violation   ? suspected   · unresolved'
    ax.text(.7,.39,legend,fontsize=9.2,color=NAVY,ha='left')
    ax.text(.7,.13,'Statuses are pilot detector outputs, NOT independent ground truth. The policy set was retrospectively selected and remains uncalibrated.',
            fontsize=8.5,color='#9B4852',ha='left')
    fig.savefig(path.with_suffix('.png'),dpi=200,bbox_inches='tight',facecolor=fig.get_facecolor())
    fig.savefig(path.with_suffix('.svg'),bbox_inches='tight',facecolor=fig.get_facecolor())
    plt.close(fig)
    return [str(path.with_suffix(ext)) for ext in ('.png','.svg')]


def _role_priority(name):
    return 0 if name=='pre_fix_detector_state' else (1 if name=='post_fix_reference_only'
            else 2+int(name.rsplit('_',1)[-1]) if name.startswith('ordinary_post_fix_') else 100)


def make_interactive(G,results,output_path):
    """Fully inspectable source-backed graph, with one hover per node and separate edge layers."""
    # Deterministic assignment: four horizontal history lanes; policy nodes shared across branches.
    positions={};case_to_y={c:18-i*12 for i,(c,_,_) in enumerate(CASES)}
    positions['django:system_sic_collection']=(-9,21)
    for k,(sid,_,_) in enumerate(SICS):
        positions[sid]=(-4.7,19-k*7.2)
    for cid,_,_ in CASES:
        y=case_to_y[cid];rr=ordered_rows(results,cid)
        for index,subset in enumerate(rr):
            sha=subset.iloc[0].commit
            # 9 checkpoint long branch requires horizontal navigation; no collapsing real nodes.
            x=index*7
            ck=f'checkpoint:{cid}:{sha}'
            positions[ck]=(x,y)
            for k,(sid,_,_) in enumerate(SICS):
                outcome=f'outcome:{cid}:{sha}:{sid}'
                positions[outcome]=(x+2.5,y+1.6-k*1.04)
            # Claims are at x+4.6 in two rows, by stable subject order
            cls=[n for n,d in G.nodes(data=True) if d.get('kind')=='source_claim'
                 and n.startswith(f'claim:{cid}:{sha}:')]
            cls.sort()
            for j,n in enumerate(cls):
                positions[n]=(x+4.85+(.22 if j%2 else 0),y+2.18-j*.55)
    # some nodes may be omitted only if malformed data; preserve with deterministic positions
    leftovers=[n for n in G.nodes if n not in positions]
    if leftovers:
        raise ValueError('Missing layout nodes: '+','.join(leftovers[:8]))
    # Render traces by edge type: toggle expensive semantic link classes.
    kind_order=['verified_git_parent','defines','evaluated_under','constraint_result','source_backed_claim']
    edge_colors={'verified_git_parent':'#32658A','defines':'#4C7690','evaluated_under':'#9AA8B6',
                 'constraint_result':'#CAC6DF','source_backed_claim':'#C8D5CA'}
    edge_traces=[]
    for kind in kind_order:
        xs=[];ys=[]
        for u,v,d in G.edges(data=True):
            if d.get('kind')!=kind:continue
            x1,y1=positions[u];x2,y2=positions[v]
            xs.extend([x1,x2,None]);ys.extend([y1,y2,None])
        edge_traces.append(go.Scatter(x=xs,y=ys,mode='lines',name=kind.replace('_',' '),
            line={'width':2.3 if kind=='verified_git_parent' else .85,'color':edge_colors[kind]},
            hoverinfo='skip',visible=(kind in ('verified_git_parent','defines','evaluated_under')),
            showlegend=True))
    result_by_key={(r.case_id,r.commit,r.sic_id):r for r in results.itertuples(index=False)}
    node_traces=[]
    colors={'system':'#224963','sic':'#AD823F','checkpoint':'#517891','sic_result':'#5A927C','source_claim':'#8E75AF'}
    for kind in ['system','sic','checkpoint','sic_result','source_claim']:
        nodes=[n for n,d in G.nodes(data=True) if d.get('kind')==kind]
        xs=[];ys=[];text=[];htext=[];col=[];sym=[]
        for n in nodes:
            d=G.nodes[n];x,y=positions[n]
            xs.append(x);ys.append(y)
            t=d.get('label',n)
            if kind=='checkpoint':
                role=d.get('role','');typ='Reference fixing state (future evidence)' if d.get('reference_only') else 'Historical source snapshot'
                extra=f'<br>Role: {role}<br>{typ}'
                col.append('#B6C9D9' if d.get('reference_only') else '#456E8D')
                sym.append('square' if d.get('reference_only') else 'circle')
            elif kind=='sic_result':
                s=d.get('status','UNKNOWN');extra='<br>Result: '+s
                try:
                    c,sha,sic=n[len('outcome:'):].rsplit(':',2)
                    r=result_by_key.get((c,sha,sic))
                    if r is not None:
                        extra+=f'<br>Structured: {r.D_structured:.3f}'
                        if pd.notna(r.D_embed): extra+=f'<br>Embedding: {r.D_embed:.3f}'
                        if pd.notna(r.D_hybrid): extra+=f'<br>Hybrid: {r.D_hybrid:.3f}'
                        extra+=f'<br>Evidence: {str(r.evidence_reason)[:170]}'
                except Exception:pass
                col.append(STATUS_COLOR.get(s,STATUS_COLOR['UNKNOWN']));sym.append('diamond')
            elif kind=='source_claim':
                extra='<br>Source-backed code-behaviour claim'
                col.append(colors[kind]);sym.append('circle-open')
            else:
                extra='';col.append(colors[kind]);sym.append('hexagon' if kind=='sic' else 'star')
            label=(t if len(t)<30 else t[:27]+'…') if kind in ('system','sic','checkpoint') else ''
            text.append(label)
            htext.append(f'<b>{t}</b><br>Kind: {kind}{extra}<br>Node: {n}')
        trace=go.Scatter(x=xs,y=ys,mode='markers+text' if kind in ('system','sic','checkpoint') else 'markers',
            name={'system':'Django security policy library','sic':'4 shared system SICs',
                  'checkpoint':'Historical commit checkpoint','sic_result':'Per-SIC detector outcomes',
                  'source_claim':'Source-backed code claims'}[kind],
            marker={'color':col,'size':13 if kind=='checkpoint' else (10 if kind=='sic_result' else 7),
                    'symbol':sym,'line':{'width':.8,'color':'#FFFFFF'}},
            text=text,textposition='top center',textfont={'size':10,'color':'#233B4B'},
            hovertext=htext,hovertemplate='%{hovertext}<extra></extra>',showlegend=True)
        node_traces.append(trace)
    fig=go.Figure(data=edge_traces+node_traces)
    fig.update_layout(
        title={'text':'S³DT — complete Django claim, evidence and SIC graph (4 incident histories)'},
        template='plotly_white',height=920,hovermode='closest',dragmode='pan',
        xaxis={'showgrid':False,'zeroline':False,'showticklabels':False,'title':'Verified historical sequence within each branch', 'scaleanchor':'y'},
        yaxis={'showgrid':False,'zeroline':False,'showticklabels':False},
        legend={'orientation':'h','y':-0.1},margin={'t':70,'l':35,'r':35,'b':80},
        annotations=[{'x':0,'y':y+4.2,'xref':'x','yref':'y','text':label,'showarrow':False,'font':{'color':NAVY,'size':14},'xanchor':'left'} for cid,label,_ in CASES for y in [case_to_y[cid]]],
        updatemenus=[{'type':'buttons','direction':'right','x':0,'y':1.06,'showactive':True,
                      'buttons':[
                        {'label':'Clean view','method':'update','args':[{'visible':[True,True,True,False,False]+[True]*5}]},
                        {'label':'Show claim links','method':'update','args':[{'visible':[True,True,True,False,True]+[True]*5}]},
                        {'label':'All relationships','method':'update','args':[{'visible':[True]*10}]}]}]
    )
    fig.write_html(str(output_path),include_plotlyjs=True,full_html=True)
    return str(output_path)



def draw_case_claim_deltas(results, claims, outdir):
    """Source-backed changed claim objects per fix (reference figure, not detector-time input)."""
    path=outdir/'django_all_histories_claim_deltas'
    fig,ax=plt.subplots(figsize=(19,12.6),dpi=160)
    fig.patch.set_facecolor('white');ax.set_facecolor('white')
    ax.set_xlim(0,19);ax.set_ylim(0,12.6);ax.axis('off')
    ax.add_patch(Rectangle((0,0),19,12.6,color=BACKGROUND,zorder=-100))
    ax.text(.55,12.05,'S³DT | Source-backed security claim changes across Django fixes',
            fontsize=17,fontweight='bold',color=NAVY)
    ax.text(.55,11.63,'The same four SICs are evaluated in each history. The claims below show only those changed by the documented fixing commit.',
            fontsize=9.4,color=SLATE)
    # one distinct incident window per lane
    for j,(cid,label,topic) in enumerate(CASES):
        top=11.2-j*2.63;bottom=top-2.38
        bx=FancyBboxPatch((.5,bottom),18,2.4,boxstyle='round,pad=.035,rounding_size=.15',
                         facecolor='white',edgecolor='#CBD9E2',linewidth=1)
        ax.add_patch(bx)
        ax.text(.74,top-.36,f'{label}   |   {topic}',fontsize=11,color=NAVY,fontweight='bold')
        lane=ordered_rows(results,cid)
        if len(lane)<2:continue
        pre=lane[0].iloc[0];fix=lane[1].iloc[0]
        # The relevant policy: the only SIC with differing pre/post status.
        before_status=dict(zip(lane[0].sic_id,lane[0].sic_outcome))
        after_status=dict(zip(lane[1].sic_id,lane[1].sic_outcome))
        changed_policies=[sid for sid,_,_ in SICS if before_status.get(sid)!=after_status.get(sid)]
        sid=changed_policies[0] if len(changed_policies)==1 else None
        if sid is None:
            ax.text(1,top-1.2,'No unique changed SIC identified in current source-evidence data',fontsize=10,color=SLATE)
            continue
        pre_claims=claims[(claims.case_id==cid)&(claims.commit==pre.commit)&(claims.policy_id==sid)&(claims.predicate=='code_behavior')]
        fix_claims=claims[(claims.case_id==cid)&(claims.commit==fix.commit)&(claims.policy_id==sid)&(claims.predicate=='code_behavior')]
        pre_map=dict(zip(pre_claims.subject,pre_claims.object));fix_map=dict(zip(fix_claims.subject,fix_claims.object))
        subjects=sorted({*pre_map,*fix_map})
        actual_changes=[s for s in subjects if pre_map.get(s)!=fix_map.get(s)]
        ax.add_patch(FancyBboxPatch((.8,bottom+.47),2.45,1.05,
                    boxstyle='round,pad=.06',facecolor='#FFF2F0',edgecolor='#BB7A7E'))
        ax.text(2.025,bottom+1.19,'PRE-FIX',ha='center',fontsize=9,fontweight='bold',color=NAVY)
        ax.text(2.025,bottom+.79,pre.commit[:10],ha='center',fontsize=9,color=SLATE)
        ax.add_patch(FancyBboxPatch((15.6,bottom+.47),2.45,1.05,
                    boxstyle='round,pad=.06',facecolor='#F0F6FA',edgecolor='#7596B0'))
        ax.text(16.825,bottom+1.19,'FIX · REFERENCE',ha='center',fontsize=9,fontweight='bold',color=NAVY)
        ax.text(16.825,bottom+.79,fix.commit[:10],ha='center',fontsize=9,color=SLATE)
        ax.text(9.4,top-.75,f'SIC: {sid}  ·  {len(actual_changes)} changed source-backed claims',
                ha='center',fontsize=9.3,color=NAVY,fontweight='bold')
        if actual_changes:
            # Render actual code fact transitions, no inferred hidden controls.
            roww=min(3.65,10.3/len(actual_changes));start=9.4-roww*len(actual_changes)/2
            for k,subject in enumerate(actual_changes):
                x=start+k*roww
                ax.add_patch(FancyBboxPatch((x+.04,bottom+.27),roww-.08,1.15,
                        boxstyle='round,pad=.04,rounding_size=.08',
                        facecolor='#F2EDF8',edgecolor='#B8A8D0',linewidth=.8))
                m=subject.split('.')[-1]
                ax.text(x+roww/2,bottom+1.18,m,ha='center',fontsize=8.3,color=NAVY,fontweight='bold')
                def cut(s,n=34):
                    return s if len(s)<=n else s[:n-1]+'…'
                ax.text(x+roww/2,bottom+.90,cut(str(pre_map.get(subject,'[absent]'))),ha='center',fontsize=7,color='#A14250')
                ax.text(x+roww/2,bottom+.66,'↓',ha='center',fontsize=8,color=SLATE)
                ax.text(x+roww/2,bottom+.44,cut(str(fix_map.get(subject,'[absent]'))),ha='center',fontsize=7,color='#2C7B62')
        else:
            ax.text(9.4,bottom+.94,'No change in the configured code-behaviour facts',ha='center',fontsize=9,color=SLATE)
        ax.annotate('',xy=(4.0,bottom+1.01),xytext=(3.33,bottom+1.01),
                    arrowprops=dict(arrowstyle='->',color='#6A8597',lw=1.3))
        ax.annotate('',xy=(15.5,bottom+1.01),xytext=(14.75,bottom+1.01),
                    arrowprops=dict(arrowstyle='->',color='#6A8597',lw=1.3))
        if cid=='django__django-11149':
            ax.text(17.95,bottom+.18,'+ 7 subsequent ordinary commits (unchanged access-control facts)',
                    ha='right',fontsize=7.8,color=SLATE)
    ax.text(.62,.16,'Changes are identified by comparing source-backed code_behavior claim objects; future fixing evidence is reference-only. No performance metric is inferred.',
            fontsize=8.7,color='#964F59')
    fig.savefig(path.with_suffix('.png'),dpi=180,bbox_inches='tight',facecolor='white')
    fig.savefig(path.with_suffix('.svg'),bbox_inches='tight',facecolor='white')
    plt.close(fig)
    return [str(path.with_suffix(ext)) for ext in ('.png','.svg')]


def run_graph_exports(system_dir):
    out=Path(system_dir);out.mkdir(parents=True,exist_ok=True)
    d=pd.read_csv(out/'system_sic_transition_results.csv').fillna({'evidence_reason':''})
    c=pd.read_csv(out/'system_sic_claim_inventory.csv')
    e=pd.read_csv(out/'verified_consecutive_commit_edges.csv')
    G=nx.read_graphml(out/'django_system_sic_thread.graphml')
    assert d.case_id.nunique()==4,'Expected the four verified Django historical windows'
    assert d.groupby(['case_id','checkpoint']).sic_id.nunique().eq(4).all(), 'SIC coverage incomplete'
    assert int(d.is_reference_only.sum())==4*len(SICS),'One post-fix reference checkpoint per case required'
    assert not any(u.startswith('checkpoint:requests') for u in G.nodes)
    # Ensure evidence graph has exactly the real provenance-supported claims rather than decorating.
    code_claims=c.loc[c.predicate=='code_behavior']
    assert len(code_claims)==sum(1 for _,x in G.nodes(data=True) if x.get('kind')=='source_claim')
    exports=[]
    exports+=draw_overview(d,c,e,out,detector_only=False)
    exports+=draw_overview(d,c,e,out,detector_only=True)
    exports+=draw_case_claim_deltas(d,c,out)
    exports.append(make_interactive(G,d,out/'django_all_histories_interactive_claim_graph.html'))
    manifest={
      'purpose':'Visualization of current Django system-level SIC evaluations, not reference truth',
      'incident_histories':int(d.case_id.nunique()),
      'historical_checkpoints':int(d[['case_id','checkpoint']].drop_duplicates().shape[0]),
      'system_level_sics':int(d.sic_id.nunique()),'sic_checkpoint_assessments':int(len(d)),
      'source_backed_code_claims':len(code_claims),
      'explicitly_verified_git_edges':int(len(e)),
      'complete_graph_nodes':int(G.number_of_nodes()),'complete_graph_edges':int(G.number_of_edges()),
      'reference_only_checkpoints':int(d.loc[d.is_reference_only,['case_id','checkpoint']].drop_duplicates().shape[0]),
      'limits':['Incident timelines are separate windows; no undocumented cross-window chronology is invented',
                'Fixing commits and later ordinary controls are reference/history data, not detector knowledge at earlier checkpoints',
                'SIC outcomes are pilot classifications; not validated accuracy or independent ground truth',
                'All source-claim nodes are derived from exported source_backed claim inventory; no fabricated artifacts',
                'SWE-bench non-security tasks and Requests are not implicitly included'],
      'exported_graphs':[Path(x).name for x in exports]
    }
    mf=out/'django_all_histories_graph_manifest.json'
    mf.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf8')
    print('Multi-history visualisations exported:',*[Path(q).name for q in exports],sep='\n- ')
    print('Graph:',G.number_of_nodes(),'nodes,',G.number_of_edges(),'edges')
    return exports+[str(mf)]


# Use files produced by the unchanged Django-wide SIC evaluation above.
MULTI_HISTORY_EXPORTS=run_graph_exports(SYSTEM_SIC_DIR)
from IPython.display import Image, display, FileLink
display(Image(filename=str(SYSTEM_SIC_DIR/'django_all_histories_full_thread.png'),width=1150))
print('Interactive claim graph (215 source/evidence/policy nodes):')
display(FileLink(str(SYSTEM_SIC_DIR/'django_all_histories_interactive_claim_graph.html')))


## 13.1D. Explicit limitations and independent-label worksheet

The system-level scores are **detector outputs**, not independently established security labels. The original S³DT fusion parameters are retained; risk factors follow one provisional Django-wide scale, never tuned per incident. A separate assessor must review the policy scope and numerical settings, then assign reference labels from historical tests/advisories **without reading these predictions**. Only then should any metrics be reported. All fix snapshots remain reference-only.

In [ ]:
# Each (checkpoint, SIC) receives its own reference label, not simply the
# CVE label associated with the incident that caused us to inspect the commit.
SYSTEM_LABEL_TEMPLATE=SYSTEM_SIC_DIR/'independent_system_sic_reference_labels.csv'
if not SYSTEM_LABEL_TEMPLATE.exists():
    label_t=system_transition_rows_df[['case_id','transition_id','sic_id','checkpoint']].copy()
    label_t['independent_case_id']=label_t.case_id
    for col in ['reference_label','annotation_status','independent_evidence_url',
                'oracle_test_record','annotation_method','annotated_by','evidence_cutoff_checked']:
        label_t[col]=''
    label_t.to_csv(SYSTEM_LABEL_TEMPLATE,index=False)

(SYSTEM_SIC_DIR/'EVALUATION_README.txt').write_text(
    'TDSC S3DT system-level SIC evaluation\n\n'
    'The same Django SIC library and factor assignments apply to every history.\n'
    'The policy collection was selected retrospectively and the numbers have NOT been independently calibrated.\n'
    'Do not infer accuracy from model predictions or fill ground truth using prediction columns.\n'
    'Fix snapshots are reference-only, not prospective detector inputs.\n'
    'SIC absence, unresolved source structures and missing evidence are NOT PASS.\n'
    'Independent external validation and incident-clustered uncertainty are required for journal claims.\n'
    'Requests is a separate software system and is not evaluated using Django SICs.\n'
    'Policy library fingerprint: '+SYSTEM_CONFIG_SHA256+'\n',encoding='utf8')
SYSTEM_RESULT_ZIP=shutil.make_archive(str(QUANT_ROOT/'TDSC_Django_System_SIC_Run'),'zip',
                                      root_dir=SYSTEM_SIC_DIR)
print('Results ZIP:',SYSTEM_RESULT_ZIP)
print('Metrics eligibility: NOT READY — independent reference labels and SIC calibration review are required.')
if IS_COLAB:
    from google.colab import files
    files.download(SYSTEM_RESULT_ZIP)


## 13.2 Import system-level evaluations for quantitative review

The aggregation below uses the **new** per-SIC results produced in Section 13.1B, not the previous view-only pilot. It rejects repeated transition IDs and preserves the independent case identifier for incident-level uncertainty analysis. No results are interpreted as ground truth.

In [ ]:
REQUIRED_SCORE_COLUMNS={'transition_id','sic_outcome','D_structured','D_embed','D_hybrid','score_only_label','detector_output','checkpoint'}

def _read_bundle(location):
    location=Path(location)
    if location.is_dir():
        def get(name):
            q=location/name
            return q.read_bytes() if q.exists() else None
    else:
        z=zipfile.ZipFile(location)
        names=z.namelist()
        def get(name):
            matches=[x for x in names if x.endswith('/'+name) or x==name]
            if len(matches)>1: raise ValueError(f'Ambiguous file {name} in {location}')
            return z.read(matches[0]) if matches else None
    raw=get('case3_transition_results.csv') or get('transition_results.csv')
    if raw is None:
        raise ValueError(f'No supported transition results CSV in {location}')
    df=pd.read_csv(io.BytesIO(raw))
    missing=REQUIRED_SCORE_COLUMNS-set(df.columns)
    if missing:raise ValueError(f'Missing columns {sorted(missing)} in {location}')
    sic_raw=get('django_sic_config_EXPLORATORY.json') or get('sic_configuration.json')
    sic=json.loads(sic_raw) if sic_raw else {}
    case_id=sic.get('case_id')
    if not case_id: raise ValueError(f'No case_id in SIC config: {location}')
    gates_raw=get('validation/case3_validation_gates.csv') or get('case_validation_gates.csv')
    gates=pd.read_csv(io.BytesIO(gates_raw)) if gates_raw else pd.DataFrame(columns=['gate','status'])
    runtime_raw=get('runtime_breakdown.csv')
    runtime=pd.read_csv(io.BytesIO(runtime_raw)) if runtime_raw else pd.DataFrame(columns=['stage','elapsed_seconds'])
    return case_id,df,sic,gates,runtime

# Active broader experiment uses the SAME frozen Django SIC library on all cases.
# Legacy per-incident pilot remains available above as an illustration only.
if 'system_transition_rows_df' not in globals() or system_transition_rows_df.empty:
    raise RuntimeError('System-level SIC run missing: execute section 13.1A--13.1D before aggregate metrics.')
ALL_RUNS=[];CASE_CONFIGS={};CASE_GATES={};CASE_TIMES={};CASE_INPUTS={}
for cid,group in system_transition_rows_df.groupby('case_id',sort=False):
    CASE_CONFIGS[cid]={'case_id':cid,'status':SYSTEM_CONFIG_STATUS,
                       'config_sha256':SYSTEM_CONFIG_SHA256,'policies':SYSTEM_SIC_BLUEPRINT}
    CASE_GATES[cid]=pd.DataFrame(columns=['gate','status'])
    CASE_TIMES[cid]=pd.DataFrame([{'stage':'system_sic_batch','elapsed_seconds':
                   next((r['elapsed_seconds'] for r in SYSTEM_RUNTIME if r['case_id']==cid),float('nan'))}])
    CASE_INPUTS[cid]={'path':str(SYSTEM_SIC_DIR),'sha256':SYSTEM_CONFIG_SHA256}
    ALL_RUNS.append(group.copy())
all_transitions=pd.concat(ALL_RUNS,ignore_index=True)
if all_transitions.duplicated(['case_id','transition_id']).any():
    raise ValueError('Repeated transition IDs within case')
all_transitions.to_csv(QUANT_ROOT/'all_observed_transition_scores.csv',index=False)
print(f'Loaded {len(all_transitions)} policy-checkpoint records from {len(CASE_CONFIGS)} Django histories.')
print('These are UNREVIEWED diagnostics, not independent accuracy measurements.')



## 13.3 Report existing outputs without creating labels

S³DT's `sic_outcome` and `score_only_label` are **predictions**, not independent ground truth. Never use them to populate the reference-label column. Score-only comparisons below follow the original case-study cutoffs; they are explicitly **internal ablations**.

In [ ]:
LABELS=('PASS','SUSPECT','VIOLATION')

def score_label(value,pass_threshold=0.25,violation_threshold=0.65):
    if pd.isna(value):return pd.NA
    if float(value)<=pass_threshold:return 'PASS'
    if float(value)<violation_threshold:return 'SUSPECT'
    return 'VIOLATION'

def decision_label(output):
    t=str(output).strip().lower()
    if t.startswith('policy-violating'):return 'VIOLATION'
    if t.startswith('emergent'):return 'SUSPECT'
    if t.startswith('benign') or t.startswith('not emitted'):return 'PASS'
    return pd.NA

all_transitions['pred_detector']=all_transitions['detector_output'].map(decision_label)
for method,col in [('structured','D_structured'),('embedding','D_embed'),('hybrid_score','D_hybrid')]:
    all_transitions['pred_'+method]=all_transitions[col].map(score_label)

score_cols=['case_id','transition_id','checkpoint','sic_outcome','D_structured','D_embed','D_hybrid',
            'pred_detector','pred_structured','pred_embedding','pred_hybrid_score']
diagnostic=all_transitions[score_cols].copy()
diagnostic.to_csv(QUANT_ROOT/'diagnostic_predictions_NOT_ACCURACY.csv',index=False)
print('These are detector outputs and internal ablations — NOT accuracy results.')
display(diagnostic)


## 13.4 Independent labelling worksheet and eligibility gates

Reference labels remain **blank**. They must be assigned for each **(checkpoint, SIC)** by an assessor using independent historical tests and source evidence **without consulting detector scores**. Non-applicable or unresolved checks and reference-only fix snapshots must not become normal PASS examples.

The full Django SIC library and common risk-factor mapping must undergo independent review and be frozen **before a fresh scoring run**. Until that gate is recorded, this notebook only produces diagnostic scores; it does not publish accuracy, F1 or confidence intervals.

In [ ]:
# One canonical, per-SIC label worksheet (also produced in Section 13.1D).
LABEL_FILE=SYSTEM_SIC_DIR/'independent_system_sic_reference_labels.csv'
columns=['case_id','transition_id','sic_id','checkpoint','independent_case_id','reference_label',
         'annotation_status','independent_evidence_url','oracle_test_record',
         'annotation_method','annotated_by','evidence_cutoff_checked']
base_labels=all_transitions[['case_id','transition_id','sic_id','checkpoint']].copy()
base_labels['independent_case_id']=base_labels.case_id
if LABEL_FILE.exists():
    preserved=pd.read_csv(LABEL_FILE,dtype=str).fillna('')
    for col in columns:
        if col not in preserved.columns:preserved[col]=''
    preserved=preserved[columns]
    if preserved.duplicated(['case_id','transition_id']).any():
        raise ValueError('Duplicate independently annotated transition')
    base_labels=base_labels.merge(preserved.drop(columns=['sic_id','checkpoint','independent_case_id']),
                                  on=['case_id','transition_id'],how='left')
else:
    for col in columns[5:]:base_labels[col]=''
for col in columns[5:]:base_labels[col]=base_labels[col].fillna('')
base_labels[columns].to_csv(LABEL_FILE,index=False)
reference=pd.read_csv(LABEL_FILE,dtype=str).fillna('')
if reference.duplicated(['case_id','transition_id']).any():raise ValueError('Duplicate reference labels')
unknown=set(reference.reference_label)-set(LABELS)-{''}
if unknown:raise ValueError('Unknown reference labels: '+repr(unknown))
reviewed=all_transitions.merge(reference.drop(columns=['sic_id','checkpoint']),
                               on=['case_id','transition_id'],how='left',validate='one_to_one')

# A case can only become score-eligible after a separate recorded parameter review
# and a new scoring run with its reviewed, frozen SIC configuration.
def is_frozen(config):
    return config.get('status')=='REVIEWED_FROZEN_BEFORE_SCORING'
reviewed['config_frozen_at_scoring']=reviewed.case_id.map(lambda cid:is_frozen(CASE_CONFIGS[cid]))
reviewed['detector_visible']=(reviewed.checkpoint.eq('pre_fix_detector_state') | reviewed.checkpoint.str.startswith('ordinary_post_fix_'))
reviewed['label_reviewed']=(reviewed.reference_label.isin(LABELS) &
                            reviewed.annotation_status.eq('VERIFIED') &
                            reviewed.independent_evidence_url.fillna('').str.len().gt(10) &
                            reviewed.annotated_by.fillna('').str.len().gt(0) &
                            reviewed.evidence_cutoff_checked.eq('YES'))
reviewed['eligible']=(reviewed.config_frozen_at_scoring & reviewed.detector_visible &
                      reviewed.sic_outcome.isin(['SATISFIED','SUSPECT','VIOLATION']) &
                      reviewed.label_reviewed & reviewed.pred_detector.notna())
reviewed.to_csv(QUANT_ROOT/'evaluation_eligibility_audit.csv',index=False)
print('Eligible transitions:',int(reviewed.eligible.sum()),'of',len(reviewed))
print('Independent incidents with eligible transitions:',reviewed.loc[reviewed.eligible,'case_id'].nunique())
print('Django SIC calibration state:',CASE_CONFIGS.get('django__django-11149',{}).get('status','unknown'))
print('Reference-label worksheet:',LABEL_FILE)


## 13.5 Per-class metrics and incident-clustered uncertainty

Reports metrics **only after eligibility gates pass**. Do not treat multiple transitions from one incident as independent cases. Confidence intervals are clustered by incident; with fewer than five independently evaluated incidents they are suppressed instead of printing misleading precision.

In [ ]:
from collections import Counter

def classification_metrics(frame,pred_column):
    d=frame.dropna(subset=['reference_label',pred_column]).copy()
    d=d[d.reference_label.isin(LABELS)&d[pred_column].isin(LABELS)]
    rows=[]
    for cls in LABELS:
        tp=int(((d.reference_label==cls)&(d[pred_column]==cls)).sum())
        fp=int(((d.reference_label!=cls)&(d[pred_column]==cls)).sum())
        fn=int(((d.reference_label==cls)&(d[pred_column]!=cls)).sum())
        support=int((d.reference_label==cls).sum())
        precision=tp/(tp+fp) if (tp+fp)>0 else float('nan')
        recall=tp/(tp+fn) if (tp+fn)>0 else float('nan')
        f1=2*precision*recall/(precision+recall) if np.isfinite(precision) and np.isfinite(recall) and (precision+recall)>0 else float('nan')
        rows.append(dict(class_label=cls,support=support,precision=precision,recall=recall,f1=f1,tp=tp,fp=fp,fn=fn))
    accuracy=(d.reference_label==d[pred_column]).mean() if len(d) else float('nan')
    return pd.DataFrame(rows),accuracy

def clustered_ci(frame,pred_column,seed=11,n_boot=1000):
    ids=frame.case_id.unique()
    if len(ids)<5:return {'CI_95_low':float('nan'),'CI_95_high':float('nan'),'bootstrap_reason':'Fewer than five independent cases'}
    rng=np.random.default_rng(seed)
    scores=[]
    for _ in range(n_boot):
        sampled=rng.choice(ids,size=len(ids),replace=True)
        parts=[frame.loc[frame.case_id==k] for k in sampled]
        s=pd.concat(parts,ignore_index=True)
        m,acc=classification_metrics(s,pred_column)
        if np.isfinite(acc):scores.append(acc)
    if len(scores)<100:return {'CI_95_low':float('nan'),'CI_95_high':float('nan'),'bootstrap_reason':'Insufficient bootstrap samples'}
    return {'CI_95_low':float(np.percentile(scores,2.5)),'CI_95_high':float(np.percentile(scores,97.5)),
            'bootstrap_reason':'95% incident-clustered percentile bootstrap'}

def incident_clustered_class_intervals(frame,pred_column,seed=43,n_boot=800):
    # Bootstrap entire incidents, NOT individual transitions.
    ids=frame.case_id.unique()
    out={cls:{key:float('nan') for key in ['precision_ci_low','precision_ci_high',
                                         'recall_ci_low','recall_ci_high',
                                         'f1_ci_low','f1_ci_high']} for cls in LABELS}
    if len(ids)<5:return out
    rng=np.random.default_rng(seed)
    recorded={cls:{'precision':[],'recall':[],'f1':[]} for cls in LABELS}
    for _ in range(n_boot):
        selected=rng.choice(ids,size=len(ids),replace=True)
        sampled=pd.concat([frame.loc[frame.case_id==name] for name in selected],ignore_index=True)
        class_metrics,_=classification_metrics(sampled,pred_column)
        for d in class_metrics.to_dict('records'):
            cls=d['class_label']
            for metric in ['precision','recall','f1']:
                if np.isfinite(d[metric]):recorded[cls][metric].append(d[metric])
    for cls in LABELS:
        # Do not claim a meaningful CI if a class appears in fewer than two distinct incidents.
        n_present=frame.loc[frame.reference_label==cls,'case_id'].nunique()
        if n_present<2:continue
        for metric in ['precision','recall','f1']:
            values=recorded[cls][metric]
            if len(values)<100:continue
            out[cls][metric+'_ci_low']=float(np.percentile(values,2.5))
            out[cls][metric+'_ci_high']=float(np.percentile(values,97.5))
    return out

eligible=reviewed.loc[reviewed.eligible].copy()
performance=[];class_reports=[]
for name,pred in [('S3DT_detector','pred_detector'),('structured_only_internal','pred_structured'),
                  ('embedding_only_internal','pred_embedding'),('hybrid_score_internal','pred_hybrid_score')]:
    if eligible.empty:continue
    sub=eligible[eligible[pred].isin(LABELS)]
    if sub.empty:continue
    classes,accuracy=classification_metrics(sub,pred)
    classes['method']=name
    perclass_ci=incident_clustered_class_intervals(sub,pred)
    for k in ['precision_ci_low','precision_ci_high','recall_ci_low','recall_ci_high',
              'f1_ci_low','f1_ci_high']:
        classes[k]=classes.class_label.map(lambda cl:perclass_ci[cl][k])
    classes['n_cases_with_reference_class']=classes.class_label.map(
        lambda cl:sub.loc[sub.reference_label==cl,'case_id'].nunique())
    class_reports.append(classes)
    ci=clustered_ci(sub,pred)
    performance.append(dict(method=name,n_transitions=len(sub),n_independent_cases=sub.case_id.nunique(),
                            accuracy=accuracy,macro_f1=classes.f1.mean(skipna=True),**ci))

perf_df=pd.DataFrame(performance,columns=['method','n_transitions','n_independent_cases','accuracy','macro_f1','CI_95_low','CI_95_high','bootstrap_reason'])
classes_df=pd.concat(class_reports,ignore_index=True) if class_reports else pd.DataFrame(columns=['class_label','support','precision','recall','f1','tp','fp','fn','method','n_cases_with_reference_class','precision_ci_low','precision_ci_high','recall_ci_low','recall_ci_high','f1_ci_low','f1_ci_high'])
perf_df.to_csv(QUANT_ROOT/'eligible_performance.csv',index=False)
classes_df.to_csv(QUANT_ROOT/'eligible_per_class.csv',index=False)
if perf_df.empty:
    print('NOT READY for independent performance claims: missing verified labels and/or frozen SIC cases.')
else:
    display(perf_df)
    display(classes_df)


## 13.6 Runtime and coverage diagnostics

These report measured stages, **not** a scalability claim. A scaling evaluation later requires repeated execution at increasing artifact/transition counts, separating model loading and graph visualisation from detection latency.

In [ ]:
runtime_rows=[]
for cid,times in CASE_TIMES.items():
    for rec in times.to_dict('records'):
        runtime_rows.append({'case_id':cid,**rec})
runtime_df=pd.DataFrame(runtime_rows,columns=['case_id','stage','elapsed_seconds'])
runtime_df.to_csv(QUANT_ROOT/'runtime_observed_by_case.csv',index=False)

coverage={
 'independent_incident_outputs_loaded':len(CASE_CONFIGS),
 'total_reported_transitions':len(all_transitions),
 'detector_visible_transitions':int(reviewed.detector_visible.sum()),
 'post_fix_reference_transitions':int((~reviewed.detector_visible).sum()),
 'transition_labels_independently_verified':int(reviewed.label_reviewed.sum()),
 'transitions_with_frozen_scoring_config':int(reviewed.config_frozen_at_scoring.sum()),
 'eligible_scored_transitions':int(reviewed.eligible.sum()),
 'eligible_independent_incidents':int(eligible.case_id.nunique()),
 'meets_multi_history_requirement':bool(eligible.case_id.nunique()>=2),
 'meets_minimum_cluster_bootstrap_case_count':bool(eligible.case_id.nunique()>=5),
 'reference_classes_present_in_eligible_data':sorted(eligible.reference_label.dropna().unique().tolist()),
 'full_three_class_reference_support':bool(set(LABELS).issubset(set(eligible.reference_label.dropna()))),
 'projection_accuracy_measured':False,
 'scaling_benchmark_measured':False,
 'note':'Counts from exported case-study packages; cannot be used as detection accuracy without the annotation and frozen-configuration gates.'
}
(QUANT_ROOT/'evaluation_readiness.json').write_text(json.dumps(coverage,indent=2),encoding='utf-8')
print(json.dumps(coverage,indent=2))
display(runtime_df)


## 13.7 Export audit and Open Science package

The audit archive includes the current system-level transition outputs, candidate registry, blank reference-label worksheet, readiness report and provenance. The structured claim graphs and full SIC configuration are exported by Section 13.1D. No extra positive cases or independent detection-accuracy claims are inferred from the CVE registry.

In [ ]:
(QUANT_ROOT/'README.txt').write_text(
    'TDSC broader quantitative evaluation — system-level SICs\n\n'
    'One fixed Django SIC library applies to every Django checkpoint.\n'
    'SIC definitions and common risk factors were selected retrospectively; calibration review is outstanding.\n'
    'The case-study-specific pilot outputs are not used for this batch.\n'
    'Fix snapshots are reference-only; source evaluation uses only checkpoint-pinned code.\n'
    'Reference labels must be independently validated for each (checkpoint, SIC).\n'
    'No accuracy, F1 or confidence intervals should be claimed until the eligibility gates pass.\n'
    'Score-only comparisons are internal ablations, not external baseline comparisons.\n'
    'The Requests registry entry needs a separate system-level policy library.\n'
    'Runtime stage timings are not scaling curves.\n', encoding='utf-8')
archive=shutil.make_archive(str(QUANT_ROOT.parent/'TDSC_Broader_Quantitative_Audit'),
                            'zip',root_dir=QUANT_ROOT)
print('Quantitative audit exported:',archive)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(archive)


# 14. Optional supplementary evidence retrieval for further test validation

This optional stage collects extra **changed files and regression tests** around the published fixes for independent source verification. It is **not** a separate SIC configuration or a detection experiment. The system-level run has already collected all policy-anchor source files above. Do not use fixing patches or future tests as detector inputs for pre-fix checkpoints.

The Requests entry is only collected for a future, separately configured Requests experiment. This optional network-intensive stage is **disabled by default**.

In [ ]:
import urllib.request, urllib.error, time as _time, re as _re

FETCH_NEW_CANDIDATE_HISTORIES = False  # Optional test-evidence download only
MAX_SOURCE_FILES_PER_CASE = 16
MAX_FILE_BYTES = 180_000
HISTORY_DIR=QUANT_ROOT/'additional_history_evidence'
HISTORY_DIR.mkdir(parents=True,exist_ok=True)

def gh_json(url):
    req=urllib.request.Request(url,headers={'User-Agent':'S3DT-TDSC-Evidence-Collection','Accept':'application/vnd.github+json'})
    with urllib.request.urlopen(req,timeout=40) as r:return json.load(r)

def raw_pinned(owner_repo,sha,path):
    url=f'https://raw.githubusercontent.com/{owner_repo}/{sha}/{path}'
    req=urllib.request.Request(url,headers={'User-Agent':'S3DT-TDSC-Evidence-Collection'})
    with urllib.request.urlopen(req,timeout=45) as r:
        blob=r.read(MAX_FILE_BYTES+1)
    if len(blob)>MAX_FILE_BYTES:raise ValueError('file_size_limit')
    return blob,url

history_rows=[]
for item in CANDIDATE_CASES:
    if item['state']!='DOCUMENTED_CANDIDATE_NOT_EVALUATED':continue
    case_id=item['case_id']
    location=HISTORY_DIR/case_id
    location.mkdir(exist_ok=True)
    manifest_file=location/'history_evidence_manifest.json'
    if manifest_file.exists():
        manifest=json.loads(manifest_file.read_text())
        print('Using cached evidence:',case_id)
    elif not FETCH_NEW_CANDIDATE_HISTORIES:
        print('Not downloaded (network stage disabled):',case_id)
        continue
    else:
        manifest={'case_id':case_id,'repo':item['repository'],'fix_commit':item['fix_commit'],
                  'security_advisory_url':item['advisory_url'],
                  'status':'IN_PROGRESS','fix_is_reference_only':True,'retrieved_files':[],
                  'errors':[]}
        try:
            obj=gh_json('https://api.github.com/repos/'+item['repository']+'/commits/'+item['fix_commit'])
            if obj.get('sha')!=item['fix_commit']:raise ValueError('Fix commit mismatch')
            parents=obj.get('parents',[])
            if not parents:raise ValueError('No parent commit')
            manifest['parent_commit']=parents[0]['sha']
            manifest['committed_at']=obj['commit']['committer']['date']
            manifest['changed_file_list']=[f['filename'] for f in obj.get('files',[])]
            relevant=[f for f in obj.get('files',[]) if f['filename'].endswith(('.py','.html','.rst','.txt','.yml','.yaml','.toml'))]
            manifest['relevant_files_available']=len(relevant)
            manifest['files_limited_to']=MAX_SOURCE_FILES_PER_CASE
            for f in relevant[:MAX_SOURCE_FILES_PER_CASE]:
                path=f['filename']
                for name,sha in [('pre_fix_detector_state',manifest['parent_commit']),('post_fix_reference_only',item['fix_commit'])]:
                    if name=='pre_fix_detector_state' and f['status']=='added':continue
                    if name=='post_fix_reference_only' and f['status']=='removed':continue
                    try:
                        blob,origin=raw_pinned(item['repository'],sha,path)
                        target=location/name/path
                        target.parent.mkdir(parents=True,exist_ok=True)
                        target.write_bytes(blob)
                        manifest['retrieved_files'].append({'role':name,'git_commit':sha,'path':path,
                            'sha256':hashlib.sha256(blob).hexdigest(),'bytes':len(blob),'url':origin,
                            'detector_visible_at_pre_fix':name=='pre_fix_detector_state'})
                    except Exception as err:
                        manifest['errors'].append({'path':path,'role':name,'error':str(err)[:220]})
            manifest['status']='RETRIEVED_PENDING_SIC_AND_TEST_VALIDATION' if manifest['retrieved_files'] else 'NO_FILES_RETRIEVED'
        except Exception as err:
            manifest['status']='DOWNLOAD_OR_METADATA_ERROR'
            manifest['errors'].append({'error':str(err)[:400]})
        manifest_file.write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    history_rows.append({'case_id':case_id,'repository':item['repository'],'status':manifest.get('status'),
        'parent_commit':manifest.get('parent_commit',''),
        'fix_commit':item['fix_commit'],
        'retrieved_snapshots':len(manifest.get('retrieved_files',[])),
        'error_count':len(manifest.get('errors',[])),
        'ready_for_scoring':False})

history_summary=pd.DataFrame(history_rows)
history_summary.to_csv(QUANT_ROOT/'additional_history_collection_status.csv',index=False)
if len(history_summary):display(history_summary)
if not FETCH_NEW_CANDIDATE_HISTORIES:
    print('To retrieve historical code on Google Colab set FETCH_NEW_CANDIDATE_HISTORIES=True and run this cell.')

# Always export the collected evidence separately from the quantitative metrics.
history_package=shutil.make_archive(str(QUANT_ROOT.parent/'TDSC_Additional_History_Evidence'),'zip',root_dir=HISTORY_DIR)
print('Evidence archive:',history_package)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(history_package)


# 15. Final focused evaluation: does hybrid scoring add value under incomplete projected evidence?

**Retrospective, fixed-parameter ablation (separate from the original S³DT evaluation).** This section consumes the ZIP already produced by the Django run. You **do not** need to rerun Sections 0–14 or retrieve new incidents. Run **only the following three code cells** in Colab, in order.

**Inputs:** upload `TDSC_Broader_Quantitative_Audit(1).zip` from your previous run. This archive includes all historical pinned source snapshots, the shared Django SIC library and the original claim inventory. **No SWE-bench CSV and no S3DT-main.zip are needed for Section 15.**

**Fixed comparison protocol:** four *advisory-supported* security incident pairs (vulnerable/pre-fix and future-fix reference) form the primary descriptive comparison; the seven post-fix routine commits are reported as source-stability controls only. Cross-policy SIC states are **not given invented reference labels**. Structural evidence is deterministically masked at rates 0%, 33%, 67%, across seeds 11, 23, 47, 89, 101. The semantic channel embeds the unchanged **pinned source method text** against the Django policy description. Both channels exclude `risk_stage`, the SIC outcome, and reference labels. Use the **original** S³DT formula, `D_s=0.4C+0.2O+0.4H` and `D_h=0.5D_s+0.5D_embed`, with fixed classification cutoffs 0.25/0.65; a clearly documented **source-fact-derived adapter** constructs C/O/H *without using the SIC verdict*. The 4 incidents, 5 mask seeds, and 8 primary reference states are **not** independent enough to support generalizable accuracy or significance claims.

**Important separation:** This is a diagnostic comparison of the channels using an explicitly changed **input adapter**. It does **not** claim to be an unmodified replay of the original 60 scores. It does not tune lambda, thresholds, code-behaviour rules, or policy definitions for a better result. An absent structural channel produces `UNKNOWN`, never zero/PASS. No method is assumed to win.

**What counts as a meaningful finding:** actual correction of missed VIOLATIONs without offsetting false alarms in future-fix reference states, accompanied by unchanged or improved coverage. Merely increasing a numeric score, or repeating results across the five mask seeds, does not establish hybrid superiority.


In [ ]:
# Section 15 ONLY: fast path from your previous TDSC_Broader_Quantitative_Audit ZIP.
%pip -q install "numpy>=1.24" "pandas>=2" "sentence-transformers>=3" "matplotlib>=3" "nbformat>=5"
from pathlib import Path
import sys, zipfile, json, hashlib, importlib.metadata, shutil

S15_ROOT = Path('/content' if 'google.colab' in sys.modules else '/mnt/data')
S15_OUT = S15_ROOT / 'tdsc_final_hybrid_ablation'
S15_OUT.mkdir(parents=True,exist_ok=True)
archives = sorted(S15_ROOT.glob('TDSC_Broader_Quantitative_Audit*.zip'))
if not archives and 'google.colab' in sys.modules:
    from google.colab import files
    print('Upload the TDSC_Broader_Quantitative_Audit(1).zip from your existing Colab outputs.')
    _up = files.upload()
    archives = []
    for _filename,_blob in _up.items():
        _p = S15_ROOT/Path(_filename).name
        _p.write_bytes(_blob)
        archives.append(_p)
if not archives:
    raise FileNotFoundError('Upload your TDSC_Broader_Quantitative_Audit(1).zip into /content or /mnt/data.')
# Prefer an archive containing BOTH the pinned original sources and system-SIC results.
S15_AUDIT = None
for p in sorted(archives, key=lambda x:x.stat().st_size,reverse=True):
    try:
        with zipfile.ZipFile(p) as z:
            if 'system_sic_run/system_sic_claim_inventory.csv' in z.namelist() and any(n.startswith('pinned_source_cache/') and n.endswith('.py') for n in z.namelist()):
                S15_AUDIT=p
                break
    except zipfile.BadZipFile:pass
if S15_AUDIT is None:
    raise RuntimeError('None of the supplied ZIP files contains the full pinned-source audit. Use the larger ~(5 MB) broader quantitative audit ZIP.')
print('Focused-evaluation input:',S15_AUDIT)
print('Input SHA-256:',hashlib.sha256(S15_AUDIT.read_bytes()).hexdigest())
print('Original Django results remain unchanged. Section 15 writes to:',S15_OUT)


In [ ]:
# Section 15, Cell 2: run the original focused MiniLM evaluation.
# The SIC collection, source-fact adapter, fusion equation, masks and thresholds
# are UNCHANGED. This cell explicitly initialises paths and records the source.
from pathlib import Path
import sys, zipfile, hashlib, types

S15_ROOT = Path('/content' if 'google.colab' in sys.modules else '/mnt/data')
S15_OUT = S15_ROOT / 'tdsc_final_hybrid_ablation'
S15_OUT.mkdir(parents=True, exist_ok=True)

# Normally S15_AUDIT was selected by the preceding setup cell. Recheck it here
# so this cell is safe even if earlier notebook cells have been executed out of order.
if 'S15_AUDIT' not in globals() or not Path(S15_AUDIT).is_file():
    S15_AUDIT = None
if S15_AUDIT is None:
    for candidate in sorted(S15_ROOT.glob('TDSC_Broader_Quantitative_Audit*.zip'),
                            key=lambda p: p.stat().st_size, reverse=True):
        try:
            with zipfile.ZipFile(candidate) as archive:
                members = set(archive.namelist())
                if ('system_sic_run/system_sic_claim_inventory.csv' in members
                    and any(name.startswith('pinned_source_cache/') and name.endswith('.py')
                            for name in members)):
                    S15_AUDIT = candidate
                    break
        except zipfile.BadZipFile:
            continue
if S15_AUDIT is None:
    raise FileNotFoundError(
        'The broader quantitative audit ZIP was not found. Run the preceding '
        'Section 15 setup/upload cell, then rerun this evaluation cell.'
    )

import types
s15 = types.ModuleType('s15_focused_ablation')
S15_SOURCE = '"""S3DT focused hybrid-evidence ablation, separate from the frozen original runs.\nOnly raw pinned source artifacts and policy descriptions feed the semantic channel.\nOnly source-backed code_behavior facts feed the structured channel.\nNo outcome labels, risk_stage or risk-factor lookup keyed on SIC verdict is used.\n"""\nfrom __future__ import annotations\nimport ast, hashlib, io, json, math, re, zipfile, sys, time, platform\nfrom pathlib import Path\nfrom collections import defaultdict\nimport numpy as np\nimport pandas as pd\n\nLAM=0.50\nPASS_CUTOFF=0.25\nVIOL_CUTOFF=0.65\nMASK_LEVELS=(0.0,0.33,0.67)\nMASK_SEEDS=(11,23,47,89,101)\nMODEL_NAME=\'sentence-transformers/all-MiniLM-L6-v2\'\nPOSITIVE={\'change_permission_controls_mutation\',\'scheme_validation_present\',\'url_valid_gate\',\'path_validation_present\',\'safe_join_path\'}\nNEGATIVE={\'view_permission_controls_mutation\',\'scheme_validation_not_identified\',\'url_valid_gate_missing\',\'path_validation_not_identified\',\'direct_untrusted_path_join\'}\n# These are the source-extraction slots produced by the frozen framework for each SIC.\n# Explicit denominators make missingness measurable without assuming hidden evidence is benign.\nEXPECTED_SLOTS={\'DJANGO-AUTHZ-001\':3,\'DJANGO-URL-001\':2,\'DJANGO-UPLOAD-001\':3,\'DJANGO-TEMPLATE-001\':1}\nPRIMARY_CASE_POLICY={\n \'django__django-11149\':\'DJANGO-AUTHZ-001\',\n \'django__CVE-2019-12308\':\'DJANGO-URL-001\',\n \'django__CVE-2021-31542\':\'DJANGO-UPLOAD-001\',\n \'django__CVE-2021-33203\':\'DJANGO-TEMPLATE-001\',\n}\nEVIDENCE_URL={\n \'django__django-11149\':\'https://code.djangoproject.com/ticket/30289\',\n \'django__CVE-2019-12308\':\'https://www.djangoproject.com/weblog/2019/jun/03/security-releases/\',\n \'django__CVE-2021-31542\':\'https://www.djangoproject.com/weblog/2021/may/04/security-releases/\',\n \'django__CVE-2021-33203\':\'https://www.djangoproject.com/weblog/2021/jun/02/security-releases/\',\n}\n# Do NOT infer oracle outcomes from SIC decisions or code-behaviour claim labels.\n# Only one relevant policy per independently described fixing incident is labelled;\n# unrelated policy-checkpoints remain UNLABELLED.\n\ndef sha256(blob): return hashlib.sha256(blob).hexdigest()\ndef stable_frac(text):return int(sha256(text.encode())[:15],16)/float(16**15)\n\ndef fetch_archive(path):\n    z=zipfile.ZipFile(path)\n    must={\'system_sic_run/system_sic_transition_results.csv\',\n          \'system_sic_run/system_sic_claim_inventory.csv\',\n          \'system_sic_run/predeclared_django_system_sics.json\'}\n    if not must.issubset(set(z.namelist())):raise RuntimeError(\'Audit ZIP missing required source records: \'+str(must-set(z.namelist())))\n    transitions=pd.read_csv(io.BytesIO(z.read(\'system_sic_run/system_sic_transition_results.csv\'))).fillna(\'\')\n    claims=pd.read_csv(io.BytesIO(z.read(\'system_sic_run/system_sic_claim_inventory.csv\'))).fillna(\'\')\n    config=json.loads(z.read(\'system_sic_run/predeclared_django_system_sics.json\'))\n    return z,transitions,claims,config\n\ndef reference_for(row):\n    """Advisory-anchored pre/post proxies; not blind or tested for three cases."""\n    if PRIMARY_CASE_POLICY.get(row.case_id)!=row.sic_id:return None,\'UNLABELLED_CROSS_POLICY\'\n    if row.checkpoint==\'pre_fix_detector_state\':return \'VIOLATION\',\'DOCUMENTED_PRE_FIX_ADVISORY_PROXY\'\n    if row.checkpoint==\'post_fix_reference_only\':return \'PASS\',\'POST_FIX_REFERENCE_NOT_PROSPECTIVE\'\n    if row.case_id==\'django__django-11149\' and str(row.checkpoint).startswith(\'ordinary_post_fix_\'):\n        return \'PASS\',\'SOURCE_STABILITY_CONTROL_NOT_INDEPENDENT_ORACLE\'\n    return None,\'UNLABELLED\'\n\ndef method_source(text, cls_name, method_name):\n    try:tree=ast.parse(text)\n    except SyntaxError:return \'\'\n    for cls in ast.walk(tree):\n        if isinstance(cls,ast.ClassDef) and cls.name==cls_name:\n            for fn in cls.body:\n                if isinstance(fn,(ast.FunctionDef,ast.AsyncFunctionDef)) and fn.name==method_name:\n                    # AST unparse preserves assignment and path-guard semantics, while\n                    # discarding comments/documentation and any repository metadata.\n                    # Fixed truncation at 1,600 chars; no vulnerability-dependent selection.\n                    normalized=ast.unparse(fn)\n                    return normalized[:1600]\n\n    return \'\'\n\ndef source_for(z, commit, path):\n    member=\'pinned_source_cache/django_django/\'+commit+\'/\'+path\n    if member not in z.namelist():return None\n    return z.read(member).decode(\'utf8\',errors=\'replace\')\n\n# Purposefully uses code itself: no outcome wording, no advisories, no future fixes.\nEXTRACT_TARGETS={\n \'DJANGO-AUTHZ-001\':[\n  (\'django/contrib/admin/options.py\',\'InlineModelAdmin\',\'has_add_permission\'),\n  (\'django/contrib/admin/options.py\',\'InlineModelAdmin\',\'has_change_permission\'),\n  (\'django/contrib/admin/options.py\',\'InlineModelAdmin\',\'has_delete_permission\')],\n \'DJANGO-URL-001\':[\n  (\'django/contrib/admin/widgets.py\',\'AdminURLFieldWidget\',\'get_context\'),\n  (\'django/contrib/admin/templates/admin/widgets/url.html\',None,None)],\n \'DJANGO-UPLOAD-001\':[\n  (\'django/core/files/storage.py\',\'Storage\',\'get_available_name\'),\n  (\'django/core/files/storage.py\',\'Storage\',\'generate_filename\'),\n  (\'django/core/files/uploadedfile.py\',\'UploadedFile\',\'_set_name\')],\n \'DJANGO-TEMPLATE-001\':[\n  (\'django/contrib/admindocs/views.py\',\'TemplateDetailView\',\'get_context_data\')]\n}\n\ndef semantic_source(z,commit,policy):\n    parts=[];provenance=[]\n    for path,cls,method in EXTRACT_TARGETS[policy]:\n        src=source_for(z,commit,path)\n        if src is None:\n            provenance.append({\'path\':path,\'missing\':True});continue\n        excerpt=(method_source(src,cls,method) if cls else src.strip()[:1350])\n        if not excerpt:\n            provenance.append({\'path\':path,\'symbol\':method,\'missing\':True});continue\n        parts.append(f\'{cls+"."+method if cls else path}: {excerpt}\')\n        provenance.append({\'path\':path,\'symbol\':method,\'sha256\':sha256(src.encode()),\'missing\':False})\n    return \' \\n\'.join(parts),provenance\n\ndef structured_score(facts,masked_subjects,policy):\n    usable=[r for r in facts if r[\'id\'] not in masked_subjects]\n    contradiction=sum(1 for r in usable if r[\'object\'] in NEGATIVE)\n    positive=sum(1 for r in usable if r[\'object\'] in POSITIVE)\n    known=contradiction+positive\n    total=EXPECTED_SLOTS[policy]\n    if known==0:return float(\'nan\'),{\'C\':float(\'nan\'),\'O\':float(\'nan\'),\'H\':float(\'nan\'),\'known\':0,\'contradictions\':0,\'positive\':0,\'total\':total}\n    # S3DT\'s fixed coefficients 0.4C + 0.2O + 0.4H and original\n    # ordinal operational/harm settings (0.75). Critically no lookup by verdict.\n    C=contradiction/total\n    O=0.75 if contradiction else 0.0\n    H=0.75 if contradiction else 0.0\n    return round(0.4*C+0.2*O+0.4*H,12),dict(C=C,O=O,H=H,known=known,contradictions=contradiction,positive=positive,total=total)\n\ndef categorize(value):\n    if not np.isfinite(value):return \'UNKNOWN\'\n    if value<=PASS_CUTOFF:return \'PASS\'\n    if value<VIOL_CUTOFF:return \'SUSPECT\'\n    return \'VIOLATION\'\n\ndef prepared_facts(claims,case_id,checkpoint,policy):\n    subset=claims[(claims.case_id==case_id)&(claims.checkpoint==checkpoint)&(claims.policy_id==policy)&(claims.predicate==\'code_behavior\')]\n    facts=[]\n    for i,r in enumerate(subset.itertuples(index=False)):\n        if r.object not in NEGATIVE|POSITIVE:raise ValueError(\'Unreviewed code_behavior vocabulary \'+r.object)\n        facts.append({\'id\':f\'{i}:{r.subject}:{r.object}\',\'subject\':r.subject,\'object\':r.object,\'evidence\':r.evidence})\n    if len(facts)!=EXPECTED_SLOTS[policy]:\n        raise ValueError(f\'Incomplete extracted claim inventory: {case_id} {checkpoint} {policy} found {len(facts)} expected {EXPECTED_SLOTS[policy]}\')\n    return facts\n\ndef mask_selection(facts,mask_rate,seed,case_id,checkpoint,policy):\n    return {r[\'id\'] for r in facts if stable_frac(f\'{seed}|{case_id}|{checkpoint}|{policy}|{r["id"]}\') < mask_rate}\n\ndef evaluate(audit_path,out_dir,encoder=None,allow_unencoded=False):\n    out=Path(out_dir);out.mkdir(parents=True,exist_ok=True)\n    z,transitions,claims,config=fetch_archive(audit_path)\n    # Provenance and exact input integrity checks.\n    policies={p[\'id\']:p[\'description\'] for p in config[\'policies\']}\n    if set(policies)!=set(EXPECTED_SLOTS):raise AssertionError(\'Expected four fixed Django policies\')\n    if any(transitions.duplicated([\'case_id\',\'checkpoint\',\'sic_id\'])):raise AssertionError(\'Duplicate case-checkpoint-policy records\')\n    observed=[];label_rows=[]; semantic_inputs=[]\n    checked_sources={}\n    for r in transitions.itertuples(index=False):\n        if r.sic_id not in policies:raise AssertionError(\'Unexpected policy\')\n        ref,ref_grade=reference_for(r)\n        label_rows.append({\'case_id\':r.case_id,\'checkpoint\':r.checkpoint,\'commit\':r.commit,\'sic_id\':r.sic_id,\n           \'reference_label\':ref or \'\', \'reference_grade\':ref_grade,\n           \'advisory_url\':EVIDENCE_URL.get(r.case_id,\'\'),\'reference_only\':r.checkpoint==\'post_fix_reference_only\'})\n        key=(r.commit,r.sic_id)\n        if key not in checked_sources:\n            src,provenance=semantic_source(z,r.commit,r.sic_id)\n            if not src:raise RuntimeError(\'No pinned source excerpt for \'+str(key))\n            checked_sources[key]=(src,provenance)\n        src,provenance=checked_sources[key]\n        if any(p.get(\'missing\') for p in provenance):\n            raise RuntimeError(\'Target symbol absent in pinned history for \'+str(key)+\' \'+str(provenance))\n        semantic_inputs.append({\'case_id\':r.case_id,\'checkpoint\':r.checkpoint,\'commit\':r.commit,\n           \'sic_id\':r.sic_id,\'policy_text\':policies[r.sic_id], \'source_excerpt\':src,\n           \'source_excerpt_sha256\':sha256(src.encode()),\'provenance\':json.dumps(provenance,sort_keys=True),\n           \'risk_stage_absent\':(\'risk_stage\' not in src), \'oracle_label_absent\':(\'VIOLATION\' not in src and \'SATISFIED\' not in src)})\n    label_df=pd.DataFrame(label_rows)\n    semantic_df=pd.DataFrame(semantic_inputs)\n    if not semantic_df.risk_stage_absent.all():raise AssertionError(\'risk_stage leakage in raw source evidence\')\n    if not semantic_df.oracle_label_absent.all():raise AssertionError(\'outcome leakage in source evidence\')\n    # No exact matching of case IDs/advisories to embedding content.\n    for r in semantic_df.itertuples(index=False):\n        if any(s in r.source_excerpt for s in (\'CVE-2019-12308\',\'CVE-2021-31542\',\'CVE-2021-33203\',\'30289\')):\n            raise AssertionError(\'Issue identifier leaked in semantic source\')\n    # Separate semantic channel, computed ONCE per unique commit-policy from original source evidence.\n    embed_dist={}\n    if encoder is None and not allow_unencoded:raise RuntimeError(\'MiniLM encoder required. Install sentence-transformers and load all-MiniLM-L6-v2\')\n    if encoder is not None:\n        keys=sorted(checked_sources)\n        texts=[policies[sid] for (commit,sid) in keys]+[checked_sources[(commit,sid)][0] for commit,sid in keys]\n        vectors=np.asarray(encoder.encode(texts,normalize_embeddings=True,show_progress_bar=False),dtype=float)\n        if vectors.shape[0]!=2*len(keys):raise AssertionError(\'Unexpected number of embeddings\')\n        for i,key in enumerate(keys):\n            x=float(np.dot(vectors[i],vectors[i+len(keys)]))\n            embed_dist[key]=min(1.,max(0.,1.-x))\n    for r in transitions.itertuples(index=False):\n        facts=prepared_facts(claims,r.case_id,r.checkpoint,r.sic_id)\n        label,grade=reference_for(r)\n        d_emb=embed_dist.get((r.commit,r.sic_id),float(\'nan\'))\n        for seed in MASK_SEEDS:\n            for mask_rate in MASK_LEVELS:\n                mask=mask_selection(facts,mask_rate,seed,r.case_id,r.checkpoint,r.sic_id)\n                ds,info=structured_score(facts,mask,r.sic_id)\n                # Hybrid is UNKNOWN when structured score is UNKNOWN: do not silently impute PASS.\n                dh=LAM*ds+(1-LAM)*d_emb if np.isfinite(ds) and np.isfinite(d_emb) else float(\'nan\')\n                observed.append({\'case_id\':r.case_id,\'checkpoint\':r.checkpoint,\'commit\':r.commit,\'sic_id\':r.sic_id,\n                  \'seed\':seed,\'target_mask_fraction\':mask_rate,\'actual_masked\':len(mask),\n                  \'available_fact_count\':info[\'known\'],\'total_fact_count\':info[\'total\'],\n                  \'contradiction_count\':info[\'contradictions\'],\'positive_count\':info[\'positive\'],\n                  \'masked_fact_ids\':\'|\'.join(sorted(mask)),\'C\':info[\'C\'],\'O\':info[\'O\'],\'H\':info[\'H\'],\n                  \'D_structured\':ds,\'D_embed\':d_emb,\'D_hybrid\':dh,\n                  \'pred_structured\':categorize(ds),\'pred_embedding\':categorize(d_emb),\n                  \'pred_hybrid\':categorize(dh),\'reference_label\':label or \'\',\n                  \'reference_grade\':grade, \'oracle_is_independent_of_predictions\':True,\n                  \'source_excerpt_sha256\':sha256(checked_sources[(r.commit,r.sic_id)][0].encode()),\n                  \'is_reference_only\':r.checkpoint==\'post_fix_reference_only\'})\n    df=pd.DataFrame(observed)\n    # Both hybrid and structured must use the same masks at the same checkpoint.\n    assert len(df)==len(transitions)*len(MASK_SEEDS)*len(MASK_LEVELS)\n    assert df.groupby([\'case_id\',\'checkpoint\',\'sic_id\',\'seed\']).target_mask_fraction.nunique().eq(3).all()\n    assert df.groupby([\'case_id\',\'checkpoint\',\'sic_id\']).D_embed.nunique(dropna=False).eq(1).all()\n    assert df[df.target_mask_fraction==0].pred_structured.ne(\'UNKNOWN\').all()\n    assert set(df[df.reference_grade.eq(\'DOCUMENTED_PRE_FIX_ADVISORY_PROXY\')].sic_id)==set(PRIMARY_CASE_POLICY.values())\n    # For a scientifically fair primary comparison, do not count the seven source-stability\n    # control states as independently labelled successes.\n    primary=df[df.reference_grade.isin([\'DOCUMENTED_PRE_FIX_ADVISORY_PROXY\',\'POST_FIX_REFERENCE_NOT_PROSPECTIVE\'])].copy()\n    assert primary.groupby([\'seed\',\'target_mask_fraction\']).size().eq(8).all()\n    methods={\'structured\':\'pred_structured\',\'embedding\':\'pred_embedding\',\'hybrid\':\'pred_hybrid\'}\n    metrics=[]; per_case=[]\n    for (seed,level),g in primary.groupby([\'seed\',\'target_mask_fraction\']):\n        for method,column in methods.items():\n            coverage=int(g[column].ne(\'UNKNOWN\').sum())\n            v=g[g.reference_label==\'VIOLATION\']\n            p=g[g.reference_label==\'PASS\']\n            tp=int((v[column]==\'VIOLATION\').sum());fp=int((p[column]==\'VIOLATION\').sum())\n            FN=len(v)-tp\n            precision=tp/(tp+fp) if tp+fp else float(\'nan\')\n            recall=tp/len(v) if len(v) else float(\'nan\')\n            f1=2*precision*recall/(precision+recall) if tp+fp and precision+recall else float(\'nan\')\n            tri_accuracy=float((g[column]==g.reference_label).sum()/len(g))\n            metrics.append(dict(seed=seed,mask_rate=level,method=method,\n               n_incidents=g.case_id.nunique(),n_reference_states=len(g),n_available=coverage,\n               coverage=coverage/len(g),violations= len(v),fixed_reference_states=len(p),\n               tp=tp,fp=fp,fn=FN,violation_precision=precision,violation_recall=recall,\n               violation_f1=f1,three_class_accuracy_with_unknown_errors=tri_accuracy))\n            for case_id,rows in g.groupby(\'case_id\'):\n                before=rows.loc[rows.checkpoint==\'pre_fix_detector_state\'].iloc[0]\n                after=rows.loc[rows.checkpoint==\'post_fix_reference_only\'].iloc[0]\n                per_case.append(dict(seed=seed,mask_rate=level,method=method,case_id=case_id,\n                  pre_fix=before[column],post_fix=after[column],\n                  pre_fix_correct=before[column]==\'VIOLATION\',post_fix_correct=after[column]==\'PASS\'))\n    perf=pd.DataFrame(metrics)\n    per_case=pd.DataFrame(per_case)\n    # Pairwise disagreement and recovery: where structured misses a known pre-fix violation,\n    # does hybrid correctly identify it WITHOUT harming the paired fix classification?\n    piv=primary.pivot_table(index=[\'case_id\',\'checkpoint\',\'seed\',\'target_mask_fraction\'],\n            columns=[],values=[\'pred_structured\'],aggfunc=\'first\').reset_index() if False else primary.copy()\n    recovery=[]\n    for level in MASK_LEVELS:\n        subset=primary[(primary.target_mask_fraction==level)&(primary.reference_label==\'VIOLATION\')]\n        n_lost=int(subset.pred_structured.ne(\'VIOLATION\').sum())\n        regained=int(((subset.pred_structured!=\'VIOLATION\')&(subset.pred_hybrid==\'VIOLATION\')).sum())\n        newly_missed=int(((subset.pred_structured==\'VIOLATION\')&(subset.pred_hybrid!=\'VIOLATION\')).sum())\n        recovery.append(dict(mask_rate=level,n_pre_fix_states=len(subset),structured_misses=n_lost,\n                             hybrid_recovered=regained,hybrid_new_misses=newly_missed))\n    recover_df=pd.DataFrame(recovery)\n    eligible_labels=label_df[label_df.reference_grade.isin([\'DOCUMENTED_PRE_FIX_ADVISORY_PROXY\',\'POST_FIX_REFERENCE_NOT_PROSPECTIVE\'])]\n    config_record={\'study\':\'Retrospective evidence ablation; not frozen before original incident selection\',\n       \'original_system_config_sha256\':sha256(json.dumps(config,sort_keys=True).encode()),\n       \'baseline_outputs_used_for_new_scoring\':False,\n       \'policy_text_source\':\'predeclared_django_system_sics.json\',\n       \'structure_facts_source\':\'code_behavior only; no risk_stage or sic_outcome\',\n       \'semantic_input_source\':\'pinned source-method excerpts; no source-level risk_stage or labels\',\n       \'scoring_formula\':\'D_s=0.4C+0.2O+0.4H, D_h=0.5D_s+0.5D_e, D_e=clipped cosine distance\',\n       \'factor_adapter\':\'C=contradictions/expected slots; O=H=0.75 when any observed contradiction, 0 otherwise; UNKNOWN when no observed facts\',\n       \'structural_projection_mask\':\'fixed SHA256(subject, object, seed, checkpoint) independent of labels\',\n       \'mask_rates\':list(MASK_LEVELS),\'mask_seeds\':list(MASK_SEEDS),\n       \'fixed_thresholds\':{\'PASS_max\':PASS_CUTOFF,\'VIOLATION_min\':VIOL_CUTOFF},\n       \'weight_lambda\':LAM,\'model\':MODEL_NAME,\n       \'reference_grade\':\'four advisory-anchored pre/post pairs, corrections reference-only, seven source-stability controls excluded from primary performance\',\n       \'limitation\':\'All policy families selected retrospectively; no blind test; four incidents, no significance/CI. Embedding model is not trained or tuned.\',\n       \'comparison_not_original_s3dt_scores\':True,\n       \'n_unique_source_policy\':len(checked_sources),\n       \'n_primary_reference_states\':len(eligible_labels),\n       \'n_unlabelled_cross_policy\':int((label_df.reference_grade==\'UNLABELLED_CROSS_POLICY\').sum())}\n    df.to_csv(out/\'ablation_all_sic_checkpoints.csv\',index=False)\n    primary.to_csv(out/\'ablation_primary_advisory_pairs.csv\',index=False)\n    perf.to_csv(out/\'ablation_per_seed_comparisons.csv\',index=False)\n    per_case.to_csv(out/\'ablation_per_incident_predictions.csv\',index=False)\n    recover_df.to_csv(out/\'ablation_recovery_diagnostics.csv\',index=False)\n    label_df.to_csv(out/\'reference_scope_and_provenance.csv\',index=False)\n    semantic_df.to_csv(out/\'semantic_inputs_audit.csv\',index=False)\n    (out/\'fixed_ablation_protocol.json\').write_text(json.dumps(config_record,indent=2))\n    lines=[\'S3DT FINAL FOCUSED ABLATION — INTERPRETATION GUIDE\',\'\',\n         \'This is a NEW DIAGNOSTIC EVIDENCE-LOSS ABLATION on the existing source snapshots.\',\n         \'It does NOT overwrite or modify the original S3DT scores, equations or published results.\',\n         \'The C/O/H adapter is source-fact based and NOT the old SIC-verdict mapping.\',\n         \'Pre-fix and fixed reference labels are advisory-anchored proxies; fixes are known future states.\',\n         \'The seven normal commits are source-stability controls only, NOT independent performance labels.\',\n         \'Each seed is a deterministic mask repeat, NOT a new independent software incident.\',\n         \'The semantic channel embeds raw relevant pinned source excerpts + policy expectation ONLY.\',\n         \'Do NOT claim hybrid superiority unless results independently support it.\',\n         \'A positive result establishes at most robustness on four selected historical incidents.\',\n         \'If original thresholds yield no benefit, report this directly. Do not retune post hoc.\',\'\',\n         \'Read ablation_per_seed_comparisons.csv for method scores AND coverage.\',\n         \'Read ablation_per_incident_predictions.csv for the actual 4 case pairs.\',\n         \'Read semantic_inputs_audit.csv for the independently inspectable textual inputs.\',\n         \'Read reference_scope_and_provenance.csv for which checkpoints lack oracle labels.\',\n         \'\']\n    (out/\'README.txt\').write_text(\'\\n\'.join(lines))\n    z.close()\n    return df,perf,per_case,recover_df,config_record\n'
exec(S15_SOURCE, s15.__dict__)
from sentence_transformers import SentenceTransformer
import torch
s15_encoder = SentenceTransformer(s15.MODEL_NAME,
    device='cuda' if torch.cuda.is_available() else 'cpu')
print('Sentence encoder:', s15.MODEL_NAME, 'device:', s15_encoder.device)
s15_rows, s15_perf, s15_per_case, s15_recovery, s15_manifest = s15.evaluate(
    S15_AUDIT, S15_OUT, encoder=s15_encoder
)

# Preserve the EXACT evaluation-source checksum on disk, so the export works
# after a Colab runtime restart without depending on Python variables.
(S15_OUT / 'source_module.sha256').write_text(
    hashlib.sha256(S15_SOURCE.encode('utf-8')).hexdigest() + '\n'
)
print('All scored policy-checkpoint-mask rows:', len(s15_rows))
print('Primary incident-pair rows (including five seeds):',
      len(s15_rows[s15_rows.reference_grade.isin([
          'DOCUMENTED_PRE_FIX_ADVISORY_PROXY',
          'POST_FIX_REFERENCE_NOT_PROSPECTIVE'])]))
print('True primary incidents:', s15_per_case.case_id.nunique())
print('No changes to lambda or cutoffs.')


In [ ]:
# Section 15, Cell 3: summary and GitHub-ready ZIP export.
# Restart-safe: reads the saved CSV results and recorded source hash from disk.
# It DOES NOT rerun scoring or require S15_OUT/S15_SOURCE from earlier cells.
from pathlib import Path
import sys, json, hashlib, zipfile, importlib.metadata, re
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

S15_ROOT = Path('/content' if 'google.colab' in sys.modules else '/mnt/data')
S15_OUT = S15_ROOT / 'tdsc_final_hybrid_ablation'
required = (
    'ablation_per_seed_comparisons.csv',
    'ablation_per_incident_predictions.csv',
    'ablation_recovery_diagnostics.csv',
    'fixed_ablation_protocol.json',
)
missing = [name for name in required if not (S15_OUT / name).is_file()]
if missing:
    raise FileNotFoundError(
        'The focused evaluation has not saved all its outputs. Missing: '
        + ', '.join(missing)
        + '. Run the preceding Section 15 evaluation cell first; '
          'you do NOT need to rerun the original Django experiment.'
    )

s15_perf = pd.read_csv(S15_OUT / 'ablation_per_seed_comparisons.csv')
s15_per_case = pd.read_csv(S15_OUT / 'ablation_per_incident_predictions.csv')
s15_recovery = pd.read_csv(S15_OUT / 'ablation_recovery_diagnostics.csv')
print('Descriptive averages across fixed missingness seeds: these are NOT confidence intervals.')
s15_summary = (s15_perf.groupby(['mask_rate','method'],as_index=False)
    [['coverage','violation_recall','violation_f1',
      'three_class_accuracy_with_unknown_errors','fp']].mean())
display(s15_summary.round(3))
print('Paired comparison on the documented vulnerability states:')
display(s15_recovery)
print('Detailed outcomes per incident, seed and mask condition:')
display(s15_per_case.head(24))
s15_summary.to_csv(S15_OUT / 'descriptive_means_NOT_confidence_intervals.csv', index=False)

fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for method, group in s15_summary.groupby('method'):
    group = group.sort_values('mask_rate')
    axs[0].plot(group.mask_rate, group.violation_recall, marker='o', label=method)
    axs[1].plot(group.mask_rate, group.coverage, marker='o', label=method)
for ax, heading in [(axs[0], 'Violation recall'), (axs[1], 'Evidence coverage')]:
    ax.set(title=heading, xlabel='Projection fact removal probability',
           ylabel='Proportion', ylim=(-0.05, 1.05))
    ax.grid(True, alpha=0.25)
axs[0].legend()
fig.suptitle('Four advisory-anchored Django incident pairs · five deterministic masks (not independent trials)')
fig.tight_layout()
fig.savefig(S15_OUT / 'hybrid_evidence_ablation_descriptive.png',
            dpi=180, bbox_inches='tight')
plt.show()

# The source checksum is written by the evaluation cell, so it survives restarts.
source_hash_file = S15_OUT / 'source_module.sha256'
source_sha256 = source_hash_file.read_text().strip() if source_hash_file.exists() else None
if source_sha256 is not None and not re.fullmatch(r'[0-9a-f]{64}', source_sha256):
    raise ValueError('Invalid evaluation-source SHA256 checksum file')

# Find the original input archive again if a kernel restart cleared variables.
audit_files = sorted(S15_ROOT.glob('TDSC_Broader_Quantitative_Audit*.zip'),
                     key=lambda p: p.stat().st_size, reverse=True)
audit_files = [p for p in audit_files if zipfile.is_zipfile(p)]
input_archive = None
if 'S15_AUDIT' in globals() and S15_AUDIT is not None and Path(S15_AUDIT).is_file():
    input_archive = Path(S15_AUDIT)
else:
    for p in audit_files:
        with zipfile.ZipFile(p) as archive:
            members = set(archive.namelist())
            if ('system_sic_run/system_sic_claim_inventory.csv' in members
                and any(n.startswith('pinned_source_cache/') and n.endswith('.py') for n in members)):
                input_archive = p
                break

def installed_version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None

environment = {
    'python': sys.version,
    'packages': {p: installed_version(p) for p in
                 ['sentence-transformers','numpy','pandas','torch']},
    'input_sha256': hashlib.sha256(input_archive.read_bytes()).hexdigest()
                    if input_archive else None,
    'source_module_sha256': source_sha256,
    'source_checksum_note': 'Captured by the evaluation cell at run time; null if absent',
}
(S15_OUT / 'environment.json').write_text(json.dumps(environment, indent=2))
zip_out = S15_ROOT / 'TDSC_Final_Focused_Hybrid_Ablation_Results.zip'
with zipfile.ZipFile(zip_out, 'w', compression=zipfile.ZIP_DEFLATED) as result_zip:
    for path in sorted(S15_OUT.iterdir()):
        if path.is_file():
            result_zip.write(path, arcname=path.name)
with zipfile.ZipFile(zip_out) as check:
    assert all(name in check.namelist() for name in required)
print('Complete results ZIP:', zip_out)
print('Important: judge hybrid recovery by the paired case outcomes, not just means.')
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(str(zip_out))
